## Route planning

In [ ]:
origin = (121.6147,31.36636)   
dest   = (-118.265,33.74021)  

#| KAOHSIUNG  | TWKHH     | (22.58425,120.3181)  | (120.3181,22.58425)            |
#| TAICHUNG   | TWTXG     | (24.255,120.5075)    | (120.5075,24.255)              |
#| KEELUNG    | TWKEL     | (25.14515,121.7532)  | (121.7532,25.14515)            |

#| SHANGHAI  | CNSHG     | (31.36636,121.6147) | (121.6147,31.36636)            |
#| NINGBO    | CNNBG     | (29.92654,121.8525) | (121.8525,29.92654)            |
#| ZHOUSHAN  | CNZOS     | (29.92161,122.2104) | (122.2104,29.92161)            |
#| SHENZHEN  | CNSZX     | (22.5045,113.8535)  | (113.8535,22.5045)             |

#| LOS ANGELES | USLAX     | (33.74021,-118.265) | (-118.265,33.74021)            |
#| SEATTLE     | USSEA     | (47.6212,-122.3643) | (-122.3643,47.6212)            |

#| TOKYO     | JPTYO     | (35.61168,139.8268) | (139.8268,35.61168)            |
#| KOBE      | JPUKB     | (34.6867,135.2671)  | (135.2671,34.6867)             |
#| WAKAYAMA  | JPUKW     | (33.468056,135.828056) | (135.828056,33.468056)       |

In [4]:
# === Lazy Visibility Search (Pacific view, great-circle, full-bbox features) ===
# Author: you + ChatGPT
# Note:
#  - Keep your original geodesy, Pacific-view drawing, land buffering & collision layers.
#  - Replace "outer tangent detour" with a Lazy Visibility Search (LVS) graph.
#  - Nodes = {O, D} ∪ {bbox features}; Edges are only validated lazily by visible().
#  - When a candidate shortest path has an invalid edge, mark it BLOCKED (and optionally inject gateways),
#    then replan until a fully valid path is found.
#
# Usage:
#   1) Set origin, dest (lon,lat) below (or call plan_route(...))
#   2) Run the script; it outputs a Folium HTML with Pacific-centered continuous arcs.

from __future__ import annotations
from pathlib import Path
import math
import heapq
import itertools
from typing import List, Tuple, Dict, Optional, Iterable

import fiona
import folium
from networkx import nodes
import shapely
from shapely.geometry import shape, Polygon, Point, LineString, GeometryCollection
from shapely.ops import unary_union
from shapely.prepared import prep
from shapely.strtree import STRtree
from pyproj import Transformer, Geod

# ---------------- Params (edit as needed) ----------------
LAND_PATH   = Path(r"C:\Users\slab\Desktop\Slab Project\Stage1\data\Land\ne_10m_land.shp")
BUFFER_KM   = 5.0                    # inner "no-go" near coast; points inside will be nudged outward
COLLISION_SAFETY_KM = 0.25           # collision dilation for visibility (rigid safety margin)
PAD_DEG     = 6.0                    # bbox padding in degrees
STEP_KM_GEODESIC = 3.0               # great-circle sampling resolution for visibility
DRAW_STEP_KM = 20.0                  # great-circle sampling for drawing
AVOID_KM    = 15.0                   # outer ring where routes/waypoints are allowed

# LVS knobs
NEIGHBOR_K = 24                      # candidate neighbors per node per expansion (plus D)
LVS_MAX_NODES = 4000                 # guard rail for very large bbox
USE_ROTATION_PENALTY = False         # you asked to keep it OFF initially
ROT_LAMBDA = 0.001                   # weight if enabled

# ---------------- CRS & geodesy ----------------
to_m  = Transformer.from_crs("EPSG:4326","EPSG:3857", always_xy=True).transform
to_ll = Transformer.from_crs("EPSG:3857","EPSG:4326", always_xy=True).transform
def to_metric(g): return shapely.ops.transform(to_m, g)
def to_wgs(g):    return shapely.ops.transform(to_ll, g)
GEOD = Geod(ellps="WGS84")


def _fmt_ll(p):
    return f"({p[0]:.5f}, {p[1]:.5f})"  # (lon, lat)

def _print_ll(msg, p):
    print(f"{msg} {_fmt_ll(p)}", flush=True)

def _print_edge(prefix, a, b):
    print(f"{prefix} {_fmt_ll(a)} -> {_fmt_ll(b)}", flush=True)


def geodesic_sample(a: Tuple[float,float], b: Tuple[float,float], step_km: float=STEP_KM_GEODESIC) -> List[Tuple[float,float]]:
    lon1,lat1=a; lon2,lat2=b
    _,_,dist_m = GEOD.inv(lon1,lat1,lon2,lat2)
    n = max(1,int(dist_m/(step_km*1000)))
    pts = GEOD.npts(lon1,lat1,lon2,lat2,n)
    return [(lon1,lat1)] + pts + [(lon2,lat2)]

def great_circle_midpoint(a,b):
    pts = geodesic_sample(a,b,step_km=500.0)
    return pts[len(pts)//2]

def gc_distance_km(a,b) -> float:
    _,_,d = GEOD.inv(a[0],a[1],b[0],b[1])
    return d/1000.0

def bearing_xy(p,q):
    px,py = to_m(p[0],p[1]); qx,qy = to_m(q[0],q[1])
    return math.degrees(math.atan2(qy-py, qx-px)) % 360.0

def angle_diff(a,b):
    return (a-b+540)%360 - 180

# ---------------- Pacific view helpers ----------------
def normalize_lon_to_pacific_view(lon: float) -> float:
    return lon if lon>=0 else lon+360

def draw_gc_polyline_continuous(m, a, b, step_km=DRAW_STEP_KM, **style):
    pts = geodesic_sample(a, b, step_km=step_km)
    folium_coords = []
    for lon, lat in pts:
        lon_pacific = normalize_lon_to_pacific_view(lon)
        folium_coords.append([lat, lon_pacific])
    folium.PolyLine(folium_coords, **style).add_to(m)

def draw_progress(
    progress: Dict,
    nodes: List[Tuple[float,float]],
    origin: Tuple[float,float],
    dest: Tuple[float,float],
    out_html: str = r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA\route_progress.html",
    land_geom=None,     # 建議傳 land_raw_wgs（可為 None）
    ring_geom=None,     # 建議傳 RING_WGS（可為 None）
    feature_nodes: Optional[List[Tuple[float,float]]] = None,  # 若要疊上候選特徵點群組
    show_features: bool = True,
):
    """
    將目前 LVS 進度畫成地圖：
      - 已驗證通過的邊（含過去輪次的 free_edges）→ 實線藍色
      - 本輪候選路徑中已通過的前綴 → 實線藍色
      - 本輪候選路徑中尚未驗證的後綴 → 橘色虛線
      - 若 O/D 曾被推移（nodes[0] != origin 或 nodes[1] != dest），補畫接駁段

    參數：
      progress: 由 lazy_visibility_search(progress=...) 維護的 dict
      nodes:    與 progress["nodes_ref"] 同一個 nodes（索引一致）
      origin,dest: 原始輸入 O/D（lon,lat）
      land_geom, ring_geom: 供背景展示；可傳 None 跳過
      feature_nodes: 要疊圖的候選特徵點（例如 convex_peaks+convex）
    """
    # ---- 讀進度 ----
    cand_idx = progress.get("candidate_path", []) or []
    k = int(progress.get("free_prefix_len", 0) or 0)
    free_edges_hist = list(progress.get("free_edges", []) or [])
    O_idx, D_idx = 0, 1  # 你的設計固定 0/1

    # 安全防呆
    if not nodes or len(nodes) < 2:
        print("[draw_progress] nodes 不足，無法畫圖")
        return
    if not cand_idx:
        # 沒有候選路徑就至少把 O/D 畫出來
        cand_idx = [O_idx, D_idx]
        k = 0

    # 地圖中心：取 O-D 大圓中點（太平洋視角）
    mid_lon, mid_lat = great_circle_midpoint(origin, dest)
    center_lon_pacific = normalize_lon_to_pacific_view(mid_lon)
    center_lat = mid_lat

    m = folium.Map(
        location=[center_lat, center_lon_pacific],
        zoom_start=4,
        max_bounds=False, world_copy_jump=False, no_wrap=False, min_lon=0, max_lon=360
    )

    # 底圖
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Ocean_Basemap/MapServer/tile/{z}/{y}/{x}',
        attr='Esri', name='Ocean Basemap', overlay=False, control=True, no_wrap=False
    ).add_to(m)
    folium.TileLayer(
        tiles='https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',
        attr='© OpenStreetMap', name='OpenStreetMap', overlay=False, control=True, no_wrap=False
    ).add_to(m)

    # 背景：陸域、5km 緩衝
    if land_geom is not None:
        folium.GeoJson(
            convert_geom_to_pacific(land_geom),
            name="陸地",
            style_function=lambda x: {"color":"#2ca02c","weight":1,"fillOpacity":0.15}
        ).add_to(m)
    if ring_geom is not None:
        folium.GeoJson(
            convert_geom_to_pacific(ring_geom),
            name=f"航道緩衝區 {BUFFER_KM}km",
            style_function=lambda x: {"color":"#6a5acd","weight":2,"fillOpacity":0.05}
        ).add_to(m)

    # O/D 標記
    folium.Marker(
        [origin[1], normalize_lon_to_pacific_view(origin[0])],
        tooltip=f"起點: ({origin[0]:.4f}, {origin[1]:.4f})",
        icon=folium.Icon(color='green', icon='ship', prefix='fa')
    ).add_to(m)
    folium.Marker(
        [dest[1], normalize_lon_to_pacific_view(dest[0])],
        tooltip=f"終點: ({dest[0]:.4f}, {dest[1]:.4f})",
        icon=folium.Icon(color='red', icon='anchor', prefix='fa')
    ).add_to(m)

    # 若 O/D 被推移，補畫接駁段（實線藍）
    O_adj = nodes[O_idx]; D_adj = nodes[D_idx]
    if O_adj != origin:
        draw_gc_polyline_continuous(m, origin, O_adj, step_km=DRAW_STEP_KM,
                                    color='#1f77b4', weight=5, opacity=0.9)
    if D_adj != dest:
        draw_gc_polyline_continuous(m, D_adj, dest, step_km=DRAW_STEP_KM,
                                    color='#1f77b4', weight=5, opacity=0.9)

    # 歷史已驗證 FREE 的邊（跨輪次）：實線藍
    # （有些邊可能不在當前候選路徑上，也照畫，幫助你理解探索過的走向）
    for (u, v) in free_edges_hist:
        if u < len(nodes) and v < len(nodes):
            a, b = nodes[u], nodes[v]
            draw_gc_polyline_continuous(
                m, a, b, step_km=DRAW_STEP_KM,
                color='#1f77b4', weight=4, opacity=0.65
            )

    # 本輪候選路徑：前綴已驗證（實線藍）
    if len(cand_idx) >= 2 and k > 0:
        for u, v in zip(cand_idx[:k], cand_idx[1:k+1]):
            if u < len(nodes) and v < len(nodes):
                a, b = nodes[u], nodes[v]
                draw_gc_polyline_continuous(
                    m, a, b, step_km=DRAW_STEP_KM,
                    color='#1f77b4', weight=6, opacity=0.95
                )

    # 本輪候選路徑：後綴未驗證（橘色虛線）
    if len(cand_idx) >= 2 and k < len(cand_idx) - 1:
        for u, v in zip(cand_idx[k:-1], cand_idx[k+1:]):
            if u < len(nodes) and v < len(nodes):
                a, b = nodes[u], nodes[v]
                draw_gc_polyline_continuous(
                    m, a, b, step_km=DRAW_STEP_KM,
                    color='#ff7f0e', weight=5, opacity=0.9, dash_array="10,6"
                )

    # 候選節點點標（可視化目前 path）
    #  - 前綴點：實心藍圈
    #  - 後綴點：橘圈
    for i, idx in enumerate(cand_idx):
        if idx >= len(nodes): 
            continue
        lon, lat = nodes[idx]
        ll = [lat, normalize_lon_to_pacific_view(lon)]
        if i <= k:
            folium.CircleMarker(ll, radius=4, color='#1f77b4', fill=True, fill_opacity=0.9,
                                tooltip=f"cand[{i}] {_fmt_ll((lon,lat))} (已驗證前綴)").add_to(m)
        else:
            folium.CircleMarker(ll, radius=4, color='#ff7f0e', fill=True, fill_opacity=0.7,
                                tooltip=f"cand[{i}] {_fmt_ll((lon,lat))} (待驗證)").add_to(m)

    # 額外：候選特徵點群組（debug 層）
    if show_features and feature_nodes:
        fg_feat = folium.FeatureGroup(name="候選特徵點（凸峰+凸點）", show=False)
        for (lon, lat) in feature_nodes:
            folium.CircleMarker(
                [lat, normalize_lon_to_pacific_view(lon)], radius=3,
                color="#1f77b4", fill=True, fill_opacity=0.8,
                tooltip=f"Feature ({lon:.3f},{lat:.3f})"
            ).add_to(fg_feat)
        fg_feat.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    m.save(out_html)
    print(f"[draw_progress] 進度地圖已輸出：{out_html}")


# ---------------- IDL-aware dynamic bboxes ----------------
def dynamic_bboxes_idl(origin, dest, pad_deg: float) -> List[Polygon]:
    o_lon = normalize_lon_to_pacific_view(origin[0])
    d_lon = normalize_lon_to_pacific_view(dest[0])
    min_lon = min(o_lon, d_lon) - pad_deg
    max_lon = max(o_lon, d_lon) + pad_deg
    min_lat = min(origin[1], dest[1]) - pad_deg
    max_lat = max(origin[1], dest[1]) + pad_deg

    bboxes=[]
    if min_lon < 0:
        bboxes.append(Polygon([(min_lon,min_lat),(0,min_lat),(0,max_lat),(min_lon,max_lat)]))
        min_lon=0
    if max_lon > 360:
        bboxes.append(Polygon([(0,min_lat),(max_lon-360,min_lat),(max_lon-360,max_lat),(0,max_lat)]))
        max_lon=360

    lon_min_std = min_lon if min_lon <= 180 else min_lon-360
    lon_max_std = max_lon if max_lon <= 180 else max_lon-360

    if lon_min_std <= lon_max_std:
        bboxes.append(Polygon([(lon_min_std,min_lat),(lon_max_std,min_lat),
                               (lon_max_std,max_lat),(lon_min_std,max_lat)]))
    else:
        bboxes.append(Polygon([(lon_min_std,min_lat),(180,min_lat),(180,max_lat),(lon_min_std,max_lat)]))
        bboxes.append(Polygon([(-180,min_lat),(lon_max_std,min_lat),(lon_max_std,max_lat),(-180,max_lat)]))
    return bboxes

def union_lonlat_bboxes(bboxes: List[Polygon]) -> Polygon:
    # just union them (they are already lon/lat)
    if not bboxes: raise ValueError("No bboxes")
    u = unary_union(bboxes)
    if isinstance(u, GeometryCollection):
        # take envelope if weird
        return u.envelope
    return u

def load_polys_in_bboxes(shp_path: Path, bbox_polys: List[Polygon]) -> List[Polygon]:
    polys=[]
    with fiona.open(shp_path) as src:
        for feat in src:
            g = shape(feat["geometry"])
            for box in bbox_polys:
                if g.intersects(box):
                    gi = g.intersection(box)
                    try:
                        parts = list(shapely.get_parts(gi))
                    except Exception:
                        parts = list(gi.geoms) if gi.geom_type=="MultiPolygon" else [gi]
                    for p in parts:
                        if not p.is_empty:
                            polys.append(p)
                    break
    return polys

# ---------------- Land layers & spatial indexes ----------------
def build_land_layers(polys: List[Polygon]):
    # Work in metric
    parts_m = [to_metric(p).buffer(0) for p in polys]
    union_m = unary_union(parts_m)
    collision_m = union_m.buffer(COLLISION_SAFETY_KM * 1000.0)
    ring_m      = union_m.buffer(BUFFER_KM * 1000.0)
    return {
        "UNION_M":          union_m,
        "COLLISION_PREP_M": prep(collision_m),
        "COLLISION_WGS":    to_wgs(collision_m),
        "RING_M":           ring_m,
        "RING_WGS":         to_wgs(ring_m),
        "LAND_RAW_WGS":     to_wgs(union_m),
        "LAND_PARTS_M":     parts_m,
        "COLLISION_M":      collision_m,
    }

def build_land_strtree(parts_m: List[shapely.geometry.base.BaseGeometry]) -> STRtree:
    # STRtree on metric parts to quickly shortlist potential intersections
    return STRtree(parts_m)

# ---------------- Nudge helpers ----------------
def nudge_to_ring_if_inside(pt_ll: Tuple[float,float], UNION_M, inner_buffer_km: float = BUFFER_KM, target_offset_km: float = AVOID_KM):
    px, py = to_m(pt_ll[0], pt_ll[1])
    p_m = Point(px, py)
    inner_ring_m  = UNION_M.buffer(inner_buffer_km * 1000.0)
    target_ring_m = UNION_M.buffer(target_offset_km * 1000.0)
    if inner_ring_m.contains(p_m):
        boundary = target_ring_m.boundary if not target_ring_m.boundary.is_empty else target_ring_m
        q_m = shapely.ops.nearest_points(p_m, boundary)[1]
        q_llx, q_lly = to_ll(q_m.x, q_m.y)
        return (q_llx, q_lly), True
    return pt_ll, False

# ---- Fast nudge: reuse precomputed rings (NO per-call buffer) ----
def nudge_to_ring_if_inside_fast(
    pt_ll,
    inner_ring_m,          # = UNION_M.buffer(BUFFER_KM*1000)   (precomputed)
    target_boundary_m      # = (UNION_M.buffer(AVOID_KM*1000)).boundary  (precomputed)
):
    """
    若點位於「近岸禁入緩衝」(inner ring) 之內，將其推到「外圈緩衝」(target ring) 的邊界上。
    使用預先計算好的圖層，避免每次都 buffer，速度快很多。
    回傳: (新座標, 是否有移動)
    """
    px, py = to_m(pt_ll[0], pt_ll[1])
    p_m = Point(px, py)

    if inner_ring_m.contains(p_m):
        q_m = shapely.ops.nearest_points(p_m, target_boundary_m)[1]
        q_llx, q_lly = to_ll(q_m.x, q_m.y)
        return (q_llx, q_lly), True
    return pt_ll, False

# ---- Debug 幫手 ------
def create_lvs_progress():
    """
    用於 Jupyter 中斷後仍能畫圖的進度物件。
    - candidate_path: 當輪 A* 的候選節點索引序列（list[int]）
    - free_prefix_len: 當輪候選路徑中，從起點起連續通過可視檢查（FREE）的邊數量
    - free_edges: 累計所有輪已被標記 FREE 的邊 (u,v)（可重複；畫圖前會過濾處理）
    - nodes_ref: 目前 nodes 的參考（list[(lon,lat)]）
    - iter: 當前 LVS 的迭代次數（純資訊）
    """
    return {
        "candidate_path": None,
        "free_prefix_len": 0,
        "free_edges": [],
        "nodes_ref": None,
        "iter": 0,
    }


# ---------------- Visibility (with STRtree fast prefilter) ----------------
def visible(a, b, COLLISION_PREP_M, land_tree: Optional[STRtree]=None) -> bool:
    # Build geodesic polyline and test against collision buffer (metric space)
    ls_ll = LineString(geodesic_sample(a, b, step_km=STEP_KM_GEODESIC))
    ls_m  = to_metric(ls_ll)

    if land_tree is not None:
        # coarse prefilter: only check parts whose bbox intersects ls
        candidates = land_tree.query(ls_m)
        if len(candidates) == 0:
            return True  # no land around; safe
        # quick reject by buffered line vs. nothing? We'll still ask COLLISION_PREP_M.
    return not COLLISION_PREP_M.intersects(ls_m)

# ---------------- Feature extraction (from your second script, no plotting) ----------------
def _bearing_deg(a, b):
    ax, ay = a; bx, by = b
    return (math.degrees(math.atan2(by - ay, bx - ax)) + 360.0) % 360.0

def _angdiff(a, b):
    d = (a - b + 540.0) % 360.0 - 180.0
    return d

def _resample_linestring_m(ls_m: LineString, step_m: float) -> LineString:
    L = ls_m.length
    if L == 0:
        return ls_m
    n = max(4, int(round(L / step_m)))
    d = L / n
    pts = [ls_m.interpolate(i * d) for i in range(n)]
    if pts[0].distance(pts[-1]) > 1e-6:
        pts.append(pts[0])
    return LineString(pts)

def _local_maxima(seq, radius):
    n = len(seq)
    peaks = []
    for i in range(radius, n - radius):
        v = seq[i]
        if all(v > seq[i - k] for k in range(1, radius + 1)) and \
           all(v >= seq[i + k] for k in range(1, radius + 1)):
            peaks.append(i)
    return peaks

def extract_convex_peaks_from_buffer(
    union_ll,
    avoid_km=AVOID_KM,
    resample_step_m=300.0,
    window_km=8.0,
    peak_radius_pts=3,
    min_turn_deg=18.0,
    dedup_m=1200.0
):
    union_m = to_metric(union_ll)
    buf_m   = union_m.buffer(avoid_km * 1000.0)

    if buf_m.geom_type == "Polygon":
        polys_m = [buf_m]
    else:
        polys_m = [p for p in buf_m.geoms if p.geom_type == "Polygon"]

    out_pts = []
    for poly in polys_m:
        ring_raw = LineString(list(poly.exterior.coords))
        ring     = _resample_linestring_m(ring_raw, resample_step_m)
        coords   = list(ring.coords)
        n        = len(coords)
        if n < 8:
            continue

        ccw = Polygon(coords).exterior.is_ccw
        W = max(1, int(round(window_km * 1000.0 / resample_step_m)))
        dturn = [0.0] * n
        for i in range(W, n - W):
            h1 = _bearing_deg(coords[i - W], coords[i])
            h2 = _bearing_deg(coords[i],     coords[i + W])
            dturn[i] = _angdiff(h2, h1)

        score = [max(0.0, v) if ccw else max(0.0, -v) for v in dturn]
        peaks = _local_maxima(score, peak_radius_pts)
        peaks = [i for i in peaks if score[i] >= min_turn_deg]

        kept = []
        for i in sorted(peaks, key=lambda j: -score[j]):
            pi = Point(coords[i])
            if all(pi.distance(Point(coords[k])) > dedup_m for k in kept):
                kept.append(i)

        for i in kept:
            x, y = coords[i]
            lon, lat = to_ll(x, y)
            out_pts.append((lon, lat))
    return out_pts

def _dedup_points_geom(pts: List[Point], tol_m: float) -> List[Point]:
    out=[]
    for p in pts:
        keep=True
        for q in out:
            if p.distance(q) <= tol_m:
                keep=False; break
        if keep: out.append(p)
    return out

def _cum_lengths(coords):
    L=[0.0]
    for (x1,y1),(x2,y2) in zip(coords, coords[1:]):
        L.append(L[-1] + math.hypot(x2-x1, y2-y1))
    return L

def _index_at_arclen(L, idx, s):
    target_back = L[idx] - s
    i_back = idx
    while i_back > 0 and L[i_back-1] > target_back:
        i_back -= 1
    target_fwd = L[idx] + s
    i_fwd = idx
    n = len(L) - 1
    while i_fwd < n and L[i_fwd+1] < target_fwd:
        i_fwd += 1
    return i_back, i_fwd

def _angle_and_prominence(a,b,c):
    ax, ay = a; bx, by = b; cx, cy = c
    v1x, v1y = ax - bx, ay - by
    v2x, v2y = cx - bx, cy - by
    n1 = math.hypot(v1x, v1y); n2 = math.hypot(v2x, v2y)
    if n1 == 0 or n2 == 0:
        return 180.0, 0.0
    cosang = max(-1.0, min(1.0, (v1x*v2x + v1y*v2y) / (n1*n2)))
    ang = math.degrees(math.acos(cosang))
    vx, vy = cx - ax, cy - ay
    vlen = math.hypot(vx, vy)
    if vlen == 0:
        prom = 0.0
    else:
        abx, aby = bx - ax, by - ay
        cross = abs(abx * vy - aby * vx)
        prom = cross / (vlen * vlen)
    return ang, prom

def score_convex_concave_on_ring(
    ring_ls_m: LineString,
    scales_km=(5, 10, 20, 40),
    angle_convex_max=170.0,
    angle_concave_min=210.0,
    min_prom_convex=0.002,
    min_prom_concave=0.002,
    simplify_before=False, simplify_m=800.0,
    dedup_m=800.0,
):
    ls = ring_ls_m
    if simplify_before:
        ls = ring_ls_m.simplify(simplify_m, preserve_topology=False)
        if not ls.is_ring:
            ls = LineString(list(ls.coords) + [ls.coords[0]])

    coords = list(ls.coords)
    n = len(coords)
    if n < 5:
        return [], [], coords

    ccw = Polygon(coords).exterior.is_ccw
    L = _cum_lengths(coords)
    scales_m = [s * 1000.0 for s in scales_km]
    min_angle = [180.0] * n
    max_prom  = [0.0]   * n

    for i in range(n):
        for s in scales_m:
            i_back, i_fwd = _index_at_arclen(L, i, s)
            if i_back == i or i_fwd == i:
                continue
            a = coords[i_back]; b = coords[i]; c = coords[i_fwd]
            ang, prom = _angle_and_prominence(a, b, c)
            if ang < min_angle[i]:
                min_angle[i] = ang
            if prom > max_prom[i]:
                max_prom[i] = prom

    convex_idx, concave_idx = [], []
    for i in range(1, n - 1):
        a = coords[i - 1]; b = coords[i]; c = coords[i + 1]
        v1x, v1y = a[0] - b[0], a[1] - b[1]
        v2x, v2y = c[0] - b[0], c[1] - b[1]
        cross = v1x * v2y - v1y * v2x
        is_concave = (cross < 0) if ccw else (cross > 0)
        ang  = min_angle[i]
        prom = max_prom[i]

        if not is_concave:
            if ang < angle_convex_max and prom >= min_prom_convex:
                convex_idx.append(i)
        else:
            if ang > angle_concave_min and prom >= min_prom_concave:
                concave_idx.append(i)

    def _dedup_by_spacing(idxs):
        kept = []
        taken = [False] * n
        for i in sorted(idxs, key=lambda j: -max_prom[j]):
            if taken[i]:
                continue
            kept.append(i)
            xi, yi = coords[i]
            for j in range(n):
                if taken[j]:
                    continue
                xj, yj = coords[j]
                if math.hypot(xj - xi, yj - yi) <= dedup_m:
                    taken[j] = True
        return kept

    convex_idx  = _dedup_by_spacing(convex_idx)
    concave_idx = _dedup_by_spacing(concave_idx)
    return convex_idx, concave_idx, coords

def extract_feature_points_bbox(
    shp_path: Path,
    bbox_ll_polygon: Polygon,
    avoid_km=AVOID_KM,
    simplify_m=1000.0,
    ANGLE_CONVEX_MAX=170.0,
    ANGLE_CONCAVE_MIN=210.0,
    MIN_PROM_CONVEX=0.002,
    MIN_PROM_CONCAVE=0.002,
    DEDUP_CONVEX_M=800.0,
    DEDUP_CONCAVE_M=800.0,
    ENABLE_UNIFORM=False,
    TARGET_SPACING_KM=25.0,
    N_MIN=8, N_MAX=64,
    PERIM_MIN_KM=20.0,
    AREA_MIN_KM2=5.0,
) -> Dict[str, List[Tuple[float,float]]]:
    # 1) clip land by bbox
    polys=[]
    with fiona.open(shp_path) as src:
        for feat in src:
            g = shape(feat["geometry"])
            if g.is_empty: 
                continue
            if g.intersects(bbox_ll_polygon):
                gi = g.intersection(bbox_ll_polygon)
                if not gi.is_empty:
                    polys.append(gi)
    if not polys:
        return {"convex":[], "concave":[], "uniform":[], "convex_peaks":[]}

    union_ll = unary_union(polys)  # lon/lat
    union_m  = to_metric(union_ll)
    buf_m    = union_m.buffer(avoid_km * 1000.0)

    if buf_m.geom_type == "Polygon":
        polys_m = [buf_m]
    else:
        polys_m = [p for p in buf_m.geoms if p.geom_type == "Polygon"]

    convex_pts_m, concave_pts_m, uniform_pts_m = [], [], []
    for poly in polys_m:
        perim_km = poly.exterior.length / 1000.0
        area_km2 = poly.area / 1e6
        if perim_km < PERIM_MIN_KM or area_km2 < AREA_MIN_KM2:
            continue
        ring_raw = shapely.LineString(poly.exterior.coords)
        ring_s   = ring_raw.simplify(simplify_m, preserve_topology=False)
        if not ring_s.is_ring:
            ring_s = shapely.LineString(list(ring_s.coords)+[ring_s.coords[0]])
        convex_idx, concave_idx, coords_used = score_convex_concave_on_ring(
            ring_s,
            scales_km=(5,10,20,40),
            angle_convex_max=ANGLE_CONVEX_MAX,
            angle_concave_min=ANGLE_CONCAVE_MIN,
            min_prom_convex=MIN_PROM_CONVEX,
            min_prom_concave=MIN_PROM_CONCAVE,
            simplify_before=False,
            dedup_m=max(DEDUP_CONVEX_M, DEDUP_CONCAVE_M),
        )
        for i in convex_idx:
            x,y = coords_used[i]; convex_pts_m.append(Point(x,y))
        for i in concave_idx:
            x,y = coords_used[i]; concave_pts_m.append(Point(x,y))

        if ENABLE_UNIFORM:
            n_uniform = max(N_MIN, min(N_MAX, int(perim_km / TARGET_SPACING_KM)))
            L = ring_raw.length
            for i in range(n_uniform):
                uniform_pts_m.append(ring_raw.interpolate(i * L / n_uniform))

    # convex peaks (direction extrema) using original union_ll
    convex_peaks_ll = extract_convex_peaks_from_buffer(
        union_ll=union_ll,
        avoid_km=avoid_km,
        resample_step_m=300.0,
        window_km=8.0,
        peak_radius_pts=3,
        min_turn_deg=18.0,
        dedup_m=1200.0
    )

    # dedup (metric)
    convex_pts_m  = _dedup_points_geom(convex_pts_m,  DEDUP_CONVEX_M)
    concave_pts_m = _dedup_points_geom(concave_pts_m, DEDUP_CONCAVE_M)
    if ENABLE_UNIFORM:
        uniform_pts_m = _dedup_points_geom(uniform_pts_m, 1500.0)

    def _to_ll_list(pts):
        out=[]
        for p in pts:
            lon,lat = to_ll(p.x,p.y)
            out.append((lon,lat))
        return out

    convex_ll   = _to_ll_list(convex_pts_m)
    concave_ll  = _to_ll_list(concave_pts_m)
    uniform_ll  = _to_ll_list(uniform_pts_m) if ENABLE_UNIFORM else []

    return {
        "convex": convex_ll,
        "concave": concave_ll,
        "uniform": uniform_ll,
        "convex_peaks": convex_peaks_ll
    }

# ---------------- Lazy Visibility Search Graph ----------------
def rotation_cost(seq: List[Tuple[float,float]]) -> float:
    rot=0.0
    for a,b,c in zip(seq[:-2], seq[1:-1], seq[2:]):
        h1 = bearing_xy(a,b); h2 = bearing_xy(b,c)
        rot += abs(angle_diff(h2,h1))
    return rot

def edge_cost(a,b, use_rot=False) -> float:
    # base = geodesic km; optional small rotation penalty is handled path-wise, not per single edge
    return gc_distance_km(a,b)

def heuristic(p, D):
    return gc_distance_km(p, D)

def neighbors_of(u_idx: int, nodes: List[Tuple[float,float]], D_idx: int, k=NEIGHBOR_K) -> List[int]:
    # choose k best forward-movers by distance to D (and also closeness to u to avoid far jumps)
    u = nodes[u_idx]; D = nodes[D_idx]
    # Score: distance to u (prefer closer) + (distance to D from v)  (lower better)
    # We'll preselect by distance-to-u to keep locality, but always include D.
    # Compute distances lazily:
    idxs = range(len(nodes))
    scored = []
    for v_idx in idxs:
        if v_idx == u_idx: 
            continue
        # ensure we don't explode; we filter later by lazy validation
        du = gc_distance_km(u, nodes[v_idx])
        dD = gc_distance_km(nodes[v_idx], D)
        score = du + 0.5*dD
        scored.append((score, v_idx))
    scored.sort(key=lambda t:t[0])
    out = [v for _,v in itertools.islice(scored, 0, k)]
    if D_idx not in out:
        out.append(D_idx)
    return out

def lazy_visibility_search(
    nodes: List[Tuple[float,float]],
    O_idx: int,
    D_idx: int,
    visible_fn,
    COLLISION_PREP_M,
    land_tree: Optional[STRtree],
    inject_gateways_fn,
    max_iters: int = 5000,
    progress: Optional[Dict]=None,   # ←① 加上參數
):
    """
    LazySP：每輪先用樂觀邊跑 A*，再逐邊做可視檢查。
    ★ 會在 Jupyter 直接印出目前檢查到的經緯度點位。
    """
    EDGE_STATE: Dict[Tuple[int,int], str] = {}  # 'FREE'|'BLOCKED'
    adj_cache: Dict[int, List[int]] = {}

    # ② progress 初始化
    if progress is None:
        progress = {"iter": 0, "candidate_path": [], "free_prefix_len": 0,
                    "free_edges": [], "nodes_ref": nodes}
    else:
        progress["nodes_ref"] = nodes

    def get_neighbors(u: int) -> List[int]:
        if u not in adj_cache:
            adj_cache[u] = neighbors_of(u, nodes, D_idx, k=NEIGHBOR_K)
        return adj_cache[u]

    def a_star() -> Optional[List[int]]:
        N_local = len(nodes)
        open_heap = []
        INF = 1e18
        g = [INF] * N_local
        parent = [-1] * N_local

        g[O_idx] = 0.0
        h0 = heuristic(nodes[O_idx], nodes[D_idx])
        heapq.heappush(open_heap, (g[O_idx] + h0, O_idx))
        closed = set()

        while open_heap:
            _, u = heapq.heappop(open_heap)
            if u in closed:
                continue
            if u == D_idx:
                path = [u]
                while parent[u] != -1:
                    u = parent[u]
                    path.append(u)
                path.reverse()
                return path

            closed.add(u)
            for v in get_neighbors(u):
                if v >= len(nodes):
                    continue
                if EDGE_STATE.get((u, v)) == 'BLOCKED':
                    continue

                c = edge_cost(nodes[u], nodes[v], USE_ROTATION_PENALTY)

                if v >= len(g):
                    extend_by = v + 1 - len(g)
                    g.extend([INF] * extend_by)
                    parent.extend([-1] * extend_by)

                alt = g[u] + c
                if alt < g[v]:
                    g[v] = alt
                    parent[v] = u
                    f = alt + heuristic(nodes[v], nodes[D_idx])
                    heapq.heappush(open_heap, (f, v))
        return None

    it = 0
    while it < max_iters:
        it += 1
        path = a_star()
        if not path:
            print("[LVS] 找不到候選路徑（圖被 BLOCKED 邊切斷）", flush=True)
            raise RuntimeError("LVS: path not found.")

        # === 插入點 A：更新本輪進度快照 ===
        progress["iter"] = it
        progress["candidate_path"] = list(path)
        progress["free_prefix_len"] = 0

        # 候選摘要
        cand_nodes = [nodes[idx] for idx in path]
        print(f"[LVS] iter {it:03d} | 候選路徑節點數 = {len(cand_nodes)}", flush=True)
        print(f"         節點總數 = {len(nodes)}", flush=True)
        _print_ll("  候選起點", cand_nodes[0])
        _print_ll("  候選終點", cand_nodes[-1])

        # 逐邊驗證
        all_valid = True
        prefix_ok = 0
        for u, v in zip(path[:-1], path[1:]):
            st = EDGE_STATE.get((u, v))
            a = nodes[u]; b = nodes[v]

            if st == 'FREE':
                _print_edge("  檢查邊(快取OK)", a, b)
                prefix_ok += 1
                progress["free_edges"].append((u, v))
                continue

            if st == 'BLOCKED':
                _print_edge("  檢查邊(快取BLOCKED)", a, b)
                all_valid = False
                break

            _print_edge("  檢查邊", a, b)
            if visible_fn(a, b, COLLISION_PREP_M, land_tree):
                EDGE_STATE[(u, v)] = EDGE_STATE[(v, u)] = 'FREE'
                print("    → OK", flush=True)
                prefix_ok += 1
                progress["free_edges"].append((u, v))
            else:
                EDGE_STATE[(u, v)] = EDGE_STATE[(v, u)] = 'BLOCKED'
                print("    → BLOCKED", flush=True)

                # 被擋：注入 gateway 節點
                new_nodes = inject_gateways_fn(a, b)
                if new_nodes:
                    existing = set((round(lon,5), round(lat,5)) for (lon,lat) in nodes)
                    filtered = []
                    for q in new_nodes:
                        key = (round(q[0],5), round(q[1],5))
                        if key not in existing:
                            filtered.append(q)
                            existing.add(key)
                    if filtered:
                        print(f"    新增節點 {len(filtered)} 個：", flush=True)
                        for q in filtered:
                            _print_ll("      +", q)
                        # ⚠ 正確縮排：extend/clear 在 for 迴圈外
                        nodes.extend(filtered)
                        adj_cache.clear()
                    else:
                        print("    無新節點可加入（皆為重複點）", flush=True)

                all_valid = False
                break  # 停止本輪驗證，回去重規劃

        # 回寫本輪已通過的前綴長度（給中斷時畫圖用）
        progress["free_prefix_len"] = prefix_ok

        if all_valid:
            print("[LVS] 成功：路徑全通過驗證，最終節點序列：", flush=True)
            for i, idx in enumerate(path):
                _print_ll(f"  [{i:02d}]", nodes[idx])
            return path

    raise RuntimeError("LVS: 超過最大迭代次數仍未找到有效路徑")





# ---------------- Gateway injection (simple + robust) ----------------
def make_inject_gateways_fn(
    UNION_M,
    features_index_ll,
    take_each=3,
    inner_ring_m=None,
    target_boundary_m=None
):
    pool = list(features_index_ll.get("convex_peaks", [])) + list(features_index_ll.get("convex", []))

    def f(u_ll, v_ll):
        if not pool:
            return []

        seg_ll = LineString(geodesic_sample(u_ll, v_ll, step_km=STEP_KM_GEODESIC))
        seg_m  = to_metric(seg_ll)

        # 以距離這條被擋邊的最近度排序
        cand = []
        for (lon, lat) in pool:
            px, py = to_m(lon, lat)
            p = Point(px, py)
            cand.append((seg_m.distance(p), (lon, lat)))
        cand.sort(key=lambda t: t[0])

        # 先取前 take_each 個候選，推離岸
        new = []
        for _, pt in itertools.islice(cand, 0, take_each):
            q, _moved = nudge_to_ring_if_inside_fast(pt, inner_ring_m, target_boundary_m)
            new.append(q)

        # 1) 本批去重
        out = []
        seen_local = set()
        for lon, lat in new:
            key = (round(lon, 5), round(lat, 5))
            if key in seen_local:
                continue
            seen_local.add(key)
            out.append((lon, lat))

        # 2) 與既有 nodes 去重（關鍵）
        # 這裡從閉包外讀不到 nodes，所以改成在 lazy_visibility_search() 裡過濾（見下方補丁）
        return out

    return f



# ---------------- Utility: convert land geom to Pacific view for folium ----------------
def convert_geom_to_pacific(geom):
    from shapely.geometry import mapping
    geom_dict = mapping(geom)
    def convert_coords(coords):
        if isinstance(coords[0], (list, tuple)):
            return [convert_coords(c) for c in coords]
        else:
            lon, lat = coords[0], coords[1]
            lon_pacific = normalize_lon_to_pacific_view(lon)
            return [lon_pacific, lat]
    if geom_dict['type'] == 'Polygon':
        geom_dict['coordinates'] = [convert_coords(ring) for ring in geom_dict['coordinates']]
    elif geom_dict['type'] == 'MultiPolygon':
        geom_dict['coordinates'] = [[convert_coords(ring) for ring in poly] for poly in geom_dict['coordinates']]
    return geom_dict

# ---------------- Orchestrator ----------------
def plan_route(
    origin: Tuple[float,float],
    dest: Tuple[float,float],
    land_path: Path = LAND_PATH,
    out_html: str = r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA\route_pacific_LVS.html",
    add_feature_layer: bool = True,
):
    print(f"起點: {origin}")
    print(f"終點: {dest}")
    print(f"太平洋視角: 起點={normalize_lon_to_pacific_view(origin[0]):.2f}°, "
          f"終點={normalize_lon_to_pacific_view(dest[0]):.2f}°")

    # 1) Load land inside IDL-aware bboxes
    bboxes = dynamic_bboxes_idl(origin, dest, pad_deg=PAD_DEG)
    polys  = load_polys_in_bboxes(land_path, bboxes)
    assert len(polys)>0, "No land polygons found in bbox"
    layers = build_land_layers(polys)
    UNION_M          = layers["UNION_M"]
    COLLISION_PREP_M = layers["COLLISION_PREP_M"]
    COLLISION_WGS    = layers["COLLISION_WGS"]
    RING_WGS         = layers["RING_WGS"]
    land_raw_wgs     = layers["LAND_RAW_WGS"]
    LAND_PARTS_M     = layers["LAND_PARTS_M"]

    INNER_RING_M = layers["RING_M"]  # 5km 內圈（已在 build_land_layers 算過）
    TARGET_RING_M = UNION_M.buffer(AVOID_KM * 1000.0)   # 15km 外圈（只算一次！）
    TARGET_BOUNDARY_M = TARGET_RING_M.boundary

    land_tree = build_land_strtree(LAND_PARTS_M)

    # 2) Nudge O/D
    origin_adj, moved_o = nudge_to_ring_if_inside_fast(origin, INNER_RING_M, TARGET_BOUNDARY_M)
    dest_adj,   moved_d = nudge_to_ring_if_inside_fast(dest,   INNER_RING_M, TARGET_BOUNDARY_M)

    if moved_o:
        _print_edge("[INFO] 起點在 5km 緩衝內 → 已推到 15km 外圈", origin, origin_adj)
    if moved_d:
        _print_edge("[INFO] 終點在 5km 緩衝內 → 已推到 15km 外圈", dest, dest_adj)

    # 3) Extract features for the whole bbox union
    bbox_union_ll = union_lonlat_bboxes(bboxes)
    feat = extract_feature_points_bbox(
        shp_path=land_path,
        bbox_ll_polygon=bbox_union_ll,
        avoid_km=AVOID_KM,
        simplify_m=1000.0,
        ANGLE_CONVEX_MAX=170.0,
        ANGLE_CONCAVE_MIN=210.0,
        MIN_PROM_CONVEX=0.002,
        MIN_PROM_CONCAVE=0.002,
        ENABLE_UNIFORM=False,  # per your choice
        PERIM_MIN_KM=20.0,
        AREA_MIN_KM2=5.0
    )
    # feature combo per your decision: convex + convex_peaks
    feature_nodes = list(feat["convex_peaks"]) + list(feat["convex"])

    # 4) Build initial node list and nudge them all
    base_nodes = [origin_adj, dest_adj] + feature_nodes
    nodes=[]
    for p in base_nodes:
        q,_ = nudge_to_ring_if_inside_fast(p, INNER_RING_M, TARGET_BOUNDARY_M)
        nodes.append(q)
    if len(nodes) > LVS_MAX_NODES:
        print(f"[WARN] nodes truncated from {len(nodes)} to {LVS_MAX_NODES}")
        nodes = nodes[:LVS_MAX_NODES]
    O_idx=0; D_idx=1

    # 5) Make injection function for blocked edges
    inject_fn = make_inject_gateways_fn(
        UNION_M,
        {"convex_peaks": feat["convex_peaks"], "convex": feat["convex"]},
        take_each=3,
        inner_ring_m=INNER_RING_M,
        target_boundary_m=TARGET_BOUNDARY_M
    )

    # 6) Run Lazy Visibility Search
    def visible_wrapper(a,b, COLLISION_PREP_M, land_tree):
        return visible(a,b, COLLISION_PREP_M, land_tree)

    print("\n開始 Lazy Visibility Search (LVS)...")
    progress = {"iter":0, "candidate_path":[], "free_prefix_len":0, "free_edges":[], "nodes_ref": None}
    path_idx = lazy_visibility_search(
        nodes, O_idx, D_idx, visible_wrapper, COLLISION_PREP_M, land_tree, inject_fn, max_iters=5000, progress=progress
    )
    print(f"[OK] LVS path with {len(path_idx)} nodes")

    # 7) Build Folium (Pacific view)
    mid_lon, mid_lat = great_circle_midpoint(origin, dest)
    center_lon_pacific = normalize_lon_to_pacific_view(mid_lon)
    center_lat = mid_lat

    m = folium.Map(
        location=[center_lat, center_lon_pacific],
        zoom_start=3,
        max_bounds=False, world_copy_jump=False, no_wrap=False, min_lon=0, max_lon=360
    )
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Ocean_Basemap/MapServer/tile/{z}/{y}/{x}',
        attr='Esri', name='Ocean Basemap', overlay=False, control=True, no_wrap=False
    ).add_to(m)
    folium.TileLayer(
        tiles='https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',
        attr='© OpenStreetMap', name='OpenStreetMap', overlay=False, control=True, no_wrap=False
    ).add_to(m)

    # Land + 5km ring
    folium.GeoJson(
        convert_geom_to_pacific(land_raw_wgs),
        name="陸地",
        style_function=lambda x: {"color":"#2ca02c","weight":1,"fillOpacity":0.15}
    ).add_to(m)
    folium.GeoJson(
        convert_geom_to_pacific(RING_WGS),
        name=f"航道緩衝區 {BUFFER_KM}km",
        style_function=lambda x: {"color":"#6a5acd","weight":2,"fillOpacity":0.05}
    ).add_to(m)

    # Markers: O/D
    origin_pacific = [origin[1], normalize_lon_to_pacific_view(origin[0])]
    dest_pacific   = [dest[1],   normalize_lon_to_pacific_view(dest[0])]
    folium.Marker(
        origin_pacific,
        tooltip=f"起點: <br>({origin[0]:.2f}°, {origin[1]:.2f}°)",
        icon=folium.Icon(color='green', icon='ship', prefix='fa')
    ).add_to(m)
    folium.Marker(
        dest_pacific,
        tooltip=f"終點: <br>({dest[0]:.2f}°, {dest[1]:.2f}°)",
        icon=folium.Icon(color='red', icon='anchor', prefix='fa')
    ).add_to(m)

    # reference great-circle
    draw_gc_polyline_continuous(
        m, origin, dest, step_km=80.0,
        color='gray', weight=2, opacity=0.4, dash_array="8,4"
    )

    # Candidate feature nodes layer (for debugging)
    if add_feature_layer:
        fg_feat = folium.FeatureGroup(name="候選特徵點（凸峰+凸點）", show=False)
        for (lon,lat) in feature_nodes:
            folium.CircleMarker(
                [lat, normalize_lon_to_pacific_view(lon)], radius=3,
                color="#1f77b4", fill=True, fill_opacity=0.8,
                tooltip=f"Feature ({lon:.3f},{lat:.3f})"
            ).add_to(fg_feat)
        fg_feat.add_to(m)

    # 8) Draw final path as continuous blue GC segments
    final_segments=[]
    # (A) 若起點被推移，先把 origin→origin_adj 這段納入與繪圖
    if moved_o and origin != origin_adj:
        final_segments.append((origin, origin_adj))
        draw_gc_polyline_continuous(
            m, origin, origin_adj, step_km=DRAW_STEP_KM,
            color='#1f77b4', weight=5, opacity=0.9
        )

    # (B) 主路徑（origin_adj … dest_adj）
    for u, v in zip(path_idx[:-1], path_idx[1:]):
        a = nodes[u]; b = nodes[v]
        final_segments.append((a, b))
        draw_gc_polyline_continuous(
            m, a, b, step_km=DRAW_STEP_KM,
            color='#1f77b4', weight=5, opacity=0.9
        )

    # (C) 若終點被推移，最後把 dest_adj→dest 也納入與繪圖
    if moved_d and dest_adj != dest:
        final_segments.append((dest_adj, dest))
        draw_gc_polyline_continuous(
            m, dest_adj, dest, step_km=DRAW_STEP_KM,
            color='#1f77b4', weight=5, opacity=0.9
        )
    # 9) Waypoint markers (exclude ends)
    for i, idx in enumerate(path_idx[1:-1], 1):
        wp = nodes[idx]
        wp_pacific = [wp[1], normalize_lon_to_pacific_view(wp[0])]
        folium.CircleMarker(
            wp_pacific, radius=4,
            tooltip=f"航點 {i}<br>({wp[0]:.2f}°, {wp[1]:.2f}°)",
            color='blue', fill=True, fill_opacity=0.8
        ).add_to(m)

    # 存檔 & 總結
    folium.LayerControl(collapsed=False).add_to(m)
    m.save(out_html)

    total_km = sum(gc_distance_km(a, b) for (a, b) in final_segments)

    print(f"\n已儲存至: {out_html}")
    print(f"總航程: {total_km:.1f} 公里")
    print(f"節點數: {len(nodes)} / 路徑節點: {len(path_idx)}")

# 額外資訊：推移距離（若有）
    if moved_o:
        print(f"[INFO] 起點推移距離 ≈ {gc_distance_km(origin, origin_adj):.2f} km")
    if moved_d:
        print(f"[INFO] 終點推移距離 ≈ {gc_distance_km(dest, dest_adj):.2f} km")



# ---------------- Example call (uncomment to run) ----------------
if __name__ == "__main__":
    # Example ports (Tokyo -> Los Angeles), adjust as you like:
    # origin = (139.8268, 35.61168)
    # dest   = (-118.265, 33.74021)

    # Taiwan → Japan sample (you can replace):
    origin = (129.033333,35.116667)   
    dest   = (113.8535,22.5045)   
    #| KAOHSIUNG  | TWKHH     | (22.58425,120.3181)  | (120.3181,22.58425)            |
    #| TAICHUNG   | TWTXG     | (24.255,120.5075)    | (120.5075,24.255)              |
    #| KEELUNG    | TWKEL     | (25.14515,121.7532)  | (121.7532,25.14515)            |
    #| SHANGHAI  | CNSHG     | (31.36636,121.6147) | (121.6147,31.36636)            |
    #| NINGBO    | CNNBG     | (29.92654,121.8525) | (121.8525,29.92654)            |
    #| ZHOUSHAN  | CNZOS     | (29.92161,122.2104) | (122.2104,29.92161)            |
    #| SHENZHEN  | CNSZX     | (22.5045,113.8535)  | (113.8535,22.5045)             |
    #| LOS ANGELES | USLAX     | (33.74021,-118.265) | (-118.265,33.74021)            |
    #| SEATTLE     | USSEA     | (47.6212,-122.3643) | (-122.3643,47.6212)            |
    #| TOKYO     | JPTYO     | (35.61168,139.8268) | (139.8268,35.61168)            |
    #| KOBE      | JPUKB     | (34.6867,135.2671)  | (135.2671,34.6867)             |
    #| WAKAYAMA  | JPUKW     | (33.468056,135.828056) | (135.828056,33.468056)       |
    #| BUSAN     | KRPUS     | (35.116667,129.033333) | (129.033333,35.116667)       |
    #|  Sihanoukville| KHKOS  |                      |   (103.5, 10.633333)       |

    plan_route(origin, dest)


KeyboardInterrupt: 

In [ ]:
try:
    plan_route(origin, dest)  # 你本來的呼叫
except KeyboardInterrupt:
    # 這裡用你在 plan_route 中可取得的物件來畫（建議把需要的東西暴露出來或存在全域/外層）
    draw_progress(
        progress=progress,         # 傳入 lazy_visibility_search 使用的 progress 物件
        nodes=nodes,               # 同一份 nodes
        origin=origin, 
        dest=dest,
        out_html=r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA\route_progress.html",
        land_geom=land_raw_wgs,    # 若不方便取到，傳 None 就好
        ring_geom=RING_WGS,
        feature_nodes=feature_nodes,   # 若要一起看
        show_features=True
    )


起點: (120.3181, 22.58425)
終點: (103.5, 10.633333)
太平洋視角: 起點=120.32°, 終點=103.50°
[INFO] 起點在 5km 緩衝內 → 已推到 15km 外圈 (120.31810, 22.58425) -> (120.18383, 22.49683)
[INFO] 終點在 5km 緩衝內 → 已推到 15km 外圈 (103.50000, 10.63333) -> (103.44603, 10.48629)

開始 Lazy Visibility Search (LVS)...
[LVS] iter 001 | 候選路徑節點數 = 2
         節點總數 = 3883
  候選起點 (120.18383, 22.49683)
  候選終點 (103.44603, 10.48629)
  檢查邊 (120.18383, 22.49683) -> (103.44603, 10.48629)
    → BLOCKED
    無新節點可加入（皆為重複點）
[LVS] iter 002 | 候選路徑節點數 = 3
         節點總數 = 3883
  候選起點 (120.18383, 22.49683)
  候選終點 (103.44603, 10.48629)
  檢查邊 (120.18383, 22.49683) -> (120.21215, 22.35222)
    → OK
  檢查邊 (120.21215, 22.35222) -> (103.44603, 10.48629)
    → BLOCKED
    無新節點可加入（皆為重複點）
[LVS] iter 003 | 候選路徑節點數 = 3
         節點總數 = 3883
  候選起點 (120.18383, 22.49683)
  候選終點 (103.44603, 10.48629)
  檢查邊 (120.18383, 22.49683) -> (120.21064, 22.33983)
    → OK
  檢查邊 (120.21064, 22.33983) -> (103.44603, 10.48629)
    → BLOCKED
    無新節點可加入（皆為重複點）
[LVS] iter 004 | 候選路

RuntimeError: LVS: path not found.

In [ ]:
# === Lazy Visibility Search (Pacific view, great-circle, full-bbox features) ===
# Author: you + ChatGPT
# Note:
#  - Keep your original geodesy, Pacific-view drawing, land buffering & collision layers.
#  - Replace "outer tangent detour" with a Lazy Visibility Search (LVS) graph.
#  - Nodes = {O, D} ∪ {bbox features}; Edges are only validated lazily by visible().
#  - When a candidate shortest path has an invalid edge, mark it BLOCKED (and optionally inject gateways),
#    then replan until a fully valid path is found.
#
# Usage:
#   1) Set origin, dest (lon,lat) below (or call plan_route(...))
#   2) Run the script; it outputs a Folium HTML with Pacific-centered continuous arcs.

from __future__ import annotations
from pathlib import Path
import math
import heapq
import itertools
from typing import List, Tuple, Dict, Optional, Iterable

import fiona
import folium
from networkx import nodes
import shapely
from shapely.geometry import shape, Polygon, Point, LineString, GeometryCollection
from shapely.ops import unary_union
from shapely.prepared import prep
from shapely.strtree import STRtree
from pyproj import Transformer, Geod

# ---------------- Params (edit as needed) ----------------
LAND_PATH   = Path(r"C:\Users\slab\Desktop\Slab Project\Stage1\data\Land\ne_10m_land.shp")
BUFFER_KM   = 5.0                    # inner "no-go" near coast; points inside will be nudged outward
COLLISION_SAFETY_KM = 0.25           # collision dilation for visibility (rigid safety margin)
PAD_DEG     = 6.0                    # bbox padding in degrees
STEP_KM_GEODESIC = 3.0               # great-circle sampling resolution for visibility
DRAW_STEP_KM = 20.0                  # great-circle sampling for drawing
AVOID_KM    = 15.0                   # outer ring where routes/waypoints are allowed

# LVS knobs
NEIGHBOR_K = 24                      # candidate neighbors per node per expansion (plus D)
LVS_MAX_NODES = 4000                 # guard rail for very large bbox
USE_ROTATION_PENALTY = False         # you asked to keep it OFF initially
ROT_LAMBDA = 0.001                   # weight if enabled

# ---------------- CRS & geodesy ----------------
to_m  = Transformer.from_crs("EPSG:4326","EPSG:3857", always_xy=True).transform
to_ll = Transformer.from_crs("EPSG:3857","EPSG:4326", always_xy=True).transform
def to_metric(g): return shapely.ops.transform(to_m, g)
def to_wgs(g):    return shapely.ops.transform(to_ll, g)
GEOD = Geod(ellps="WGS84")


def _fmt_ll(p):
    return f"({p[0]:.5f}, {p[1]:.5f})"  # (lon, lat)

def _print_ll(msg, p):
    print(f"{msg} {_fmt_ll(p)}", flush=True)

def _print_edge(prefix, a, b):
    print(f"{prefix} {_fmt_ll(a)} -> {_fmt_ll(b)}", flush=True)


def geodesic_sample(a: Tuple[float,float], b: Tuple[float,float], step_km: float=STEP_KM_GEODESIC) -> List[Tuple[float,float]]:
    lon1,lat1=a; lon2,lat2=b
    _,_,dist_m = GEOD.inv(lon1,lat1,lon2,lat2)
    n = max(1,int(dist_m/(step_km*1000)))
    pts = GEOD.npts(lon1,lat1,lon2,lat2,n)
    return [(lon1,lat1)] + pts + [(lon2,lat2)]

def great_circle_midpoint(a,b):
    pts = geodesic_sample(a,b,step_km=500.0)
    return pts[len(pts)//2]

def gc_distance_km(a,b) -> float:
    _,_,d = GEOD.inv(a[0],a[1],b[0],b[1])
    return d/1000.0

def bearing_xy(p,q):
    px,py = to_m(p[0],p[1]); qx,qy = to_m(q[0],q[1])
    return math.degrees(math.atan2(qy-py, qx-px)) % 360.0

def angle_diff(a,b):
    return (a-b+540)%360 - 180

# ---------------- Pacific view helpers ----------------
def normalize_lon_to_pacific_view(lon: float) -> float:
    return lon if lon>=0 else lon+360

def draw_gc_polyline_continuous(m, a, b, step_km=DRAW_STEP_KM, **style):
    pts = geodesic_sample(a, b, step_km=step_km)
    folium_coords = []
    for lon, lat in pts:
        lon_pacific = normalize_lon_to_pacific_view(lon)
        folium_coords.append([lat, lon_pacific])
    folium.PolyLine(folium_coords, **style).add_to(m)

def draw_progress(
    progress: Dict,
    nodes: List[Tuple[float,float]],
    origin: Tuple[float,float],
    dest: Tuple[float,float],
    out_html: str = r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA\route_progress.html",
    land_geom=None,     # 建議傳 land_raw_wgs（可為 None）
    ring_geom=None,     # 建議傳 RING_WGS（可為 None）
    feature_nodes: Optional[List[Tuple[float,float]]] = None,  # 若要疊上候選特徵點群組
    show_features: bool = True,
):
    """
    將目前 LVS 進度畫成地圖：
      - 已驗證通過的邊（含過去輪次的 free_edges）→ 實線藍色
      - 本輪候選路徑中已通過的前綴 → 實線藍色
      - 本輪候選路徑中尚未驗證的後綴 → 橘色虛線
      - 若 O/D 曾被推移（nodes[0] != origin 或 nodes[1] != dest），補畫接駁段

    參數：
      progress: 由 lazy_visibility_search(progress=...) 維護的 dict
      nodes:    與 progress["nodes_ref"] 同一個 nodes（索引一致）
      origin,dest: 原始輸入 O/D（lon,lat）
      land_geom, ring_geom: 供背景展示；可傳 None 跳過
      feature_nodes: 要疊圖的候選特徵點（例如 convex_peaks+convex）
    """
    # ---- 讀進度 ----
    cand_idx = progress.get("candidate_path", []) or []
    k = int(progress.get("free_prefix_len", 0) or 0)
    free_edges_hist = list(progress.get("free_edges", []) or [])
    O_idx, D_idx = 0, 1  # 你的設計固定 0/1

    # 安全防呆
    if not nodes or len(nodes) < 2:
        print("[draw_progress] nodes 不足，無法畫圖")
        return
    if not cand_idx:
        # 沒有候選路徑就至少把 O/D 畫出來
        cand_idx = [O_idx, D_idx]
        k = 0

    # 地圖中心：取 O-D 大圓中點（太平洋視角）
    mid_lon, mid_lat = great_circle_midpoint(origin, dest)
    center_lon_pacific = normalize_lon_to_pacific_view(mid_lon)
    center_lat = mid_lat

    m = folium.Map(
        location=[center_lat, center_lon_pacific],
        zoom_start=4,
        max_bounds=False, world_copy_jump=False, no_wrap=False, min_lon=0, max_lon=360
    )

    # 底圖
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Ocean_Basemap/MapServer/tile/{z}/{y}/{x}',
        attr='Esri', name='Ocean Basemap', overlay=False, control=True, no_wrap=False
    ).add_to(m)
    folium.TileLayer(
        tiles='https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',
        attr='© OpenStreetMap', name='OpenStreetMap', overlay=False, control=True, no_wrap=False
    ).add_to(m)

    # 背景：陸域、5km 緩衝
    if land_geom is not None:
        folium.GeoJson(
            convert_geom_to_pacific(land_geom),
            name="陸地",
            style_function=lambda x: {"color":"#2ca02c","weight":1,"fillOpacity":0.15}
        ).add_to(m)
    if ring_geom is not None:
        folium.GeoJson(
            convert_geom_to_pacific(ring_geom),
            name=f"航道緩衝區 {BUFFER_KM}km",
            style_function=lambda x: {"color":"#6a5acd","weight":2,"fillOpacity":0.05}
        ).add_to(m)

    # O/D 標記
    folium.Marker(
        [origin[1], normalize_lon_to_pacific_view(origin[0])],
        tooltip=f"起點: ({origin[0]:.4f}, {origin[1]:.4f})",
        icon=folium.Icon(color='green', icon='ship', prefix='fa')
    ).add_to(m)
    folium.Marker(
        [dest[1], normalize_lon_to_pacific_view(dest[0])],
        tooltip=f"終點: ({dest[0]:.4f}, {dest[1]:.4f})",
        icon=folium.Icon(color='red', icon='anchor', prefix='fa')
    ).add_to(m)

    # 若 O/D 被推移，補畫接駁段（實線藍）
    O_adj = nodes[O_idx]; D_adj = nodes[D_idx]
    if O_adj != origin:
        draw_gc_polyline_continuous(m, origin, O_adj, step_km=DRAW_STEP_KM,
                                    color='#1f77b4', weight=5, opacity=0.9)
    if D_adj != dest:
        draw_gc_polyline_continuous(m, D_adj, dest, step_km=DRAW_STEP_KM,
                                    color='#1f77b4', weight=5, opacity=0.9)

    # 歷史已驗證 FREE 的邊（跨輪次）：實線藍
    # （有些邊可能不在當前候選路徑上，也照畫，幫助你理解探索過的走向）
    for (u, v) in free_edges_hist:
        if u < len(nodes) and v < len(nodes):
            a, b = nodes[u], nodes[v]
            draw_gc_polyline_continuous(
                m, a, b, step_km=DRAW_STEP_KM,
                color='#1f77b4', weight=4, opacity=0.65
            )

    # 本輪候選路徑：前綴已驗證（實線藍）
    if len(cand_idx) >= 2 and k > 0:
        for u, v in zip(cand_idx[:k], cand_idx[1:k+1]):
            if u < len(nodes) and v < len(nodes):
                a, b = nodes[u], nodes[v]
                draw_gc_polyline_continuous(
                    m, a, b, step_km=DRAW_STEP_KM,
                    color='#1f77b4', weight=6, opacity=0.95
                )

    # 本輪候選路徑：後綴未驗證（橘色虛線）
    if len(cand_idx) >= 2 and k < len(cand_idx) - 1:
        for u, v in zip(cand_idx[k:-1], cand_idx[k+1:]):
            if u < len(nodes) and v < len(nodes):
                a, b = nodes[u], nodes[v]
                draw_gc_polyline_continuous(
                    m, a, b, step_km=DRAW_STEP_KM,
                    color='#ff7f0e', weight=5, opacity=0.9, dash_array="10,6"
                )

    # 候選節點點標（可視化目前 path）
    #  - 前綴點：實心藍圈
    #  - 後綴點：橘圈
    for i, idx in enumerate(cand_idx):
        if idx >= len(nodes): 
            continue
        lon, lat = nodes[idx]
        ll = [lat, normalize_lon_to_pacific_view(lon)]
        if i <= k:
            folium.CircleMarker(ll, radius=4, color='#1f77b4', fill=True, fill_opacity=0.9,
                                tooltip=f"cand[{i}] {_fmt_ll((lon,lat))} (已驗證前綴)").add_to(m)
        else:
            folium.CircleMarker(ll, radius=4, color='#ff7f0e', fill=True, fill_opacity=0.7,
                                tooltip=f"cand[{i}] {_fmt_ll((lon,lat))} (待驗證)").add_to(m)

    # 額外：候選特徵點群組（debug 層）
    if show_features and feature_nodes:
        fg_feat = folium.FeatureGroup(name="候選特徵點（凸峰+凸點）", show=False)
        for (lon, lat) in feature_nodes:
            folium.CircleMarker(
                [lat, normalize_lon_to_pacific_view(lon)], radius=3,
                color="#1f77b4", fill=True, fill_opacity=0.8,
                tooltip=f"Feature ({lon:.3f},{lat:.3f})"
            ).add_to(fg_feat)
        fg_feat.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    m.save(out_html)
    print(f"[draw_progress] 進度地圖已輸出：{out_html}")


# ---------------- IDL-aware dynamic bboxes ----------------
def dynamic_bboxes_idl(origin, dest, pad_deg: float) -> List[Polygon]:
    o_lon = normalize_lon_to_pacific_view(origin[0])
    d_lon = normalize_lon_to_pacific_view(dest[0])
    min_lon = min(o_lon, d_lon) - pad_deg
    max_lon = max(o_lon, d_lon) + pad_deg
    min_lat = min(origin[1], dest[1]) - pad_deg
    max_lat = max(origin[1], dest[1]) + pad_deg

    bboxes=[]
    if min_lon < 0:
        bboxes.append(Polygon([(min_lon,min_lat),(0,min_lat),(0,max_lat),(min_lon,max_lat)]))
        min_lon=0
    if max_lon > 360:
        bboxes.append(Polygon([(0,min_lat),(max_lon-360,min_lat),(max_lon-360,max_lat),(0,max_lat)]))
        max_lon=360

    lon_min_std = min_lon if min_lon <= 180 else min_lon-360
    lon_max_std = max_lon if max_lon <= 180 else max_lon-360

    if lon_min_std <= lon_max_std:
        bboxes.append(Polygon([(lon_min_std,min_lat),(lon_max_std,min_lat),
                               (lon_max_std,max_lat),(lon_min_std,max_lat)]))
    else:
        bboxes.append(Polygon([(lon_min_std,min_lat),(180,min_lat),(180,max_lat),(lon_min_std,max_lat)]))
        bboxes.append(Polygon([(-180,min_lat),(lon_max_std,min_lat),(lon_max_std,max_lat),(-180,max_lat)]))
    return bboxes

def union_lonlat_bboxes(bboxes: List[Polygon]) -> Polygon:
    # just union them (they are already lon/lat)
    if not bboxes: raise ValueError("No bboxes")
    u = unary_union(bboxes)
    if isinstance(u, GeometryCollection):
        # take envelope if weird
        return u.envelope
    return u

def load_polys_in_bboxes(shp_path: Path, bbox_polys: List[Polygon]) -> List[Polygon]:
    polys=[]
    with fiona.open(shp_path) as src:
        for feat in src:
            g = shape(feat["geometry"])
            for box in bbox_polys:
                if g.intersects(box):
                    gi = g.intersection(box)
                    try:
                        parts = list(shapely.get_parts(gi))
                    except Exception:
                        parts = list(gi.geoms) if gi.geom_type=="MultiPolygon" else [gi]
                    for p in parts:
                        if not p.is_empty:
                            polys.append(p)
                    break
    return polys

# ---------------- Land layers & spatial indexes ----------------
def build_land_layers(polys: List[Polygon]):
    # Work in metric
    parts_m = [to_metric(p).buffer(0) for p in polys]
    union_m = unary_union(parts_m)
    collision_m = union_m.buffer(COLLISION_SAFETY_KM * 1000.0)
    ring_m      = union_m.buffer(BUFFER_KM * 1000.0)
    return {
        "UNION_M":          union_m,
        "COLLISION_PREP_M": prep(collision_m),
        "COLLISION_WGS":    to_wgs(collision_m),
        "RING_M":           ring_m,
        "RING_WGS":         to_wgs(ring_m),
        "LAND_RAW_WGS":     to_wgs(union_m),
        "LAND_PARTS_M":     parts_m,
        "COLLISION_M":      collision_m,
    }

def build_land_strtree(parts_m: List[shapely.geometry.base.BaseGeometry]) -> STRtree:
    # STRtree on metric parts to quickly shortlist potential intersections
    return STRtree(parts_m)

# ---------------- Nudge helpers ----------------
def nudge_to_ring_if_inside(pt_ll: Tuple[float,float], UNION_M, inner_buffer_km: float = BUFFER_KM, target_offset_km: float = AVOID_KM):
    px, py = to_m(pt_ll[0], pt_ll[1])
    p_m = Point(px, py)
    inner_ring_m  = UNION_M.buffer(inner_buffer_km * 1000.0)
    target_ring_m = UNION_M.buffer(target_offset_km * 1000.0)
    if inner_ring_m.contains(p_m):
        boundary = target_ring_m.boundary if not target_ring_m.boundary.is_empty else target_ring_m
        q_m = shapely.ops.nearest_points(p_m, boundary)[1]
        q_llx, q_lly = to_ll(q_m.x, q_m.y)
        return (q_llx, q_lly), True
    return pt_ll, False

# ---- Fast nudge: reuse precomputed rings (NO per-call buffer) ----
def nudge_to_ring_if_inside_fast(
    pt_ll,
    inner_ring_m,          # = UNION_M.buffer(BUFFER_KM*1000)   (precomputed)
    target_boundary_m      # = (UNION_M.buffer(AVOID_KM*1000)).boundary  (precomputed)
):
    """
    若點位於「近岸禁入緩衝」(inner ring) 之內，將其推到「外圈緩衝」(target ring) 的邊界上。
    使用預先計算好的圖層，避免每次都 buffer，速度快很多。
    回傳: (新座標, 是否有移動)
    """
    px, py = to_m(pt_ll[0], pt_ll[1])
    p_m = Point(px, py)

    if inner_ring_m.contains(p_m):
        q_m = shapely.ops.nearest_points(p_m, target_boundary_m)[1]
        q_llx, q_lly = to_ll(q_m.x, q_m.y)
        return (q_llx, q_lly), True
    return pt_ll, False

# ---- Debug 幫手 ------
def create_lvs_progress():
    """
    用於 Jupyter 中斷後仍能畫圖的進度物件。
    - candidate_path: 當輪 A* 的候選節點索引序列（list[int]）
    - free_prefix_len: 當輪候選路徑中，從起點起連續通過可視檢查（FREE）的邊數量
    - free_edges: 累計所有輪已被標記 FREE 的邊 (u,v)（可重複；畫圖前會過濾處理）
    - nodes_ref: 目前 nodes 的參考（list[(lon,lat)]）
    - iter: 當前 LVS 的迭代次數（純資訊）
    """
    return {
        "candidate_path": None,
        "free_prefix_len": 0,
        "free_edges": [],
        "nodes_ref": None,
        "iter": 0,
    }


# ---------------- Visibility (with STRtree fast prefilter) ----------------
def visible(a, b, COLLISION_PREP_M, land_tree: Optional[STRtree]=None) -> bool:
    # Build geodesic polyline and test against collision buffer (metric space)
    ls_ll = LineString(geodesic_sample(a, b, step_km=STEP_KM_GEODESIC))
    ls_m  = to_metric(ls_ll)

    if land_tree is not None:
        # coarse prefilter: only check parts whose bbox intersects ls
        candidates = land_tree.query(ls_m)
        if len(candidates) == 0:
            return True  # no land around; safe
        # quick reject by buffered line vs. nothing? We'll still ask COLLISION_PREP_M.
    return not COLLISION_PREP_M.intersects(ls_m)

# ---------------- Feature extraction (from your second script, no plotting) ----------------
def _bearing_deg(a, b):
    ax, ay = a; bx, by = b
    return (math.degrees(math.atan2(by - ay, bx - ax)) + 360.0) % 360.0

def _angdiff(a, b):
    d = (a - b + 540.0) % 360.0 - 180.0
    return d

def _resample_linestring_m(ls_m: LineString, step_m: float) -> LineString:
    L = ls_m.length
    if L == 0:
        return ls_m
    n = max(4, int(round(L / step_m)))
    d = L / n
    pts = [ls_m.interpolate(i * d) for i in range(n)]
    if pts[0].distance(pts[-1]) > 1e-6:
        pts.append(pts[0])
    return LineString(pts)

def _local_maxima(seq, radius):
    n = len(seq)
    peaks = []
    for i in range(radius, n - radius):
        v = seq[i]
        if all(v > seq[i - k] for k in range(1, radius + 1)) and \
           all(v >= seq[i + k] for k in range(1, radius + 1)):
            peaks.append(i)
    return peaks

def extract_convex_peaks_from_buffer(
    union_ll,
    avoid_km=AVOID_KM,
    resample_step_m=300.0,
    window_km=8.0,
    peak_radius_pts=3,
    min_turn_deg=18.0,
    dedup_m=1200.0
):
    union_m = to_metric(union_ll)
    buf_m   = union_m.buffer(avoid_km * 1000.0)

    if buf_m.geom_type == "Polygon":
        polys_m = [buf_m]
    else:
        polys_m = [p for p in buf_m.geoms if p.geom_type == "Polygon"]

    out_pts = []
    for poly in polys_m:
        ring_raw = LineString(list(poly.exterior.coords))
        ring     = _resample_linestring_m(ring_raw, resample_step_m)
        coords   = list(ring.coords)
        n        = len(coords)
        if n < 8:
            continue

        ccw = Polygon(coords).exterior.is_ccw
        W = max(1, int(round(window_km * 1000.0 / resample_step_m)))
        dturn = [0.0] * n
        for i in range(W, n - W):
            h1 = _bearing_deg(coords[i - W], coords[i])
            h2 = _bearing_deg(coords[i],     coords[i + W])
            dturn[i] = _angdiff(h2, h1)

        score = [max(0.0, v) if ccw else max(0.0, -v) for v in dturn]
        peaks = _local_maxima(score, peak_radius_pts)
        peaks = [i for i in peaks if score[i] >= min_turn_deg]

        kept = []
        for i in sorted(peaks, key=lambda j: -score[j]):
            pi = Point(coords[i])
            if all(pi.distance(Point(coords[k])) > dedup_m for k in kept):
                kept.append(i)

        for i in kept:
            x, y = coords[i]
            lon, lat = to_ll(x, y)
            out_pts.append((lon, lat))
    return out_pts

def _dedup_points_geom(pts: List[Point], tol_m: float) -> List[Point]:
    out=[]
    for p in pts:
        keep=True
        for q in out:
            if p.distance(q) <= tol_m:
                keep=False; break
        if keep: out.append(p)
    return out

def _cum_lengths(coords):
    L=[0.0]
    for (x1,y1),(x2,y2) in zip(coords, coords[1:]):
        L.append(L[-1] + math.hypot(x2-x1, y2-y1))
    return L

def _index_at_arclen(L, idx, s):
    target_back = L[idx] - s
    i_back = idx
    while i_back > 0 and L[i_back-1] > target_back:
        i_back -= 1
    target_fwd = L[idx] + s
    i_fwd = idx
    n = len(L) - 1
    while i_fwd < n and L[i_fwd+1] < target_fwd:
        i_fwd += 1
    return i_back, i_fwd

def _angle_and_prominence(a,b,c):
    ax, ay = a; bx, by = b; cx, cy = c
    v1x, v1y = ax - bx, ay - by
    v2x, v2y = cx - bx, cy - by
    n1 = math.hypot(v1x, v1y); n2 = math.hypot(v2x, v2y)
    if n1 == 0 or n2 == 0:
        return 180.0, 0.0
    cosang = max(-1.0, min(1.0, (v1x*v2x + v1y*v2y) / (n1*n2)))
    ang = math.degrees(math.acos(cosang))
    vx, vy = cx - ax, cy - ay
    vlen = math.hypot(vx, vy)
    if vlen == 0:
        prom = 0.0
    else:
        abx, aby = bx - ax, by - ay
        cross = abs(abx * vy - aby * vx)
        prom = cross / (vlen * vlen)
    return ang, prom

def score_convex_concave_on_ring(
    ring_ls_m: LineString,
    scales_km=(5, 10, 20, 40),
    angle_convex_max=170.0,
    angle_concave_min=210.0,
    min_prom_convex=0.002,
    min_prom_concave=0.002,
    simplify_before=False, simplify_m=800.0,
    dedup_m=800.0,
):
    ls = ring_ls_m
    if simplify_before:
        ls = ring_ls_m.simplify(simplify_m, preserve_topology=False)
        if not ls.is_ring:
            ls = LineString(list(ls.coords) + [ls.coords[0]])

    coords = list(ls.coords)
    n = len(coords)
    if n < 5:
        return [], [], coords

    ccw = Polygon(coords).exterior.is_ccw
    L = _cum_lengths(coords)
    scales_m = [s * 1000.0 for s in scales_km]
    min_angle = [180.0] * n
    max_prom  = [0.0]   * n

    for i in range(n):
        for s in scales_m:
            i_back, i_fwd = _index_at_arclen(L, i, s)
            if i_back == i or i_fwd == i:
                continue
            a = coords[i_back]; b = coords[i]; c = coords[i_fwd]
            ang, prom = _angle_and_prominence(a, b, c)
            if ang < min_angle[i]:
                min_angle[i] = ang
            if prom > max_prom[i]:
                max_prom[i] = prom

    convex_idx, concave_idx = [], []
    for i in range(1, n - 1):
        a = coords[i - 1]; b = coords[i]; c = coords[i + 1]
        v1x, v1y = a[0] - b[0], a[1] - b[1]
        v2x, v2y = c[0] - b[0], c[1] - b[1]
        cross = v1x * v2y - v1y * v2x
        is_concave = (cross < 0) if ccw else (cross > 0)
        ang  = min_angle[i]
        prom = max_prom[i]

        if not is_concave:
            if ang < angle_convex_max and prom >= min_prom_convex:
                convex_idx.append(i)
        else:
            if ang > angle_concave_min and prom >= min_prom_concave:
                concave_idx.append(i)

    def _dedup_by_spacing(idxs):
        kept = []
        taken = [False] * n
        for i in sorted(idxs, key=lambda j: -max_prom[j]):
            if taken[i]:
                continue
            kept.append(i)
            xi, yi = coords[i]
            for j in range(n):
                if taken[j]:
                    continue
                xj, yj = coords[j]
                if math.hypot(xj - xi, yj - yi) <= dedup_m:
                    taken[j] = True
        return kept

    convex_idx  = _dedup_by_spacing(convex_idx)
    concave_idx = _dedup_by_spacing(concave_idx)
    return convex_idx, concave_idx, coords

def extract_feature_points_bbox(
    shp_path: Path,
    bbox_ll_polygon: Polygon,
    avoid_km=AVOID_KM,
    simplify_m=1000.0,
    ANGLE_CONVEX_MAX=170.0,
    ANGLE_CONCAVE_MIN=210.0,
    MIN_PROM_CONVEX=0.002,
    MIN_PROM_CONCAVE=0.002,
    DEDUP_CONVEX_M=800.0,
    DEDUP_CONCAVE_M=800.0,
    ENABLE_UNIFORM=False,
    TARGET_SPACING_KM=25.0,
    N_MIN=8, N_MAX=64,
    PERIM_MIN_KM=20.0,
    AREA_MIN_KM2=5.0,
) -> Dict[str, List[Tuple[float,float]]]:
    # 1) clip land by bbox
    polys=[]
    with fiona.open(shp_path) as src:
        for feat in src:
            g = shape(feat["geometry"])
            if g.is_empty: 
                continue
            if g.intersects(bbox_ll_polygon):
                gi = g.intersection(bbox_ll_polygon)
                if not gi.is_empty:
                    polys.append(gi)
    if not polys:
        return {"convex":[], "concave":[], "uniform":[], "convex_peaks":[]}

    union_ll = unary_union(polys)  # lon/lat
    union_m  = to_metric(union_ll)
    buf_m    = union_m.buffer(avoid_km * 1000.0)

    if buf_m.geom_type == "Polygon":
        polys_m = [buf_m]
    else:
        polys_m = [p for p in buf_m.geoms if p.geom_type == "Polygon"]

    convex_pts_m, concave_pts_m, uniform_pts_m = [], [], []
    for poly in polys_m:
        perim_km = poly.exterior.length / 1000.0
        area_km2 = poly.area / 1e6
        if perim_km < PERIM_MIN_KM or area_km2 < AREA_MIN_KM2:
            continue
        ring_raw = shapely.LineString(poly.exterior.coords)
        ring_s   = ring_raw.simplify(simplify_m, preserve_topology=False)
        if not ring_s.is_ring:
            ring_s = shapely.LineString(list(ring_s.coords)+[ring_s.coords[0]])
        convex_idx, concave_idx, coords_used = score_convex_concave_on_ring(
            ring_s,
            scales_km=(5,10,20,40),
            angle_convex_max=ANGLE_CONVEX_MAX,
            angle_concave_min=ANGLE_CONCAVE_MIN,
            min_prom_convex=MIN_PROM_CONVEX,
            min_prom_concave=MIN_PROM_CONCAVE,
            simplify_before=False,
            dedup_m=max(DEDUP_CONVEX_M, DEDUP_CONCAVE_M),
        )
        for i in convex_idx:
            x,y = coords_used[i]; convex_pts_m.append(Point(x,y))
        for i in concave_idx:
            x,y = coords_used[i]; concave_pts_m.append(Point(x,y))

        if ENABLE_UNIFORM:
            n_uniform = max(N_MIN, min(N_MAX, int(perim_km / TARGET_SPACING_KM)))
            L = ring_raw.length
            for i in range(n_uniform):
                uniform_pts_m.append(ring_raw.interpolate(i * L / n_uniform))

    # convex peaks (direction extrema) using original union_ll
    convex_peaks_ll = extract_convex_peaks_from_buffer(
        union_ll=union_ll,
        avoid_km=avoid_km,
        resample_step_m=300.0,
        window_km=8.0,
        peak_radius_pts=3,
        min_turn_deg=18.0,
        dedup_m=1200.0
    )

    # dedup (metric)
    convex_pts_m  = _dedup_points_geom(convex_pts_m,  DEDUP_CONVEX_M)
    concave_pts_m = _dedup_points_geom(concave_pts_m, DEDUP_CONCAVE_M)
    if ENABLE_UNIFORM:
        uniform_pts_m = _dedup_points_geom(uniform_pts_m, 1500.0)

    def _to_ll_list(pts):
        out=[]
        for p in pts:
            lon,lat = to_ll(p.x,p.y)
            out.append((lon,lat))
        return out

    convex_ll   = _to_ll_list(convex_pts_m)
    concave_ll  = _to_ll_list(concave_pts_m)
    uniform_ll  = _to_ll_list(uniform_pts_m) if ENABLE_UNIFORM else []

    return {
        "convex": convex_ll,
        "concave": concave_ll,
        "uniform": uniform_ll,
        "convex_peaks": convex_peaks_ll
    }

# ---------------- Lazy Visibility Search Graph ----------------
def rotation_cost(seq: List[Tuple[float,float]]) -> float:
    rot=0.0
    for a,b,c in zip(seq[:-2], seq[1:-1], seq[2:]):
        h1 = bearing_xy(a,b); h2 = bearing_xy(b,c)
        rot += abs(angle_diff(h2,h1))
    return rot

def edge_cost(a,b, use_rot=False) -> float:
    # base = geodesic km; optional small rotation penalty is handled path-wise, not per single edge
    return gc_distance_km(a,b)

def heuristic(p, D):
    return gc_distance_km(p, D)

def neighbors_of(u_idx: int, nodes: List[Tuple[float,float]], D_idx: int, k=NEIGHBOR_K) -> List[int]:
    # choose k best forward-movers by distance to D (and also closeness to u to avoid far jumps)
    u = nodes[u_idx]; D = nodes[D_idx]
    # Score: distance to u (prefer closer) + (distance to D from v)  (lower better)
    # We'll preselect by distance-to-u to keep locality, but always include D.
    # Compute distances lazily:
    idxs = range(len(nodes))
    scored = []
    for v_idx in idxs:
        if v_idx == u_idx: 
            continue
        # ensure we don't explode; we filter later by lazy validation
        du = gc_distance_km(u, nodes[v_idx])
        dD = gc_distance_km(nodes[v_idx], D)
        score = du + 0.5*dD
        scored.append((score, v_idx))
    scored.sort(key=lambda t:t[0])
    out = [v for _,v in itertools.islice(scored, 0, k)]
    if D_idx not in out:
        out.append(D_idx)
    return out

def lazy_visibility_search(
    nodes: List[Tuple[float,float]],
    O_idx: int,
    D_idx: int,
    visible_fn,
    COLLISION_PREP_M,
    land_tree: Optional[STRtree],
    inject_gateways_fn,
    max_iters: int = 5000,
    progress: Optional[Dict]=None,
):
    """
    LazySP：每輪先用樂觀邊跑 A*，再逐邊做可視檢查。
    ★ 會在 Jupyter 直接印出目前檢查到的經緯度點位。
    """
    EDGE_STATE: Dict[Tuple[int,int], str] = {}  # 'FREE'|'BLOCKED'
    adj_cache: Dict[int, List[int]] = {}

    if progress is None:
        progress = {"iter": 0, "candidate_path": [], "free_prefix_len": 0,
                    "free_edges": [], "nodes_ref": nodes}
    else:
        progress["nodes_ref"] = nodes

    def get_neighbors(u: int) -> List[int]:
        if u not in adj_cache:
            adj_cache[u] = neighbors_of(u, nodes, D_idx, k=NEIGHBOR_K)
        return adj_cache[u]

    def a_star() -> Optional[List[int]]:
        N_local = len(nodes)
        open_heap = []
        INF = 1e18
        g = [INF] * N_local
        parent = [-1] * N_local

        g[O_idx] = 0.0
        h0 = heuristic(nodes[O_idx], nodes[D_idx])
        heapq.heappush(open_heap, (g[O_idx] + h0, O_idx))
        closed = set()

        while open_heap:
            _, u = heapq.heappop(open_heap)
            if u in closed:
                continue
            if u == D_idx:
                path = [u]
                while parent[u] != -1:
                    u = parent[u]
                    path.append(u)
                path.reverse()
                return path

            closed.add(u)
            for v in get_neighbors(u):
                if v >= len(nodes):
                    continue
                if EDGE_STATE.get((u, v)) == 'BLOCKED':
                    continue

                c = edge_cost(nodes[u], nodes[v], USE_ROTATION_PENALTY)

                if v >= len(g):
                    extend_by = v + 1 - len(g)
                    g.extend([INF] * extend_by)
                    parent.extend([-1] * extend_by)

                alt = g[u] + c
                if alt < g[v]:
                    g[v] = alt
                    parent[v] = u
                    f = alt + heuristic(nodes[v], nodes[D_idx])
                    heapq.heappush(open_heap, (f, v))
        return None

    it = 0
    while it < max_iters:
        it += 1
        path = a_star()
        if not path:
            print("[LVS] 找不到候選路徑（圖被 BLOCKED 邊切斷）", flush=True)
            raise RuntimeError("LVS: path not found.")

        progress["iter"] = it
        progress["candidate_path"] = list(path)
        progress["free_prefix_len"] = 0

        cand_nodes = [nodes[idx] for idx in path]
        print(f"[LVS] iter {it:03d} | 候選路徑節點數 = {len(cand_nodes)}", flush=True)
        print(f"         節點總數 = {len(nodes)}", flush=True)
        _print_ll("  候選起點", cand_nodes[0])
        _print_ll("  候選終點", cand_nodes[-1])

        all_valid = True
        prefix_ok = 0
        for u, v in zip(path[:-1], path[1:]):
            st = EDGE_STATE.get((u, v))
            a = nodes[u]; b = nodes[v]

            if st == 'FREE':
                _print_edge("  檢查邊(快取OK)", a, b)
                prefix_ok += 1
                progress["free_edges"].append((u, v))
                continue

            if st == 'BLOCKED':
                _print_edge("  檢查邊(快取BLOCKED)", a, b)
                all_valid = False
                break

            _print_edge("  檢查邊", a, b)
            if visible_fn(a, b, COLLISION_PREP_M, land_tree):
                EDGE_STATE[(u, v)] = EDGE_STATE[(v, u)] = 'FREE'
                print("    → OK", flush=True)
                prefix_ok += 1
                progress["free_edges"].append((u, v))
            else:
                EDGE_STATE[(u, v)] = EDGE_STATE[(v, u)] = 'BLOCKED'
                print("    → BLOCKED", flush=True)

                new_nodes = inject_gateways_fn(a, b)
                if new_nodes:
                    existing = set((round(lon,5), round(lat,5)) for (lon,lat) in nodes)
                    filtered = []
                    for q in new_nodes:
                        key = (round(q[0],5), round(q[1],5))
                        if key not in existing:
                            filtered.append(q)
                            existing.add(key)
                    if filtered:
                        print(f"    新增節點 {len(filtered)} 個：", flush=True)
                        for q in filtered:
                            _print_ll("      +", q)
                        nodes.extend(filtered)
                        adj_cache.clear()
                    else:
                        print("    無新節點可加入（皆為重複點）", flush=True)

                all_valid = False
                break

        progress["free_prefix_len"] = prefix_ok

        if all_valid:
            print("[LVS] 成功：路徑全通過驗證，最終節點序列：", flush=True)
            for i, idx in enumerate(path):
                _print_ll(f"  [{i:02d}]", nodes[idx])
            return path

    raise RuntimeError("LVS: 超過最大迭代次數仍未找到有效路徑")

# ---------------- Gateway injection (simple + robust) ----------------
def make_inject_gateways_fn(
    UNION_M,
    features_index_ll,
    take_each=3,
    inner_ring_m=None,
    target_boundary_m=None
):
    pool = list(features_index_ll.get("convex_peaks", [])) + list(features_index_ll.get("convex", []))

    def f(u_ll, v_ll):
        if not pool:
            return []

        seg_ll = LineString(geodesic_sample(u_ll, v_ll, step_km=STEP_KM_GEODESIC))
        seg_m  = to_metric(seg_ll)

        cand = []
        for (lon, lat) in pool:
            px, py = to_m(lon, lat)
            p = Point(px, py)
            cand.append((seg_m.distance(p), (lon, lat)))
        cand.sort(key=lambda t: t[0])

        new = []
        for _, pt in itertools.islice(cand, 0, take_each):
            q, _moved = nudge_to_ring_if_inside_fast(pt, inner_ring_m, target_boundary_m)
            new.append(q)

        out = []
        seen_local = set()
        for lon, lat in new:
            key = (round(lon, 5), round(lat, 5))
            if key in seen_local:
                continue
            seen_local.add(key)
            out.append((lon, lat))

        return out

    return f

# ---------------- Utility: convert land geom to Pacific view for folium ----------------
def convert_geom_to_pacific(geom):
    from shapely.geometry import mapping
    geom_dict = mapping(geom)
    def convert_coords(coords):
        if isinstance(coords[0], (list, tuple)):
            return [convert_coords(c) for c in coords]
        else:
            lon, lat = coords[0], coords[1]
            lon_pacific = normalize_lon_to_pacific_view(lon)
            return [lon_pacific, lat]
    if geom_dict['type'] == 'Polygon':
        geom_dict['coordinates'] = [convert_coords(ring) for ring in geom_dict['coordinates']]
    elif geom_dict['type'] == 'MultiPolygon':
        geom_dict['coordinates'] = [[convert_coords(ring) for ring in poly] for poly in geom_dict['coordinates']]
    return geom_dict

# ---------------- Orchestrator ----------------
def plan_route(
    origin: Tuple[float,float],
    dest: Tuple[float,float],
    land_path: Path = LAND_PATH,
    out_html: str = r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA\route_pacific_LVS.html",
    add_feature_layer: bool = True,
):
    print(f"起點: {origin}")
    print(f"終點: {dest}")
    print(f"太平洋視角: 起點={normalize_lon_to_pacific_view(origin[0]):.2f}°, "
          f"終點={normalize_lon_to_pacific_view(dest[0]):.2f}°")

    # 1) Load land inside IDL-aware bboxes
    bboxes = dynamic_bboxes_idl(origin, dest, pad_deg=PAD_DEG)
    polys  = load_polys_in_bboxes(land_path, bboxes)
    assert len(polys)>0, "No land polygons found in bbox"
    layers = build_land_layers(polys)
    UNION_M          = layers["UNION_M"]
    COLLISION_PREP_M = layers["COLLISION_PREP_M"]
    COLLISION_WGS    = layers["COLLISION_WGS"]
    RING_WGS         = layers["RING_WGS"]
    land_raw_wgs     = layers["LAND_RAW_WGS"]
    LAND_PARTS_M     = layers["LAND_PARTS_M"]

    INNER_RING_M = layers["RING_M"]
    TARGET_RING_M = UNION_M.buffer(AVOID_KM * 1000.0)
    TARGET_BOUNDARY_M = TARGET_RING_M.boundary

    land_tree = build_land_strtree(LAND_PARTS_M)

    # 2) Nudge O/D
    origin_adj, moved_o = nudge_to_ring_if_inside_fast(origin, INNER_RING_M, TARGET_BOUNDARY_M)
    dest_adj,   moved_d = nudge_to_ring_if_inside_fast(dest,   INNER_RING_M, TARGET_BOUNDARY_M)

    if moved_o:
        _print_edge("[INFO] 起點在 5km 緩衝內 → 已推到 15km 外圈", origin, origin_adj)
    if moved_d:
        _print_edge("[INFO] 終點在 5km 緩衝內 → 已推到 15km 外圈", dest, dest_adj)

    # 3) Extract features for the whole bbox union
    bbox_union_ll = union_lonlat_bboxes(bboxes)
    feat = extract_feature_points_bbox(
        shp_path=land_path,
        bbox_ll_polygon=bbox_union_ll,
        avoid_km=AVOID_KM,
        simplify_m=1000.0,
        ANGLE_CONVEX_MAX=170.0,
        ANGLE_CONCAVE_MIN=210.0,
        MIN_PROM_CONVEX=0.002,
        MIN_PROM_CONCAVE=0.002,
        ENABLE_UNIFORM=False,
        PERIM_MIN_KM=20.0,
        AREA_MIN_KM2=5.0
    )
    feature_nodes = list(feat["convex_peaks"]) + list(feat["convex"])

    # 4) Build initial node list and nudge them all
    base_nodes = [origin_adj, dest_adj] + feature_nodes
    nodes=[]
    for p in base_nodes:
        q,_ = nudge_to_ring_if_inside_fast(p, INNER_RING_M, TARGET_BOUNDARY_M)
        nodes.append(q)
    if len(nodes) > LVS_MAX_NODES:
        print(f"[WARN] nodes truncated from {len(nodes)} to {LVS_MAX_NODES}")
        nodes = nodes[:LVS_MAX_NODES]
    O_idx=0; D_idx=1

    # 5) Make injection function for blocked edges
    inject_fn = make_inject_gateways_fn(
        UNION_M,
        {"convex_peaks": feat["convex_peaks"], "convex": feat["convex"]},
        take_each=3,
        inner_ring_m=INNER_RING_M,
        target_boundary_m=TARGET_BOUNDARY_M
    )

    # 6) Run Lazy Visibility Search (with progress tracking)
    def visible_wrapper(a,b, COLLISION_PREP_M, land_tree):
        return visible(a,b, COLLISION_PREP_M, land_tree)

    print("\n開始 Lazy Visibility Search (LVS)...")
    progress = {"iter":0, "candidate_path":[], "free_prefix_len":0, "free_edges":[], "nodes_ref": None}
    
    try:
        path_idx = lazy_visibility_search(
            nodes, O_idx, D_idx, visible_wrapper, COLLISION_PREP_M, land_tree, inject_fn, max_iters=5000, progress=progress
        )
        print(f"[OK] LVS path with {len(path_idx)} nodes")
    except RuntimeError as e:
        # === Path not found: 自動繪製進度圖 ===
        print(f"\n {e}")
        print("正在繪製當前進度圖...")
        draw_progress(
            progress=progress,
            nodes=nodes,
            origin=origin,
            dest=dest,
            out_html=r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA\route_progress.html",
            land_geom=land_raw_wgs,
            ring_geom=RING_WGS,
            feature_nodes=feature_nodes,
            show_features=True
        )
        raise  # 重新拋出錯誤

    # 7) Build Folium (Pacific view)
    mid_lon, mid_lat = great_circle_midpoint(origin, dest)
    center_lon_pacific = normalize_lon_to_pacific_view(mid_lon)
    center_lat = mid_lat

    m = folium.Map(
        location=[center_lat, center_lon_pacific],
        zoom_start=3,
        max_bounds=False, world_copy_jump=False, no_wrap=False, min_lon=0, max_lon=360
    )
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Ocean_Basemap/MapServer/tile/{z}/{y}/{x}',
        attr='Esri', name='Ocean Basemap', overlay=False, control=True, no_wrap=False
    ).add_to(m)
    folium.TileLayer(
        tiles='https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',
        attr='© OpenStreetMap', name='OpenStreetMap', overlay=False, control=True, no_wrap=False
    ).add_to(m)

    # Land + 5km ring
    folium.GeoJson(
        convert_geom_to_pacific(land_raw_wgs),
        name="陸地",
        style_function=lambda x: {"color":"#2ca02c","weight":1,"fillOpacity":0.15}
    ).add_to(m)
    folium.GeoJson(
        convert_geom_to_pacific(RING_WGS),
        name=f"航道緩衝區 {BUFFER_KM}km",
        style_function=lambda x: {"color":"#6a5acd","weight":2,"fillOpacity":0.05}
    ).add_to(m)

    # Markers: O/D
    origin_pacific = [origin[1], normalize_lon_to_pacific_view(origin[0])]
    dest_pacific   = [dest[1],   normalize_lon_to_pacific_view(dest[0])]
    folium.Marker(
        origin_pacific,
        tooltip=f"起點: <br>({origin[0]:.2f}°, {origin[1]:.2f}°)",
        icon=folium.Icon(color='green', icon='ship', prefix='fa')
    ).add_to(m)
    folium.Marker(
        dest_pacific,
        tooltip=f"終點: <br>({dest[0]:.2f}°, {dest[1]:.2f}°)",
        icon=folium.Icon(color='red', icon='anchor', prefix='fa')
    ).add_to(m)

    # reference great-circle
    draw_gc_polyline_continuous(
        m, origin, dest, step_km=80.0,
        color='gray', weight=2, opacity=0.4, dash_array="8,4"
    )

    # Candidate feature nodes layer (for debugging)
    if add_feature_layer:
        fg_feat = folium.FeatureGroup(name="候選特徵點（凸峰+凸點）", show=False)
        for (lon,lat) in feature_nodes:
            folium.CircleMarker(
                [lat, normalize_lon_to_pacific_view(lon)], radius=3,
                color="#1f77b4", fill=True, fill_opacity=0.8,
                tooltip=f"Feature ({lon:.3f},{lat:.3f})"
            ).add_to(fg_feat)
        fg_feat.add_to(m)

    # 8) Draw final path as continuous blue GC segments
    final_segments=[]
    if moved_o and origin != origin_adj:
        final_segments.append((origin, origin_adj))
        draw_gc_polyline_continuous(
            m, origin, origin_adj, step_km=DRAW_STEP_KM,
            color='#1f77b4', weight=5, opacity=0.9
        )

    for u, v in zip(path_idx[:-1], path_idx[1:]):
        a = nodes[u]; b = nodes[v]
        final_segments.append((a, b))
        draw_gc_polyline_continuous(
            m, a, b, step_km=DRAW_STEP_KM,
            color='#1f77b4', weight=5, opacity=0.9
        )

    if moved_d and dest_adj != dest:
        final_segments.append((dest_adj, dest))
        draw_gc_polyline_continuous(
            m, dest_adj, dest, step_km=DRAW_STEP_KM,
            color='#1f77b4', weight=5, opacity=0.9
        )
        
    # 9) Waypoint markers (exclude ends)
    for i, idx in enumerate(path_idx[1:-1], 1):
        wp = nodes[idx]
        wp_pacific = [wp[1], normalize_lon_to_pacific_view(wp[0])]
        folium.CircleMarker(
            wp_pacific, radius=4,
            tooltip=f"航點 {i}<br>({wp[0]:.2f}°, {wp[1]:.2f}°)",
            color='blue', fill=True, fill_opacity=0.8
        ).add_to(m)

    # 存檔 & 總結
    folium.LayerControl(collapsed=False).add_to(m)
    m.save(out_html)

    total_km = sum(gc_distance_km(a, b) for (a, b) in final_segments)

    print(f"\n已儲存至: {out_html}")
    print(f"總航程: {total_km:.1f} 公里")
    print(f"節點數: {len(nodes)} / 路徑節點: {len(path_idx)}")

    # 額外資訊：推移距離（若有）
    if moved_o:
        print(f"[INFO] 起點推移距離 ≈ {gc_distance_km(origin, origin_adj):.2f} km")
    if moved_d:
        print(f"[INFO] 終點推移距離 ≈ {gc_distance_km(dest, dest_adj):.2f} km")


# ---------------- Example call ----------------
if __name__ == "__main__":
    origin = (121.6147,31.36636)   
    dest   = (135.828056,33.468056)   
    #| KAOHSIUNG  | TWKHH     | (22.58425,120.3181)  | (120.3181,22.58425)            |
    #| TAICHUNG   | TWTXG     | (24.255,120.5075)    | (120.5075,24.255)              |
    #| KEELUNG    | TWKEL     | (25.14515,121.7532)  | (121.7532,25.14515)            |
    #| SHANGHAI   | CNSHG     | (31.36636,121.6147)  | (121.6147,31.36636)            |
    #| NINGBO     | CNNBG     | (29.92654,121.8525)  | (121.8525,29.92654)            |
    #| ZHOUSHAN   | CNZOS     | (29.92161,122.2104)  | (122.2104,29.92161)            |
    #| SHENZHEN   | CNSZX     | (22.5045,113.8535)   | (113.8535,22.5045)             |
    #| LOS ANGELES| USLAX     | (33.74021,-118.265)  | (-118.265,33.74021)            |
    #| SEATTLE    | USSEA     | (47.6212,-122.3643)  | (-122.3643,47.6212)            |
    #| TOKYO      | JPTYO     | (35.61168,139.8268)  | (139.8268,35.61168)            |
    #| KOBE       | JPUKB     | (34.6867,135.2671)   | (135.2671,34.6867)             |
    #| WAKAYAMA   | JPUKW     | (33.468056,135.828056) | (135.828056,33.468056)       |
    #| BUSAN      | KRPUS     | (35.116667,129.033333) | (129.033333,35.116667)       |
    #| Sihanoukville| KHKOS   |                      | (103.5, 10.633333)             |

    try:
        plan_route(origin, dest)
    except KeyboardInterrupt:
        print("\n 用戶中斷 (Ctrl+C)，進度圖已保存")
    except RuntimeError:
        print("\n 路徑規劃失敗，進度圖已保存供除錯使用")

起點: (121.6147, 31.36636)
終點: (135.828056, 33.468056)
太平洋視角: 起點=121.61°, 終點=135.83°
[INFO] 起點在 5km 緩衝內 → 已推到 15km 外圈 (121.61470, 31.36636) -> (121.94711, 31.21201)
[INFO] 終點在 5km 緩衝內 → 已推到 15km 外圈 (135.82806, 33.46806) -> (135.90525, 33.35744)

開始 Lazy Visibility Search (LVS)...
[LVS] iter 001 | 候選路徑節點數 = 2
         節點總數 = 2562
  候選起點 (121.94711, 31.21201)
  候選終點 (135.90525, 33.35744)
  檢查邊 (121.94711, 31.21201) -> (135.90525, 33.35744)
    → BLOCKED
    無新節點可加入（皆為重複點）
[LVS] iter 002 | 候選路徑節點數 = 3
         節點總數 = 2562
  候選起點 (121.94711, 31.21201)
  候選終點 (135.90525, 33.35744)
  檢查邊 (121.94711, 31.21201) -> (121.94711, 31.21201)
    → OK
  檢查邊 (121.94711, 31.21201) -> (135.90525, 33.35744)
    → BLOCKED
    無新節點可加入（皆為重複點）
[LVS] iter 003 | 候選路徑節點數 = 3
         節點總數 = 2562
  候選起點 (121.94711, 31.21201)
  候選終點 (135.90525, 33.35744)
  檢查邊 (121.94711, 31.21201) -> (121.99884, 31.26974)
    → OK
  檢查邊 (121.99884, 31.26974) -> (135.90525, 33.35744)
    → BLOCKED
    無新節點可加入（皆為重複點）
[LVS] iter 004 

### 10/24 修正版

新增線條簡化、相反推論

In [ ]:
# === Lazy Visibility Search (Pacific view, great-circle, full-bbox features) ===
# Author: you + ChatGPT
# Note:
#  - Keep geodesy, Pacific-view drawing, land buffering & collision layers.
#  - Replace "outer tangent detour" with a Lazy Visibility Search (LVS) graph.
#  - Nodes = {O, D} ∪ {bbox features}; Edges validated lazily by visible().
#  - Always run bidirectional (O→D and D→O), simplify both, compare total distance,
#    and only output the shorter one.
#  - After path found, run iterative visibility-based simplification passes.
#
# Usage:
#   1) Set origin, dest (lon,lat) below (or call plan_route(...))
#   2) Run; it outputs a Folium HTML with Pacific-centered continuous arcs.

from __future__ import annotations
from pathlib import Path
import math
import heapq
import itertools
from typing import List, Tuple, Dict, Optional

import fiona
import folium
import shapely
from shapely.geometry import shape, Polygon, Point, LineString, GeometryCollection
from shapely.ops import unary_union
from shapely.prepared import prep
from shapely.strtree import STRtree
from pyproj import Transformer, Geod

# ---------------- Params (edit as needed) ----------------
LAND_PATH   = Path(r"C:\Users\slab\Desktop\Slab Project\Stage1\data\Land\ne_10m_land.shp")
BUFFER_KM   = 5.0                    # inner "no-go" near coast; points inside will be nudged outward
COLLISION_SAFETY_KM = 0.25           # collision dilation for visibility (rigid safety margin)
PAD_DEG     = 6.0                    # bbox padding in degrees
STEP_KM_GEODESIC = 3.0               # great-circle sampling resolution for visibility
DRAW_STEP_KM = 20.0                  # great-circle sampling for drawing
AVOID_KM    = 15.0                   # outer ring where routes/waypoints are allowed

# LVS knobs
NEIGHBOR_K = 24                      # candidate neighbors per node per expansion (plus D)
LVS_MAX_NODES = 4000                 # guard rail for very large bbox
USE_ROTATION_PENALTY = False
ROT_LAMBDA = 0.001

# Simplifier knobs
SIMPLIFY_MAX_PASSES = 8              # run up to N simplification passes (stop early if no changes)

# ---------------- CRS & geodesy ----------------
to_m  = Transformer.from_crs("EPSG:4326","EPSG:3857", always_xy=True).transform
to_ll = Transformer.from_crs("EPSG:3857","EPSG:4326", always_xy=True).transform
def to_metric(g): return shapely.ops.transform(to_m, g)
def to_wgs(g):    return shapely.ops.transform(to_ll, g)
GEOD = Geod(ellps="WGS84")


def _fmt_ll(p):
    return f"({p[0]:.5f}, {p[1]:.5f})"  # (lon, lat)

def _print_ll(msg, p):
    print(f"{msg} {_fmt_ll(p)}", flush=True)

def _print_edge(prefix, a, b):
    print(f"{prefix} {_fmt_ll(a)} -> {_fmt_ll(b)}", flush=True)


def geodesic_sample(a: Tuple[float,float], b: Tuple[float,float], step_km: float=STEP_KM_GEODESIC) -> List[Tuple[float,float]]:
    lon1,lat1=a; lon2,lat2=b
    _,_,dist_m = GEOD.inv(lon1,lat1,lon2,lat2)
    n = max(1,int(dist_m/(step_km*1000)))
    pts = GEOD.npts(lon1,lat1,lon2,lat2,n)
    return [(lon1,lat1)] + pts + [(lon2,lat2)]

def great_circle_midpoint(a,b):
    pts = geodesic_sample(a,b,step_km=500.0)
    return pts[len(pts)//2]

def gc_distance_km(a,b) -> float:
    _,_,d = GEOD.inv(a[0],a[1],b[0],b[1])
    return d/1000.0

def bearing_xy(p,q):
    px,py = to_m(p[0],p[1]); qx,qy = to_m(q[0],q[1])
    return math.degrees(math.atan2(qy-py, qx-px)) % 360.0

def angle_diff(a,b):
    return (a-b+540)%360 - 180

# ---------------- Pacific view helpers ----------------
def normalize_lon_to_pacific_view(lon: float) -> float:
    return lon if lon>=0 else lon+360

def draw_gc_polyline_continuous(m, a, b, step_km=DRAW_STEP_KM, **style):
    pts = geodesic_sample(a, b, step_km=step_km)
    folium_coords = []
    for lon, lat in pts:
        lon_pacific = normalize_lon_to_pacific_view(lon)
        folium_coords.append([lat, lon_pacific])
    folium.PolyLine(folium_coords, **style).add_to(m)

# ---------------- Progress map (optional debug) ----------------
def draw_progress(
    progress: Dict,
    nodes: List[Tuple[float,float]],
    origin: Tuple[float,float],
    dest: Tuple[float,float],
    out_html: str = r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA\route_progress.html",
    land_geom=None,
    ring_geom=None,
    feature_nodes: Optional[List[Tuple[float,float]]] = None,
    show_features: bool = True,
):
    cand_idx = progress.get("candidate_path", []) or []
    k = int(progress.get("free_prefix_len", 0) or 0)
    free_edges_hist = list(progress.get("free_edges", []) or [])
    O_idx, D_idx = 0, 1

    if not nodes or len(nodes) < 2:
        print("[draw_progress] nodes 不足，無法畫圖")
        return
    if not cand_idx:
        cand_idx = [O_idx, D_idx]
        k = 0

    mid_lon, mid_lat = great_circle_midpoint(origin, dest)
    center_lon_pacific = normalize_lon_to_pacific_view(mid_lon)
    center_lat = mid_lat

    m = folium.Map(
        location=[center_lat, center_lon_pacific],
        zoom_start=4,
        max_bounds=False, world_copy_jump=False, no_wrap=False, min_lon=0, max_lon=360
    )

    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Ocean_Basemap/MapServer/tile/{z}/{y}/{x}',
        attr='Esri', name='Ocean Basemap', overlay=False, control=True, no_wrap=False
    ).add_to(m)
    folium.TileLayer(
        tiles='https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',
        attr='© OpenStreetMap', name='OpenStreetMap', overlay=False, control=True, no_wrap=False
    ).add_to(m)

    if land_geom is not None:
        folium.GeoJson(
            convert_geom_to_pacific(land_geom),
            name="陸地",
            style_function=lambda x: {"color":"#2ca02c","weight":1,"fillOpacity":0.15}
        ).add_to(m)
    if ring_geom is not None:
        folium.GeoJson(
            convert_geom_to_pacific(ring_geom),
            name=f"航道緩衝區 {BUFFER_KM}km",
            style_function=lambda x: {"color":"#6a5acd","weight":2,"fillOpacity":0.05}
        ).add_to(m)

    folium.Marker(
        [origin[1], normalize_lon_to_pacific_view(origin[0])],
        tooltip=f"起點: ({origin[0]:.4f}, {origin[1]:.4f})",
        icon=folium.Icon(color='green', icon='ship', prefix='fa')
    ).add_to(m)
    folium.Marker(
        [dest[1], normalize_lon_to_pacific_view(dest[0])],
        tooltip=f"終點: ({dest[0]:.4f}, {dest[1]:.4f})",
        icon=folium.Icon(color='red', icon='anchor', prefix='fa')
    ).add_to(m)

    O_adj = nodes[O_idx]; D_adj = nodes[D_idx]
    if O_adj != origin:
        draw_gc_polyline_continuous(m, origin, O_adj, step_km=DRAW_STEP_KM,
                                    color='#1f77b4', weight=5, opacity=0.9)
    if D_adj != dest:
        draw_gc_polyline_continuous(m, D_adj, dest, step_km=DRAW_STEP_KM,
                                    color='#1f77b4', weight=5, opacity=0.9)

    for (u, v) in free_edges_hist:
        if u < len(nodes) and v < len(nodes):
            a, b = nodes[u], nodes[v]
            draw_gc_polyline_continuous(
                m, a, b, step_km=DRAW_STEP_KM,
                color='#1f77b4', weight=4, opacity=0.65
            )

    if len(cand_idx) >= 2 and k > 0:
        for u, v in zip(cand_idx[:k], cand_idx[1:k+1]):
            if u < len(nodes) and v < len(nodes):
                a, b = nodes[u], nodes[v]
                draw_gc_polyline_continuous(
                    m, a, b, step_km=DRAW_STEP_KM,
                    color='#1f77b4', weight=6, opacity=0.95
                )

    if len(cand_idx) >= 2 and k < len(cand_idx) - 1:
        for u, v in zip(cand_idx[k:-1], cand_idx[k+1:]):
            if u < len(nodes) and v < len(nodes):
                a, b = nodes[u], nodes[v]
                draw_gc_polyline_continuous(
                    m, a, b, step_km=DRAW_STEP_KM,
                    color='#ff7f0e', weight=5, opacity=0.9, dash_array="10,6"
                )

    for i, idx in enumerate(cand_idx):
        if idx >= len(nodes):
            continue
        lon, lat = nodes[idx]
        ll = [lat, normalize_lon_to_pacific_view(lon)]
        if i <= k:
            folium.CircleMarker(ll, radius=4, color='#1f77b4', fill=True, fill_opacity=0.9,
                                tooltip=f"cand[{i}] {_fmt_ll((lon,lat))} (已驗證前綴)").add_to(m)
        else:
            folium.CircleMarker(ll, radius=4, color='#ff7f0e', fill=True, fill_opacity=0.7,
                                tooltip=f"cand[{i}] {_fmt_ll((lon,lat))} (待驗證)").add_to(m)

    if show_features and feature_nodes:
        fg_feat = folium.FeatureGroup(name="候選特徵點（凸峰+凸點）", show=False)
        for (lon, lat) in feature_nodes:
            folium.CircleMarker(
                [lat, normalize_lon_to_pacific_view(lon)], radius=3,
                color="#1f77b4", fill=True, fill_opacity=0.8,
                tooltip=f"Feature ({lon:.3f},{lat:.3f})"
            ).add_to(fg_feat)
        fg_feat.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    m.save(out_html)
    print(f"[draw_progress] 進度地圖已輸出：{out_html}")

# ---------------- IDL-aware dynamic bboxes ----------------
def dynamic_bboxes_idl(origin, dest, pad_deg: float) -> List[Polygon]:
    o_lon = normalize_lon_to_pacific_view(origin[0])
    d_lon = normalize_lon_to_pacific_view(dest[0])
    min_lon = min(o_lon, d_lon) - pad_deg
    max_lon = max(o_lon, d_lon) + pad_deg
    min_lat = min(origin[1], dest[1]) - pad_deg
    max_lat = max(origin[1], dest[1]) + pad_deg

    bboxes=[]
    if min_lon < 0:
        bboxes.append(Polygon([(min_lon,min_lat),(0,min_lat),(0,max_lat),(min_lon,max_lat)]))
        min_lon=0
    if max_lon > 360:
        bboxes.append(Polygon([(0,min_lat),(max_lon-360,min_lat),(max_lon-360,max_lat),(0,max_lat)]))
        max_lon=360

    lon_min_std = min_lon if min_lon <= 180 else min_lon-360
    lon_max_std = max_lon if max_lon <= 180 else max_lon-360

    if lon_min_std <= lon_max_std:
        bboxes.append(Polygon([(lon_min_std,min_lat),(lon_max_std,min_lat),
                               (lon_max_std,max_lat),(lon_min_std,max_lat)]))
    else:
        bboxes.append(Polygon([(lon_min_std,min_lat),(180,min_lat),(180,max_lat),(lon_min_std,max_lat)]))
        bboxes.append(Polygon([(-180,min_lat),(lon_max_std,min_lat),(lon_max_std,max_lat),(-180,max_lat)]))
    return bboxes

def union_lonlat_bboxes(bboxes: List[Polygon]) -> Polygon:
    if not bboxes: raise ValueError("No bboxes")
    u = unary_union(bboxes)
    if isinstance(u, GeometryCollection):
        return u.envelope
    return u

def load_polys_in_bboxes(shp_path: Path, bbox_polys: List[Polygon]) -> List[Polygon]:
    polys=[]
    with fiona.open(shp_path) as src:
        for feat in src:
            g = shape(feat["geometry"])
            for box in bbox_polys:
                if g.intersects(box):
                    gi = g.intersection(box)
                    try:
                        parts = list(shapely.get_parts(gi))
                    except Exception:
                        parts = list(gi.geoms) if gi.geom_type=="MultiPolygon" else [gi]
                    for p in parts:
                        if not p.is_empty:
                            polys.append(p)
                    break
    return polys

# ---------------- Land layers & spatial indexes ----------------
def build_land_layers(polys: List[Polygon]):
    parts_m = [to_metric(p).buffer(0) for p in polys]
    union_m = unary_union(parts_m)
    collision_m = union_m.buffer(COLLISION_SAFETY_KM * 1000.0)
    ring_m      = union_m.buffer(BUFFER_KM * 1000.0)
    return {
        "UNION_M":          union_m,
        "COLLISION_PREP_M": prep(collision_m),
        "COLLISION_WGS":    to_wgs(collision_m),
        "RING_M":           ring_m,
        "RING_WGS":         to_wgs(ring_m),
        "LAND_RAW_WGS":     to_wgs(union_m),
        "LAND_PARTS_M":     parts_m,
        "COLLISION_M":      collision_m,
    }

def build_land_strtree(parts_m: List[shapely.geometry.base.BaseGeometry]) -> STRtree:
    return STRtree(parts_m)

# ---------------- Nudge helpers ----------------
def nudge_to_ring_if_inside(pt_ll: Tuple[float,float], UNION_M, inner_buffer_km: float = BUFFER_KM, target_offset_km: float = AVOID_KM):
    px, py = to_m(pt_ll[0], pt_ll[1])
    p_m = Point(px, py)
    inner_ring_m  = UNION_M.buffer(inner_buffer_km * 1000.0)
    target_ring_m = UNION_M.buffer(target_offset_km * 1000.0)
    if inner_ring_m.contains(p_m):
        boundary = target_ring_m.boundary if not target_ring_m.boundary.is_empty else target_ring_m
        q_m = shapely.ops.nearest_points(p_m, boundary)[1]
        q_llx, q_lly = to_ll(q_m.x, q_m.y)
        return (q_llx, q_lly), True
    return pt_ll, False

def nudge_to_ring_if_inside_fast(pt_ll, inner_ring_m, target_boundary_m):
    px, py = to_m(pt_ll[0], pt_ll[1])
    p_m = Point(px, py)
    if inner_ring_m.contains(p_m):
        q_m = shapely.ops.nearest_points(p_m, target_boundary_m)[1]
        q_llx, q_lly = to_ll(q_m.x, q_m.y)
        return (q_llx, q_lly), True
    return pt_ll, False

# ---------------- Visibility (with STRtree fast prefilter) ----------------
def visible(a, b, COLLISION_PREP_M, land_tree: Optional[STRtree]=None) -> bool:
    ls_ll = LineString(geodesic_sample(a, b, step_km=STEP_KM_GEODESIC))
    ls_m  = to_metric(ls_ll)
    if land_tree is not None:
        candidates = land_tree.query(ls_m)
        if len(candidates) == 0:
            return True
    return not COLLISION_PREP_M.intersects(ls_m)

# ---------------- Feature extraction (concise; as you had) ----------------
def _bearing_deg(a, b):
    ax, ay = a; bx, by = b
    return (math.degrees(math.atan2(by - ay, bx - ax)) + 360.0) % 360.0

def _angdiff(a, b):
    d = (a - b + 540.0) % 360.0 - 180.0
    return d

def _resample_linestring_m(ls_m: LineString, step_m: float) -> LineString:
    L = ls_m.length
    if L == 0:
        return ls_m
    n = max(4, int(round(L / step_m)))
    d = L / n
    pts = [ls_m.interpolate(i * d) for i in range(n)]
    if pts[0].distance(pts[-1]) > 1e-6:
        pts.append(pts[0])
    return LineString(pts)

def _local_maxima(seq, radius):
    n = len(seq)
    peaks = []
    for i in range(radius, n - radius):
        v = seq[i]
        if all(v > seq[i - k] for k in range(1, radius + 1)) and \
           all(v >= seq[i + k] for k in range(1, radius + 1)):
            peaks.append(i)
    return peaks

def extract_convex_peaks_from_buffer(
    union_ll,
    avoid_km=AVOID_KM,
    resample_step_m=300.0,
    window_km=8.0,
    peak_radius_pts=3,
    min_turn_deg=18.0,
    dedup_m=1200.0
):
    union_m = to_metric(union_ll)
    buf_m   = union_m.buffer(avoid_km * 1000.0)

    if buf_m.geom_type == "Polygon":
        polys_m = [buf_m]
    else:
        polys_m = [p for p in buf_m.geoms if p.geom_type == "Polygon"]

    out_pts = []
    for poly in polys_m:
        ring_raw = LineString(list(poly.exterior.coords))
        ring     = _resample_linestring_m(ring_raw, resample_step_m)
        coords   = list(ring.coords)
        n        = len(coords)
        if n < 8:
            continue

        ccw = Polygon(coords).exterior.is_ccw
        W = max(1, int(round(window_km * 1000.0 / resample_step_m)))
        dturn = [0.0] * n
        for i in range(W, n - W):
            h1 = _bearing_deg(coords[i - W], coords[i])
            h2 = _bearing_deg(coords[i],     coords[i + W])
            dturn[i] = _angdiff(h2, h1)

        score = [max(0.0, v) if ccw else max(0.0, -v) for v in dturn]
        peaks = _local_maxima(score, peak_radius_pts)
        peaks = [i for i in peaks if score[i] >= min_turn_deg]

        kept = []
        for i in sorted(peaks, key=lambda j: -score[j]):
            pi = Point(coords[i])
            if all(pi.distance(Point(coords[k])) > dedup_m for k in kept):
                kept.append(i)

        for i in kept:
            x, y = coords[i]
            lon, lat = to_ll(x, y)
            out_pts.append((lon, lat))
    return out_pts

def _dedup_points_geom(pts: List[Point], tol_m: float) -> List[Point]:
    out=[]
    for p in pts:
        keep=True
        for q in out:
            if p.distance(q) <= tol_m:
                keep=False; break
        if keep: out.append(p)
    return out

def _cum_lengths(coords):
    L=[0.0]
    for (x1,y1),(x2,y2) in zip(coords, coords[1:]):
        L.append(L[-1] + math.hypot(x2-x1, y2-y1))
    return L

def _index_at_arclen(L, idx, s):
    target_back = L[idx] - s
    i_back = idx
    while i_back > 0 and L[i_back-1] > target_back:
        i_back -= 1
    target_fwd = L[idx] + s
    i_fwd = idx
    n = len(L) - 1
    while i_fwd < n and L[i_fwd+1] < target_fwd:
        i_fwd += 1
    return i_back, i_fwd

def _angle_and_prominence(a,b,c):
    ax, ay = a; bx, by = b; cx, cy = c
    v1x, v1y = ax - bx, ay - by
    v2x, v2y = cx - bx, cy - by
    n1 = math.hypot(v1x, v1y); n2 = math.hypot(v2x, v2y)
    if n1 == 0 or n2 == 0:
        return 180.0, 0.0
    cosang = max(-1.0, min(1.0, (v1x*v2x + v1y*v2y) / (n1*n2)))
    ang = math.degrees(math.acos(cosang))
    vx, vy = cx - ax, cy - ay
    vlen = math.hypot(vx, vy)
    if vlen == 0:
        prom = 0.0
    else:
        abx, aby = bx - ax, by - ay
        cross = abs(abx * vy - aby * vx)
        prom = cross / (vlen * vlen)
    return ang, prom

def score_convex_concave_on_ring(
    ring_ls_m: LineString,
    scales_km=(5, 10, 20, 40),
    angle_convex_max=170.0,
    angle_concave_min=210.0,
    min_prom_convex=0.002,
    min_prom_concave=0.002,
    simplify_before=False, simplify_m=800.0,
    dedup_m=800.0,
):
    ls = ring_ls_m
    if simplify_before:
        ls = ring_ls_m.simplify(simplify_m, preserve_topology=False)
        if not ls.is_ring:
            ls = LineString(list(ls.coords) + [ls.coords[0]])

    coords = list(ls.coords)
    n = len(coords)
    if n < 5:
        return [], [], coords

    ccw = Polygon(coords).exterior.is_ccw
    L = _cum_lengths(coords)
    scales_m = [s * 1000.0 for s in scales_km]
    min_angle = [180.0] * n
    max_prom  = [0.0]   * n

    for i in range(n):
        for s in scales_m:
            i_back, i_fwd = _index_at_arclen(L, i, s)
            if i_back == i or i_fwd == i:
                continue
            a = coords[i_back]; b = coords[i]; c = coords[i_fwd]
            ang, prom = _angle_and_prominence(a, b, c)
            if ang < min_angle[i]:
                min_angle[i] = ang
            if prom > max_prom[i]:
                max_prom[i] = prom

    convex_idx, concave_idx = [], []
    for i in range(1, n - 1):
        a = coords[i - 1]; b = coords[i]; c = coords[i + 1]
        v1x, v1y = a[0] - b[0], a[1] - b[1]
        v2x, v2y = c[0] - b[0], c[1] - b[1]
        cross = v1x * v2y - v1y * v2x
        is_concave = (cross < 0) if ccw else (cross > 0)
        ang  = min_angle[i]
        prom = max_prom[i]

        if not is_concave:
            if ang < angle_convex_max and prom >= min_prom_convex:
                convex_idx.append(i)
        else:
            if ang > angle_concave_min and prom >= min_prom_concave:
                concave_idx.append(i)

    def _dedup_by_spacing(idxs):
        kept = []
        taken = [False] * n
        for i in sorted(idxs, key=lambda j: -max_prom[j]):
            if taken[i]:
                continue
            kept.append(i)
            xi, yi = coords[i]
            for j in range(n):
                if taken[j]:
                    continue
                xj, yj = coords[j]
                if math.hypot(xj - xi, yj - yi) <= dedup_m:
                    taken[j] = True
        return kept

    convex_idx  = _dedup_by_spacing(convex_idx)
    concave_idx = _dedup_by_spacing(concave_idx)
    return convex_idx, concave_idx, coords

def extract_feature_points_bbox(
    shp_path: Path,
    bbox_ll_polygon: Polygon,
    avoid_km=AVOID_KM,
    simplify_m=1000.0,
    ANGLE_CONVEX_MAX=170.0,
    ANGLE_CONCAVE_MIN=210.0,
    MIN_PROM_CONVEX=0.002,
    MIN_PROM_CONCAVE=0.002,
    DEDUP_CONVEX_M=800.0,
    DEDUP_CONCAVE_M=800.0,
    ENABLE_UNIFORM=False,
    TARGET_SPACING_KM=25.0,
    N_MIN=8, N_MAX=64,
    PERIM_MIN_KM=20.0,
    AREA_MIN_KM2=5.0,
) -> Dict[str, List[Tuple[float,float]]]:
    polys=[]
    with fiona.open(shp_path) as src:
        for feat in src:
            g = shape(feat["geometry"])
            if g.is_empty:
                continue
            if g.intersects(bbox_ll_polygon):
                gi = g.intersection(bbox_ll_polygon)
                if not gi.is_empty:
                    polys.append(gi)
    if not polys:
        return {"convex":[], "concave":[], "uniform":[], "convex_peaks":[]}

    union_ll = unary_union(polys)
    union_m  = to_metric(union_ll)
    buf_m    = union_m.buffer(avoid_km * 1000.0)

    if buf_m.geom_type == "Polygon":
        polys_m = [buf_m]
    else:
        polys_m = [p for p in buf_m.geoms if p.geom_type == "Polygon"]

    convex_pts_m, concave_pts_m, uniform_pts_m = [], [], []
    for poly in polys_m:
        perim_km = poly.exterior.length / 1000.0
        area_km2 = poly.area / 1e6
        if perim_km < PERIM_MIN_KM or area_km2 < AREA_MIN_KM2:
            continue
        ring_raw = shapely.LineString(poly.exterior.coords)
        ring_s   = ring_raw.simplify(simplify_m, preserve_topology=False)
        if not ring_s.is_ring:
            ring_s = shapely.LineString(list(ring_s.coords)+[ring_s.coords[0]])
        convex_idx, concave_idx, coords_used = score_convex_concave_on_ring(
            ring_s,
            scales_km=(5,10,20,40),
            angle_convex_max=ANGLE_CONVEX_MAX,
            angle_concave_min=ANGLE_CONCAVE_MIN,
            min_prom_convex=MIN_PROM_CONVEX,
            min_prom_concave=MIN_PROM_CONCAVE,
            simplify_before=False,
            dedup_m=max(DEDUP_CONVEX_M, DEDUP_CONCAVE_M),
        )
        for i in convex_idx:
            x,y = coords_used[i]; convex_pts_m.append(Point(x,y))
        for i in concave_idx:
            x,y = coords_used[i]; concave_pts_m.append(Point(x,y))

        if ENABLE_UNIFORM:
            n_uniform = max(N_MIN, min(N_MAX, int(perim_km / TARGET_SPACING_KM)))
            L = ring_raw.length
            for i in range(n_uniform):
                uniform_pts_m.append(ring_raw.interpolate(i * L / n_uniform))

    convex_peaks_ll = extract_convex_peaks_from_buffer(
        union_ll=union_ll,
        avoid_km=avoid_km,
        resample_step_m=300.0,
        window_km=8.0,
        peak_radius_pts=3,
        min_turn_deg=18.0,
        dedup_m=1200.0
    )

    convex_pts_m  = _dedup_points_geom(convex_pts_m,  DEDUP_CONVEX_M)
    concave_pts_m = _dedup_points_geom(concave_pts_m, DEDUP_CONCAVE_M)
    if ENABLE_UNIFORM:
        uniform_pts_m = _dedup_points_geom(uniform_pts_m, 1500.0)

    def _to_ll_list(pts):
        out=[]
        for p in pts:
            lon,lat = to_ll(p.x,p.y)
            out.append((lon,lat))
        return out

    convex_ll   = _to_ll_list(convex_pts_m)
    concave_ll  = _to_ll_list(concave_pts_m)
    uniform_ll  = _to_ll_list(uniform_pts_m) if ENABLE_UNIFORM else []

    return {
        "convex": convex_ll,
        "concave": concave_ll,
        "uniform": uniform_ll,
        "convex_peaks": convex_peaks_ll
    }

# ---------------- Lazy Visibility Search Graph ----------------
def rotation_cost(seq: List[Tuple[float,float]]) -> float:
    rot=0.0
    for a,b,c in zip(seq[:-2], seq[1:-1], seq[2:]):
        h1 = bearing_xy(a,b); h2 = bearing_xy(b,c)
        rot += abs(angle_diff(h2,h1))
    return rot

def edge_cost(a,b, use_rot=False) -> float:
    return gc_distance_km(a,b)

def heuristic(p, D):
    return gc_distance_km(p, D)

def neighbors_of(u_idx: int, nodes: List[Tuple[float,float]], D_idx: int, k=NEIGHBOR_K) -> List[int]:
    u = nodes[u_idx]; D = nodes[D_idx]
    idxs = range(len(nodes))
    scored = []
    for v_idx in idxs:
        if v_idx == u_idx:
            continue
        du = gc_distance_km(u, nodes[v_idx])
        dD = gc_distance_km(nodes[v_idx], D)
        score = du + 0.5*dD
        scored.append((score, v_idx))
    scored.sort(key=lambda t:t[0])
    out = [v for _,v in itertools.islice(scored, 0, k)]
    if D_idx not in out:
        out.append(D_idx)
    return out

def create_lvs_progress():
    return {
        "candidate_path": None,
        "free_prefix_len": 0,
        "free_edges": [],
        "nodes_ref": None,
        "iter": 0,
    }

def lazy_visibility_search(
    nodes: List[Tuple[float,float]],
    O_idx: int,
    D_idx: int,
    visible_fn,
    COLLISION_PREP_M,
    land_tree: Optional[STRtree],
    inject_gateways_fn,
    max_iters: int = 5000,
    progress: Optional[Dict]=None,
):
    EDGE_STATE: Dict[Tuple[int,int], str] = {}
    adj_cache: Dict[int, List[int]] = {}

    if progress is None:
        progress = create_lvs_progress()
    progress["nodes_ref"] = nodes

    def get_neighbors(u: int) -> List[int]:
        if u not in adj_cache:
            adj_cache[u] = neighbors_of(u, nodes, D_idx, k=NEIGHBOR_K)
        return adj_cache[u]

    def a_star() -> Optional[List[int]]:
        N_local = len(nodes)
        open_heap = []
        INF = 1e18
        g = [INF] * N_local
        parent = [-1] * N_local

        g[O_idx] = 0.0
        h0 = heuristic(nodes[O_idx], nodes[D_idx])
        heapq.heappush(open_heap, (g[O_idx] + h0, O_idx))
        closed = set()

        while open_heap:
            _, u = heapq.heappop(open_heap)
            if u in closed:
                continue
            if u == D_idx:
                path = [u]
                while parent[u] != -1:
                    u = parent[u]
                    path.append(u)
                path.reverse()
                return path

            closed.add(u)
            for v in get_neighbors(u):
                if v >= len(nodes):
                    continue
                if EDGE_STATE.get((u, v)) == 'BLOCKED':
                    continue

                c = edge_cost(nodes[u], nodes[v], USE_ROTATION_PENALTY)

                if v >= len(g):
                    extend_by = v + 1 - len(g)
                    g.extend([INF] * extend_by)
                    parent.extend([-1] * extend_by)

                alt = g[u] + c
                if alt < g[v]:
                    g[v] = alt
                    parent[v] = u
                    f = alt + heuristic(nodes[v], nodes[D_idx])
                    heapq.heappush(open_heap, (f, v))
        return None

    it = 0
    while it < max_iters:
        it += 1
        path = a_star()
        if not path:
            print("[LVS] 找不到候選路徑（圖被 BLOCKED 邊切斷）", flush=True)
            raise RuntimeError("LVS: path not found.")

        progress["iter"] = it
        progress["candidate_path"] = list(path)
        progress["free_prefix_len"] = 0

        cand_nodes = [nodes[idx] for idx in path]
        print(f"[LVS] iter {it:03d} | 候選路徑節點數 = {len(cand_nodes)}", flush=True)
        print(f"         節點總數 = {len(nodes)}", flush=True)
        _print_ll("  候選起點", cand_nodes[0])
        _print_ll("  候選終點", cand_nodes[-1])

        all_valid = True
        prefix_ok = 0
        for u, v in zip(path[:-1], path[1:]):
            st = EDGE_STATE.get((u, v))
            a = nodes[u]; b = nodes[v]

            if st == 'FREE':
                _print_edge("  檢查邊(快取OK)", a, b)
                prefix_ok += 1
                progress["free_edges"].append((u, v))
                continue

            if st == 'BLOCKED':
                _print_edge("  檢查邊(快取BLOCKED)", a, b)
                all_valid = False
                break

            _print_edge("  檢查邊", a, b)
            if visible_fn(a, b, COLLISION_PREP_M, land_tree):
                EDGE_STATE[(u, v)] = EDGE_STATE[(v, u)] = 'FREE'
                print("    → OK", flush=True)
                prefix_ok += 1
                progress["free_edges"].append((u, v))
            else:
                EDGE_STATE[(u, v)] = EDGE_STATE[(v, u)] = 'BLOCKED'
                print("    → BLOCKED", flush=True)

                new_nodes = inject_gateways_fn(a, b)
                if new_nodes:
                    existing = set((round(lon,5), round(lat,5)) for (lon,lat) in nodes)
                    filtered = []
                    for q in new_nodes:
                        key = (round(q[0],5), round(q[1],5))
                        if key not in existing:
                            filtered.append(q)
                            existing.add(key)
                    if filtered:
                        print(f"    新增節點 {len(filtered)} 個：", flush=True)
                        for q in filtered:
                            _print_ll("      +", q)
                        nodes.extend(filtered)
                        adj_cache.clear()
                    else:
                        print("    無新節點可加入（皆為重複點）", flush=True)

                all_valid = False
                break

        progress["free_prefix_len"] = prefix_ok

        if all_valid:
            print("[LVS] 成功：路徑全通過驗證，最終節點序列：", flush=True)
            for i, idx in enumerate(path):
                _print_ll(f"  [{i:02d}]", nodes[idx])
            return path

    raise RuntimeError("LVS: 超過最大迭代次數仍未找到有效路徑")

# ---------------- Gateway injection ----------------
def make_inject_gateways_fn(
    UNION_M,
    features_index_ll,
    take_each=3,
    inner_ring_m=None,
    target_boundary_m=None
):
    pool = list(features_index_ll.get("convex_peaks", [])) + list(features_index_ll.get("convex", []))

    def f(u_ll, v_ll):
        if not pool:
            return []

        seg_ll = LineString(geodesic_sample(u_ll, v_ll, step_km=STEP_KM_GEODESIC))
        seg_m  = to_metric(seg_ll)

        cand = []
        for (lon, lat) in pool:
            px, py = to_m(lon, lat)
            p = Point(px, py)
            cand.append((seg_m.distance(p), (lon, lat)))
        cand.sort(key=lambda t: t[0])

        new = []
        for _, pt in itertools.islice(cand, 0, take_each):
            q, _moved = nudge_to_ring_if_inside_fast(pt, inner_ring_m, target_boundary_m)
            new.append(q)

        out = []
        seen_local = set()
        for lon, lat in new:
            key = (round(lon, 5), round(lat, 5))
            if key in seen_local:
                continue
            seen_local.add(key)
            out.append((lon, lat))
        return out

    return f

# ---------------- Utility: convert land geom to Pacific view for folium ----------------
def convert_geom_to_pacific(geom):
    from shapely.geometry import mapping
    geom_dict = mapping(geom)
    def convert_coords(coords):
        if isinstance(coords[0], (list, tuple)):
            return [convert_coords(c) for c in coords]
        else:
            lon, lat = coords[0], coords[1]
            lon_pacific = normalize_lon_to_pacific_view(lon)
            return [lon_pacific, lat]
    if geom_dict['type'] == 'Polygon':
        geom_dict['coordinates'] = [convert_coords(ring) for ring in geom_dict['coordinates']]
    elif geom_dict['type'] == 'MultiPolygon':
        geom_dict['coordinates'] = [[convert_coords(ring) for ring in poly] for poly in geom_dict['coordinates']]
    return geom_dict

# ---------------- Path simplifier (iterative) ----------------
def simplify_path_gc(
    path_idx: List[int],
    nodes: List[Tuple[float,float]],
    visible_fn,
    COLLISION_PREP_M,
    land_tree: Optional[STRtree],
    max_passes: int = SIMPLIFY_MAX_PASSES
) -> List[Tuple[float,float]]:
    """
    以「任兩點可視即可直連」的原則做迭代簡化。
    回傳：簡化後的座標序列（不再是 nodes 索引）。
    """
    if not path_idx or len(path_idx) < 2:
        return [nodes[i] for i in path_idx]

    pts = [nodes[i] for i in path_idx]
    passes = 0
    while passes < max_passes:
        passes += 1
        changed = False
        new_pts = [pts[0]]
        i = 0
        while i < len(pts)-1:
            jumped = False
            for j in range(len(pts)-1, i+1, -1):
                if visible_fn(pts[i], pts[j], COLLISION_PREP_M, land_tree):
                    if j > i+1:
                        new_pts.append(pts[j])
                        i = j
                        changed = True
                        jumped = True
                        break
            if not jumped:
                new_pts.append(pts[i+1])
                i += 1
        if len(new_pts) >= 2 and new_pts[-1] == new_pts[-2]:
            new_pts.pop()

        pts = new_pts
        if not changed:
            break
    return pts

# ---------------- NEW Helpers: single-run LVS wrapper & distance sum ----------------
def run_lvs_once(
    base_nodes: List[Tuple[float,float]],
    O_idx: int,
    D_idx: int,
    visible_fn,
    COLLISION_PREP_M,
    land_tree: Optional[STRtree],
    inject_fn,
    max_iters: int = 5000
):
    """對給定 nodes 與 O/D 索引，跑一次 LVS，回傳 path_idx, nodes(本地副本，可能被注入節點擴充), progress。"""
    nodes_local = list(base_nodes)  # 本地副本避免互汙
    progress_local = {"iter":0, "candidate_path":[], "free_prefix_len":0, "free_edges":[], "nodes_ref": None}
    path_idx = lazy_visibility_search(
        nodes_local, O_idx, D_idx, visible_fn, COLLISION_PREP_M, land_tree, inject_fn,
        max_iters=max_iters, progress=progress_local
    )
    return path_idx, nodes_local, progress_local

def total_km_after_simplify(
    simplified_pts: List[Tuple[float,float]],
    origin: Tuple[float,float],
    origin_adj: Tuple[float,float],
    dest_adj: Tuple[float,float],
    dest: Tuple[float,float],
    moved_o: bool,
    moved_d: bool
) -> float:
    """計算『簡化後航線』的總距離，包含 O/ D 若被推離時的接駁段。"""
    segs = []
    if moved_o and origin != origin_adj:
        segs.append((origin, origin_adj))
    if simplified_pts and len(simplified_pts) >= 2:
        segs.extend(zip(simplified_pts[:-1], simplified_pts[1:]))
    if moved_d and dest_adj != dest:
        segs.append((dest_adj, dest))
    return sum(gc_distance_km(a, b) for (a, b) in segs)

# ---------------- Orchestrator ----------------
def plan_route(
    origin: Tuple[float,float],
    dest: Tuple[float,float],
    land_path: Path = LAND_PATH,
    out_html: str = r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA\route_pacific_LVS.html",
    add_feature_layer: bool = True,
):
    print(f"起點: {origin}")
    print(f"終點: {dest}")
    print(f"太平洋視角: 起點={normalize_lon_to_pacific_view(origin[0]):.2f}°, "
          f"終點={normalize_lon_to_pacific_view(dest[0]):.2f}°")

    # 1) Load land inside IDL-aware bboxes
    bboxes = dynamic_bboxes_idl(origin, dest, pad_deg=PAD_DEG)
    polys  = load_polys_in_bboxes(land_path, bboxes)
    assert len(polys)>0, "No land polygons found in bbox"
    layers = build_land_layers(polys)
    UNION_M          = layers["UNION_M"]
    COLLISION_PREP_M = layers["COLLISION_PREP_M"]
    COLLISION_WGS    = layers["COLLISION_WGS"]
    RING_WGS         = layers["RING_WGS"]
    land_raw_wgs     = layers["LAND_RAW_WGS"]
    LAND_PARTS_M     = layers["LAND_PARTS_M"]

    INNER_RING_M = layers["RING_M"]              # 5km 內圈
    TARGET_RING_M = UNION_M.buffer(AVOID_KM * 1000.0)   # 15km 外圈
    TARGET_BOUNDARY_M = TARGET_RING_M.boundary

    land_tree = build_land_strtree(LAND_PARTS_M)

    # 2) Nudge O/D
    origin_adj, moved_o = nudge_to_ring_if_inside_fast(origin, INNER_RING_M, TARGET_BOUNDARY_M)
    dest_adj,   moved_d = nudge_to_ring_if_inside_fast(dest,   INNER_RING_M, TARGET_BOUNDARY_M)

    if moved_o:
        _print_edge("[INFO] 起點在 5km 緩衝內 → 已推到 15km 外圈", origin, origin_adj)
    if moved_d:
        _print_edge("[INFO] 終點在 5km 緩衝內 → 已推到 15km 外圈", dest, dest_adj)

    # 3) Extract features for the whole bbox union
    bbox_union_ll = union_lonlat_bboxes(bboxes)
    feat = extract_feature_points_bbox(
        shp_path=land_path,
        bbox_ll_polygon=bbox_union_ll,
        avoid_km=AVOID_KM,
        simplify_m=1000.0,
        ANGLE_CONVEX_MAX=170.0,
        ANGLE_CONCAVE_MIN=210.0,
        MIN_PROM_CONVEX=0.002,
        MIN_PROM_CONCAVE=0.002,
        ENABLE_UNIFORM=False,
        PERIM_MIN_KM=20.0,
        AREA_MIN_KM2=5.0
    )
    feature_nodes = list(feat["convex_peaks"]) + list(feat["convex"])

    # 4) Build initial node list and nudge them all
    base_nodes = [origin_adj, dest_adj] + feature_nodes
    nodes=[]
    for p in base_nodes:
        q,_ = nudge_to_ring_if_inside_fast(p, INNER_RING_M, TARGET_BOUNDARY_M)
        nodes.append(q)
    if len(nodes) > LVS_MAX_NODES:
        print(f"[WARN] nodes truncated from {len(nodes)} to {LVS_MAX_NODES}")
        nodes = nodes[:LVS_MAX_NODES]
    O_idx=0; D_idx=1

    # 5) Make injection function for blocked edges
    inject_fn = make_inject_gateways_fn(
        UNION_M,
        {"convex_peaks": feat["convex_peaks"], "convex": feat["convex"]},
        take_each=3,
        inner_ring_m=INNER_RING_M,
        target_boundary_m=TARGET_BOUNDARY_M
    )

    # 6) Run Lazy Visibility Search (永遠雙向各跑一次)
    def visible_wrapper(a,b, COLLISION_PREP_M, land_tree):
        return visible(a,b, COLLISION_PREP_M, land_tree)

    print("\n開始 Lazy Visibility Search (LVS)：雙向演算")

    # --- 前向 O→D ---
    path_fwd, nodes_fwd, prog_fwd = run_lvs_once(
        base_nodes=nodes, O_idx=O_idx, D_idx=D_idx,
        visible_fn=visible_wrapper, COLLISION_PREP_M=COLLISION_PREP_M,
        land_tree=land_tree, inject_fn=inject_fn, max_iters=5000
    )
    print(f"[OK] O→D LVS 節點數: {len(path_fwd)}（總節點池 {len(nodes_fwd)}）")

    # --- 反向 D→O（以對調 0/1 的 nodes 副本進行）---
    nodes_rev = list(nodes)
    nodes_rev[0], nodes_rev[1] = nodes_rev[1], nodes_rev[0]
    path_rev_do, nodes_do, prog_do = run_lvs_once(
        base_nodes=nodes_rev, O_idx=0, D_idx=1,
        visible_fn=visible_wrapper, COLLISION_PREP_M=COLLISION_PREP_M,
        land_tree=land_tree, inject_fn=inject_fn, max_iters=5000
    )
    # 反向結果翻回 O→D（索引序列反轉即可；nodes_do 保持反向時的 nodes）
    path_rev = list(reversed(path_rev_do))
    print(f"[OK] D→O LVS（已翻回 O→D）節點數: {len(path_rev)}（總節點池 {len(nodes_do)}）")

    # 7) 各自做簡化
    simp_fwd = simplify_path_gc(
        path_idx=path_fwd, nodes=nodes_fwd,
        visible_fn=visible_wrapper, COLLISION_PREP_M=COLLISION_PREP_M,
        land_tree=land_tree, max_passes=SIMPLIFY_MAX_PASSES
    )
    simp_rev = simplify_path_gc(
        path_idx=path_rev, nodes=nodes_do,
        visible_fn=visible_wrapper, COLLISION_PREP_M=COLLISION_PREP_M,
        land_tree=land_tree, max_passes=SIMPLIFY_MAX_PASSES
    )

    # 8) 計算雙邊「簡化後總距離」（含 O/D 推離接駁）
    total_simple_fwd = total_km_after_simplify(
        simplified_pts=simp_fwd,
        origin=origin, origin_adj=origin_adj,
        dest_adj=dest_adj, dest=dest,
        moved_o=moved_o, moved_d=moved_d
    )
    total_simple_rev = total_km_after_simplify(
        simplified_pts=simp_rev,
        origin=origin, origin_adj=origin_adj,
        dest_adj=dest_adj, dest=dest,
        moved_o=moved_o, moved_d=moved_d
    )

    print("\n=== 雙向簡化後總距離比較 ===")
    print(f"O→D: {total_simple_fwd:.1f} km")
    print(f"D→O: {total_simple_rev:.1f} km")

    # 9) 只輸出較短的那條（選擇 path 與 nodes 作為後續繪圖依據）
    if total_simple_fwd <= total_simple_rev:
        chosen_path_idx = path_fwd
        chosen_nodes    = nodes_fwd
        chosen_simplified = simp_fwd
        chosen_label = "O→D（較短）"
    else:
        chosen_path_idx = path_rev
        chosen_nodes    = nodes_do
        chosen_simplified = simp_rev
        chosen_label = "D→O（較短）"

    print(f"選擇輸出：{chosen_label}")

    # 10) 建 Folium 圖（僅畫較短的那條）
    mid_lon, mid_lat = great_circle_midpoint(origin, dest)
    center_lon_pacific = normalize_lon_to_pacific_view(mid_lon)
    center_lat = mid_lat

    m = folium.Map(
        location=[center_lat, center_lon_pacific],
        zoom_start=3,
        max_bounds=False, world_copy_jump=False, no_wrap=False, min_lon=0, max_lon=360
    )
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Ocean_Basemap/MapServer/tile/{z}/{y}/{x}',
        attr='Esri', name='Ocean Basemap', overlay=False, control=True, no_wrap=False
    ).add_to(m)
    folium.TileLayer(
        tiles='https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',
        attr='© OpenStreetMap', name='OpenStreetMap', overlay=False, control=True, no_wrap=False
    ).add_to(m)

    # Land + 5km ring
    folium.GeoJson(
        convert_geom_to_pacific(land_raw_wgs),
        name="陸地",
        style_function=lambda x: {"color":"#2ca02c","weight":1,"fillOpacity":0.15}
    ).add_to(m)
    folium.GeoJson(
        convert_geom_to_pacific(RING_WGS),
        name=f"航道緩衝區 {BUFFER_KM}km",
        style_function=lambda x: {"color":"#6a5acd","weight":2,"fillOpacity":0.05}
    ).add_to(m)

    # Markers: O/D
    origin_pacific = [origin[1], normalize_lon_to_pacific_view(origin[0])]
    dest_pacific   = [dest[1],   normalize_lon_to_pacific_view(dest[0])]
    folium.Marker(
        origin_pacific,
        tooltip=f"起點: <br>({origin[0]:.2f}°, {origin[1]:.2f}°)",
        icon=folium.Icon(color='green', icon='ship', prefix='fa')
    ).add_to(m)
    folium.Marker(
        dest_pacific,
        tooltip=f"終點: <br>({dest[0]:.2f}°, {dest[1]:.2f}°)",
        icon=folium.Icon(color='red', icon='anchor', prefix='fa')
    ).add_to(m)

    # 參考大圓
    draw_gc_polyline_continuous(
        m, origin, dest, step_km=80.0,
        color='gray', weight=2, opacity=0.4, dash_array="8,4"
    )

    # 特徵點（可開關）
    if add_feature_layer:
        fg_feat = folium.FeatureGroup(name="候選特徵點（凸峰+凸點）", show=False)
        for (lon,lat) in feature_nodes:
            folium.CircleMarker(
                [lat, normalize_lon_to_pacific_view(lon)], radius=3,
                color="#1f77b4", fill=True, fill_opacity=0.8,
                tooltip=f"Feature ({lon:.3f},{lat:.3f})"
            ).add_to(fg_feat)
        fg_feat.add_to(m)

    # 原始選中路徑（藍色）
    final_segments=[]
    if moved_o and origin != origin_adj:
        final_segments.append((origin, origin_adj))
        draw_gc_polyline_continuous(
            m, origin, origin_adj, step_km=DRAW_STEP_KM,
            color='#1f77b4', weight=5, opacity=0.9
        )
    for u, v in zip(chosen_path_idx[:-1], chosen_path_idx[1:]):
        a = chosen_nodes[u]; b = chosen_nodes[v]
        final_segments.append((a, b))
        draw_gc_polyline_continuous(
            m, a, b, step_km=DRAW_STEP_KM,
            color='#1f77b4', weight=5, opacity=0.9
        )
    if moved_d and dest_adj != dest:
        final_segments.append((dest_adj, dest))
        draw_gc_polyline_continuous(
            m, dest_adj, dest, step_km=DRAW_STEP_KM,
            color='#1f77b4', weight=5, opacity=0.9
        )

    # 簡化後覆蓋（紅色）
    if chosen_simplified and len(chosen_simplified) >= 2:
        fg_simplified = folium.FeatureGroup(name="簡化後航線 (可視直連)", show=True)
        if moved_o and origin != origin_adj:
            draw_gc_polyline_continuous(
                fg_simplified, origin, origin_adj, step_km=DRAW_STEP_KM,
                color='#d62728', weight=4, opacity=0.8, dash_array="6,4"
            )
        for a, b in zip(chosen_simplified[:-1], chosen_simplified[1:]):
            draw_gc_polyline_continuous(
                fg_simplified, a, b, step_km=DRAW_STEP_KM,
                color='#d62728', weight=6, opacity=0.9
            )
        if moved_d and dest_adj != dest:
            draw_gc_polyline_continuous(
                fg_simplified, dest_adj, dest, step_km=DRAW_STEP_KM,
                color='#d62728', weight=4, opacity=0.8, dash_array="6,4"
            )
        fg_simplified.add_to(m)

    # 航點（只顯示選中的那條）
    for i, idx in enumerate(chosen_path_idx[1:-1], 1):
        wp = chosen_nodes[idx]
        wp_pacific = [wp[1], normalize_lon_to_pacific_view(wp[0])]
        folium.CircleMarker(
            wp_pacific, radius=4,
            tooltip=f"航點 {i}<br>({wp[0]:.2f}°, {wp[1]:.2f}°)",
            color='blue', fill=True, fill_opacity=0.8
        ).add_to(m)

    # 存檔 & 總結
    folium.LayerControl(collapsed=False).add_to(m)
    m.save(out_html)

    total_km_original = sum(gc_distance_km(a, b) for (a, b) in final_segments)
    total_km_simple   = total_km_after_simplify(
        simplified_pts=chosen_simplified,
        origin=origin, origin_adj=origin_adj,
        dest_adj=dest_adj, dest=dest,
        moved_o=moved_o, moved_d=moved_d
    )

    print(f"\n已儲存至: {out_html}")
    print(f"總航程(原始選中LVS): {total_km_original:.1f} 公里")
    print(f"總航程(選中後簡化):  {total_km_simple:.1f} 公里")
    print(f"雙向差距（簡化後）:  {abs(total_simple_fwd - total_simple_rev):.1f} 公里")
    print(f"節點數(選中): {len(chosen_nodes)} / 路徑節點: {len(chosen_path_idx)}")
    if moved_o:
        print(f"[INFO] 起點推移距離 ≈ {gc_distance_km(origin, origin_adj):.2f} km")
    if moved_d:
        print(f"[INFO] 終點推移距離 ≈ {gc_distance_km(dest, dest_adj):.2f} km")


# ---------------- Example call ----------------
if __name__ == "__main__":
    # 你可以替換成任何一組港口
    origin = (135.828056,33.468056)          # Shanghai
    dest   = (121.6147,31.36636)          # Kaohsiung
    #| KAOHSIUNG  | TWKHH     | (22.58425,120.3181)  | (120.3181,22.58425)            |
    #| TAICHUNG   | TWTXG     | (24.255,120.5075)    | (120.5075,24.255)              |
    #| KEELUNG    | TWKEL     | (25.14515,121.7532)  | (121.7532,25.14515)            |
    #| SHANGHAI   | CNSHG     | (31.36636,121.6147)  | (121.6147,31.36636)            |
    #| NINGBO     | CNNBG     | (29.92654,121.8525)  | (121.8525,29.92654)            |
    #| ZHOUSHAN   | CNZOS     | (29.92161,122.2104)  | (122.2104,29.92161)            |
    #| SHENZHEN   | CNSZX     | (22.5045,113.8535)   | (113.8535,22.5045)             |
    #| LOS ANGELES| USLAX     | (33.74021,-118.265)  | (-118.265,33.74021)            |
    #| SEATTLE    | USSEA     | (47.6212,-122.3643)  | (-122.3643,47.6212)            |
    #| TOKYO      | JPTYO     | (35.61168,139.8268)  | (139.8268,35.61168)            |
    #| KOBE       | JPUKB     | (34.6867,135.2671)   | (135.2671,34.6867)             |
    #| WAKAYAMA   | JPUKW     | (33.468056,135.828056) | (135.828056,33.468056)       |
    #| BUSAN      | KRPUS     | (35.116667,129.033333) | (129.033333,35.116667)       |
    #| Sihanoukville| KHKOS   |                      | (103.5, 10.633333)             |

    try:
        plan_route(origin, dest)
    except KeyboardInterrupt:
        print("\n 用戶中斷 (Ctrl+C)，進度圖已保存")
    except RuntimeError as e:
        print("\n 路徑規劃失敗，建議以 draw_progress() 的方式輸出進度供除錯：", e)


起點: (135.828056, 33.468056)
終點: (121.6147, 31.36636)
太平洋視角: 起點=135.83°, 終點=121.61°
[INFO] 起點在 5km 緩衝內 → 已推到 15km 外圈 (135.82806, 33.46806) -> (135.90525, 33.35744)
[INFO] 終點在 5km 緩衝內 → 已推到 15km 外圈 (121.61470, 31.36636) -> (121.94711, 31.21201)

開始 Lazy Visibility Search (LVS)：雙向演算
[LVS] iter 001 | 候選路徑節點數 = 2
         節點總數 = 2562
  候選起點 (135.90525, 33.35744)
  候選終點 (121.94711, 31.21201)
  檢查邊 (135.90525, 33.35744) -> (121.94711, 31.21201)
    → BLOCKED
    無新節點可加入（皆為重複點）
[LVS] iter 002 | 候選路徑節點數 = 3
         節點總數 = 2562
  候選起點 (135.90525, 33.35744)
  候選終點 (121.94711, 31.21201)
  檢查邊 (135.90525, 33.35744) -> (134.31182, 33.19971)
    → OK
  檢查邊 (134.31182, 33.19971) -> (121.94711, 31.21201)
    → BLOCKED
    無新節點可加入（皆為重複點）
[LVS] iter 003 | 候選路徑節點數 = 3
         節點總數 = 2562
  候選起點 (135.90525, 33.35744)
  候選終點 (121.94711, 31.21201)
  檢查邊 (135.90525, 33.35744) -> (135.69360, 33.33546)
    → OK
  檢查邊 (135.69360, 33.33546) -> (121.94711, 31.21201)
    → BLOCKED
    無新節點可加入（皆為重複點）
[LVS] iter 00

### visualize

In [ ]:
import webbrowser
webbrowser.open("route_pacific_LVS.html")

True

In [ ]:
import webbrowser
# path not found 的 debug 圖片
webbrowser.open("route_progress.html")

True

## Modulize

In [ ]:
from land_node_route_planner import plan_route
from route_sampler import sample_along_route

origin = (120.3181,22.58425)   # (lon, lat)
dest   = (103.5, 10.633333)

waypoints, total_km, html_path, meta = plan_route(
    origin, dest,
    land_path=r"C:\Users\slab\Desktop\Slab Project\Stage1\data\Land\ne_10m_land.shp",
    out_html=r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA\route_pacific_LVS.html"
)

samples = sample_along_route(
    waypoints_ll=waypoints,
    speed_knots=16.0,   # 或 speed_kmh=30.0 （二選一）
    every_h=1.0,
    start_time="2025-10-27T08:00:00Z"  # 可省略
)

samples[:3]  # 看前三個點


    #| KAOHSIUNG  | TWKHH     | (22.58425,120.3181)  | (120.3181,22.58425)            |
    #| TAICHUNG   | TWTXG     | (24.255,120.5075)    | (120.5075,24.255)              |
    #| KEELUNG    | TWKEL     | (25.14515,121.7532)  | (121.7532,25.14515)            |
    #| SHANGHAI   | CNSHG     | (31.36636,121.6147)  | (121.6147,31.36636)            |
    #| NINGBO     | CNNBG     | (29.92654,121.8525)  | (121.8525,29.92654)            |
    #| ZHOUSHAN   | CNZOS     | (29.92161,122.2104)  | (122.2104,29.92161)            |
    #| SHENZHEN   | CNSZX     | (22.5045,113.8535)   | (113.8535,22.5045)             |
    #| LOS ANGELES| USLAX     | (33.74021,-118.265)  | (-118.265,33.74021)            |
    #| SEATTLE    | USSEA     | (47.6212,-122.3643)  | (-122.3643,47.6212)            |
    #| TOKYO      | JPTYO     | (35.61168,139.8268)  | (139.8268,35.61168)            |
    #| KOBE       | JPUKB     | (34.6867,135.2671)   | (135.2671,34.6867)             |
    #| WAKAYAMA   | JPUKW     | (33.468056,135.828056) | (135.828056,33.468056)       |
    #| BUSAN      | KRPUS     | (35.116667,129.033333) | (129.033333,35.116667)       |
    #| Sihanoukville| KHKOS   |                      | (103.5, 10.633333)             |
    #| QINZHOU    | CNQZH     | ----                 | (108.647, 21.68317)            |


KeyboardInterrupt: 

In [ ]:
# === Batch route sampling to CSV (Notebook-ready) ===

import os
import csv
import json
from itertools import permutations

from land_node_route_planner import plan_route
from route_sampler import sample_along_route

# ----- Settings you asked for -----
LAND_PATH = r"C:\Users\slab\Desktop\Slab Project\Stage1\data\Land\ne_10m_land.shp"
OUT_DIR   = r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA"
CSV_PATH  = os.path.join(OUT_DIR, "routes_samples.csv")

SOG_KNOTS_LIST = [8, 10, 12]   # 單位：節 (knots)
EVERY_HOURS    = 4.0           # 每 4 小時一點

# 港口 (代號 -> (lon, lat))
PORTS = {
    "TWKHH": (120.3181,   22.58425),
    #"TWTXG": (120.5075,   24.255),
    #"CNSHG": (121.6147,   31.36636),
    #"KRPUS": (129.033333, 35.116667),
    #"JPUKW": (135.828056, 33.468056),
    "KHKOS": (103.5, 10.633333),
}

# 確保輸出資料夾存在
os.makedirs(OUT_DIR, exist_ok=True)

# 建立所有「有向」的起迄組合（不含起終相同）
pairs = list(permutations(PORTS.keys(), 2))   # 5P2 = 20

rows = []  # 將要寫到 CSV 的列

#控制是否保留 HTML
GENERATE_HTML = False

for frm, to in pairs:
    origin = PORTS[frm]
    dest   = PORTS[to]

    # 為此起迄準備一個對應的 HTML 檔案名稱
    html_name = f"route_{frm}_{to}.html"
    html_path = os.path.join(OUT_DIR, html_name)

    print(f"\n=== Planning route {frm} -> {to} ===")
    print(f"Origin (lon,lat): {origin}, Dest: {dest}")
    try:
        # 規劃一次（雙向比較 + 簡化，輸出較短者），並存出 HTML
        waypoints_ll, total_km, saved_html, meta = plan_route(
            origin=origin,
            dest=dest,
            land_path=LAND_PATH,
            out_html=html_path,
            add_feature_layer=True,  # 需要可以關掉
        )
        print(f"  - Planned OK. Simplified distance: {total_km:.1f} km")
        #print(f"  - HTML saved: {saved_html}")
                # >>> 新增：不保留 HTML 就立刻刪除
        if not GENERATE_HTML and saved_html and os.path.exists(saved_html):
            try:
                os.remove(saved_html)
                # print(f"  - (deleted) {saved_html}")  # 如需除錯可開
            except Exception as _e:
                # 避免刪除失敗影響流程
                pass
        # 針對三個 SOG 做取樣（每 4 小時）
        for sog_knots in SOG_KNOTS_LIST:
            samples = sample_along_route(
                waypoints_ll=waypoints_ll,# (lon, lat)
                speed_knots=sog_knots,    # 單位：節
                every_h=EVERY_HOURS,      # 你想改幾小時就改這個變數 h
                start_time=None,          # 可加起航時間
                include_waypoints=False,   # 預設就是 True，其實可省略
                snap_km=0.5               # 需要可調
            )
            # 取出 [lon, lat] 串列，並序列化成 JSON 陣列字串
            coor_list = [[round(s["lon"], 6), round(s["lat"], 6)] for s in samples]
            coor_json = json.dumps(coor_list, ensure_ascii=False)
            rows.append({
                "From": frm,
                "To": to,
                "Sog": sog_knots,
                "coor": coor_json,
            })

    except Exception as e:
        # 若規劃失敗，仍然為 8/10/12 各寫一列，coor=ERROR
        print(f"  !! ERROR for {frm}->{to}: {e}")
        for sog_knots in SOG_KNOTS_LIST:
            rows.append({
                "From": frm,
                "To": to,
                "Sog": sog_knots,
                "coor": "ERROR",
            })

# 寫出 CSV（欄位順序照你的範例）
fieldnames = ["From", "To", "Sog", "coor"]
with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter=',')
    writer.writeheader()
    writer.writerows(rows)

print(f"\nAll done. CSV saved to: {CSV_PATH}")
print(f"Total rows: {len(rows)}")  # 應為 5*4*3 = 60



=== Planning route TWKHH -> KHKOS ===
Origin (lon,lat): (120.3181, 22.58425), Dest: (103.5, 10.633333)
  !! ERROR for TWKHH->KHKOS: LVS failed in both directions. fwd=LVS: path not found., rev=LVS: path not found.

=== Planning route KHKOS -> TWKHH ===
Origin (lon,lat): (103.5, 10.633333), Dest: (120.3181, 22.58425)
  !! ERROR for KHKOS->TWKHH: LVS failed in both directions. fwd=LVS: path not found., rev=LVS: path not found.

All done. CSV saved to: C:\Users\slab\Desktop\Slab Project\Stage2 ETA\routes_samples.csv
Total rows: 6


### 檢查有沒有吃到更新模組

In [ ]:
import route_sampler, importlib, inspect
print("Loaded from:", route_sampler.__file__)
print("Signature:", inspect.signature(route_sampler.sample_along_route))


Loaded from: c:\Users\slab\Desktop\Slab Project\Stage2 ETA\route_sampler.py
Signature: (waypoints_ll: 'List[Tuple[float, float]]', *, speed_kmh: 'Optional[float]' = None, speed_knots: 'Optional[float]' = None, every_h: 'float' = 1.0, start_time: 'Optional[str | datetime]' = None, ellps: 'str' = 'WGS84', include_waypoints: 'bool' = True, snap_km: 'float' = 0.5) -> 'List[Dict[str, Any]]'


### Visualize

In [ ]:
# === Visualize routes_samples.csv with Folium (Notebook-ready) ===

import os
import csv
import json
import folium

# ---------- Settings ----------
CSV_PATH  = r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA\routes_samples.csv"
OUT_DIR   = r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA"
OUT_HTML  = os.path.join(OUT_DIR, "routes_samples_viewer.html")

# 過濾器（留空 = 不過濾）
FILTER_FROM = None         # 例如 "TWKHH" 或 None
FILTER_TO   = None         # 例如 "CNSHG" 或 None
FILTER_SOG  = None         # 例如 10 或 None

# 視覺：不同 SOG 對應不同顏色
SOG_COLOR = {
    8:  "#1f77b4",  # blue
    10: "#ff7f0e",  # orange
    12: "#2ca02c",  # green
}

# 太平洋視角：把負經度換到 [0, 360)
def normalize_lon_to_pacific_view(lon: float) -> float:
    return lon if lon >= 0 else lon + 360.0

# ---------- 讀檔與整理 ----------
rows = []
with open(CSV_PATH, encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for r in reader:
        # 基本欄位與型別
        frm = r["From"].strip()
        to  = r["To"].strip()
        try:
            sog = int(r["Sog"])
        except Exception:
            # 容錯：若 Sog 不是整數
            try:
                sog = round(float(r["Sog"]))
            except Exception:
                sog = r["Sog"]  # 保留原字串

        # 濾掉 ERROR 或不需要的列
        if r["coor"] == "ERROR":
            continue
        if FILTER_FROM and frm != FILTER_FROM:
            continue
        if FILTER_TO and to != FILTER_TO:
            continue
        if FILTER_SOG and sog != FILTER_SOG:
            continue

        try:
            points = json.loads(r["coor"])  # [[lon, lat], ...]
            if not isinstance(points, list) or len(points) < 2:
                continue
        except Exception:
            continue

        rows.append({
            "From": frm,
            "To": to,
            "Sog": sog,
            "points": points,  # [[lon, lat], ...]
        })

if not rows:
    raise RuntimeError("沒有可視化的路線（可能全是 ERROR 或被過濾掉）。")

# ---------- 決定地圖中心 ----------
# 簡單取第一條路線的中點作為視角
def midpoint_lonlat(p0, p1):
    # 粗略中點（不做大圓中點）：足夠用於初始視角
    lon = (p0[0] + p1[0]) / 2.0
    lat = (p0[1] + p1[1]) / 2.0
    return lon, lat

lon0, lat0 = midpoint_lonlat(rows[0]["points"][0], rows[0]["points"][-1])
center_lon_pacific = normalize_lon_to_pacific_view(lon0)

m = folium.Map(
    location=[lat0, center_lon_pacific],
    zoom_start=4,
    max_bounds=False, world_copy_jump=False, no_wrap=False, min_lon=0, max_lon=360
)

# 底圖
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Ocean_Basemap/MapServer/tile/{z}/{y}/{x}',
    attr='Esri', name='Ocean Basemap', overlay=False, control=True, no_wrap=False
).add_to(m)
folium.TileLayer(
    tiles='https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',
    attr='© OpenStreetMap', name='OpenStreetMap', overlay=False, control=True, no_wrap=False
).add_to(m)

# ---------- 畫每條路 ----------
for r in rows:
    frm, to, sog = r["From"], r["To"], r["Sog"]
    pts = r["points"]  # [[lon, lat], ...]

    # 轉為 Folium 需要的 [lat, lon_pacific]
    folium_pts = [[lat, normalize_lon_to_pacific_view(lon)] for lon, lat in pts]
    color = SOG_COLOR.get(sog, "#d62728")  # 預設紅色

    layer_name = f"{frm} → {to} @ {sog} kn"
    fg = folium.FeatureGroup(name=layer_name, show=False)

    # 航線
    folium.PolyLine(
        folium_pts,
        color=color,
        weight=5,
        opacity=0.9,
    ).add_to(fg)

    # 起點 / 終點
    lat_s, lon_s = folium_pts[0]
    lat_e, lon_e = folium_pts[-1]
    folium.Marker(
        [lat_s, lon_s],
        tooltip=f"Start: {frm} ({pts[0][0]:.4f}, {pts[0][1]:.4f})",
        icon=folium.Icon(color='green', icon='play', prefix='fa')
    ).add_to(fg)
    folium.Marker(
        [lat_e, lon_e],
        tooltip=f"End: {to} ({pts[-1][0]:.4f}, {pts[-1][1]:.4f})",
        icon=folium.Icon(color='red', icon='stop', prefix='fa')
    ).add_to(fg)

    fg.add_to(m)

# 圖層控制
folium.LayerControl(collapsed=False).add_to(m)

# 存檔
os.makedirs(OUT_DIR, exist_ok=True)
m.save(OUT_HTML)
print(f"Map saved to: {OUT_HTML}")

# 小提示：在 Notebook 內直接顯示（可選）
m
import webbrowser
# path not found 的 debug 圖片
webbrowser.open("routes_samples_viewer.html")

RuntimeError: 沒有可視化的路線（可能全是 ERROR 或被過濾掉）。

In [ ]:
import webbrowser
# path not found 的 debug 圖片
webbrowser.open("routes_samples_viewer.html")

True

### 10/29 UI 用

#### Python讀取航跡陣列

In [ ]:
import csv, json

with open(r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA\routes_samples.csv", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        if row["coor"] != "ERROR":
            pts = json.loads(row["coor"])  # -> [[lon, lat], [lon, lat], ...]
            # 這裡就能自己畫圖或做驗證


#### 存成json格式

In [ ]:
# === 批次計算 -> 扁平 JSON（track: 加密簡化折線, coor: 等時取樣點, distance_km, duration_h） ===
import os, json
from itertools import permutations
from pyproj import Geod

from land_node_route_planner import plan_route
from route_sampler import sample_along_route

# ---- Settings ----
LAND_PATH = r"C:\Users\slab\Desktop\Slab Project\Stage1\data\Land\ne_10m_land.shp"
OUT_DIR   = r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA"
JSON_PATH = os.path.join(OUT_DIR, "routes_samples.json")

SOG_KNOTS_LIST = [8, 10, 12]   # 節
EVERY_HOURS    = 4.0           # 每 h 小時取樣
DENSIFY_STEP_KM = 20.0         # 大圓加密步距（畫線更貼近大圓）

PORTS = {
    "TWKHH": (120.3181,   22.58425),
    #"TWTXG": (120.5075,   24.255),
    #"CNSHG": (121.6147,   31.36636),
    #"KRPUS": (129.033333, 35.116667),
    #"JPUKW": (135.828056, 33.468056),
    "KHKOS": (103.5, 10.633333),
}
#| Sihanoukville| KHKOS   |                      | (103.5, 10.633333)             |
GENERATE_HTML = False  # 不保留地圖

# ---- Helpers ----
GEOD = Geod(ellps="WGS84")

def densify_geodesic(line_ll, step_km=DENSIFY_STEP_KM):
    """對 [(lon,lat), ...] 每段用大圓取樣加密，回傳密集折線。"""
    if not line_ll or len(line_ll) < 2:
        return line_ll
    out = []
    for (lon1,lat1),(lon2,lat2) in zip(line_ll[:-1], line_ll[1:]):
        _, _, dist_m = GEOD.inv(lon1,lat1,lon2,lat2)
        n = max(1, int(dist_m / (step_km*1000)))
        pts = GEOD.npts(lon1,lat1,lon2,lat2,n)  # 不含端點
        if not out:
            out.append((lon1,lat1))
        out.extend(pts)
        out.append((lon2,lat2))
    return out

def duration_hours(distance_km: float, sog_knots: float) -> float:
    """以距離(km)與速度(knots)計算小時數（1 kn = 1.852 km/h）。"""
    v_kmh = float(sog_knots) * 1.852
    return distance_km / v_kmh if v_kmh > 0 else float("inf")

# ---- Main ----
os.makedirs(OUT_DIR, exist_ok=True)
pairs = list(permutations(PORTS.keys(), 2))
data = []

for frm, to in pairs:
    origin = PORTS[frm]; dest = PORTS[to]
    html_path = os.path.join(OUT_DIR, f"route_{frm}_{to}.html")
    print(f"\n=== Planning {frm} -> {to} ===")

    try:
        waypoints_ll, total_km, saved_html, meta = plan_route(
            origin=origin, dest=dest,
            land_path=LAND_PATH, out_html=html_path,
            add_feature_layer=True,
        )

        # 不保留 HTML：立刻刪除
        if not GENERATE_HTML and saved_html and os.path.exists(saved_html):
            try: os.remove(saved_html)
            except: pass

        # 1) 總距離（優先：簡化後）
        distance_km = float(meta.get("total_km_simplified", total_km))

        # 2) 軌跡基底：優先用「簡化後最終軌跡」
        track_base = meta.get("track_simplified_ll")
        if not track_base:
            track_base = meta.get("track_ll", waypoints_ll)  # 後備

        # 3) 取樣 + 輸出
        for sog_knots in SOG_KNOTS_LIST:
            # 等時取樣（沿簡化後軌跡），並必取轉折點
            samples = sample_along_route(
                waypoints_ll=track_base,
                speed_knots=sog_knots,
                every_h=EVERY_HOURS,
                start_time=None,
                include_waypoints=False,
                snap_km=0.5,
            )
            coor_list = [[round(s["lon"], 6), round(s["lat"], 6)] for s in samples]

            # 畫線：把簡化後軌跡做大圓加密
            track_gc = densify_geodesic(track_base, step_km=DENSIFY_STEP_KM)
            track_gc = [[round(lon,6), round(lat,6)] for (lon,lat) in track_gc]

            # 總航行時間（小時）
            dur_h = duration_hours(distance_km, sog_knots)

            data.append({
                "From": frm, "To": to, "Sog": sog_knots,
                "track": track_gc,            # 畫線用（密集大圓，基於簡化後軌跡）
                "coor":  coor_list,           # 畫點用（每 h 等速取樣，必含轉折）
                "distance_km": round(distance_km, 3),
                "duration_h":  round(dur_h, 3),
            })

    except Exception as e:
        print(f"  !! ERROR {frm}->{to}: {e}")
        for sog_knots in SOG_KNOTS_LIST:
            data.append({
                "From": frm, "To": to, "Sog": sog_knots,
                "track": "ERROR", "coor": "ERROR",
                "distance_km": "ERROR", "duration_h": "ERROR",
                "error_msg": str(e),
            })

# 寫出 JSON
with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"\nSaved JSON: {JSON_PATH} (records: {len(data)})")



=== Planning TWKHH -> KHKOS ===
  !! ERROR TWKHH->KHKOS: LVS failed in both directions. fwd=LVS: path not found., rev=LVS: path not found.

=== Planning KHKOS -> TWKHH ===
  !! ERROR KHKOS->TWKHH: LVS failed in both directions. fwd=LVS: path not found., rev=LVS: path not found.

Saved JSON: C:\Users\slab\Desktop\Slab Project\Stage2 ETA\routes_samples.json (records: 6)


#### 看json畫的圖

In [ ]:
# === Visualize flat JSON (track polyline + hourly circle markers) ===
import os, json, folium

JSON_PATH = r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA\routes_samples.json"
OUT_DIR   = r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA"
OUT_HTML  = os.path.join(OUT_DIR, "routes_samples_track_plus_markers.html")

FILTER_FROM = None
FILTER_TO   = None
FILTER_SOG  = None

SOG_COLOR = {8: "#1f77b4", 10: "#ff7f0e", 12: "#2ca02c"}
POINT_RADIUS = 3
POINT_OPACITY = 0.9

def lon_pacific(lon: float) -> float:
    return lon if lon >= 0 else lon + 360.0

with open(JSON_PATH, encoding="utf-8") as f:
    records = json.load(f)

routes = []
for rec in records:
    frm, to, sog = rec.get("From"), rec.get("To"), rec.get("Sog")
    track = rec.get("track"); coor = rec.get("coor")
    if track == "ERROR" or coor == "ERROR": continue
    if not isinstance(track, list) or len(track) < 2: continue
    if not isinstance(coor, list)  or len(coor)  < 2: continue
    if FILTER_FROM and frm != FILTER_FROM: continue
    if FILTER_TO   and to  != FILTER_TO:   continue
    if FILTER_SOG  and sog != FILTER_SOG:  continue
    routes.append({"From": frm, "To": to, "Sog": int(sog), "track": track, "coor": coor})

if not routes:
    raise RuntimeError("沒有可視化的路線。")

p0, p1 = routes[0]["track"][0], routes[0]["track"][-1]
mid_lon, mid_lat = (p0[0]+p1[0])/2, (p0[1]+p1[1])/2

m = folium.Map(
    location=[mid_lat, lon_pacific(mid_lon)],
    zoom_start=4, max_bounds=False, world_copy_jump=False, no_wrap=False, min_lon=0, max_lon=360
)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Ocean_Basemap/MapServer/tile/{z}/{y}/{x}',
    attr='Esri', name='Ocean Basemap', overlay=False, control=True, no_wrap=False
).add_to(m)
folium.TileLayer(
    tiles='https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',
    attr='© OpenStreetMap', name='OpenStreetMap', overlay=False, control=True, no_wrap=False
).add_to(m)

for r in routes:
    frm, to, sog = r["From"], r["To"], r["Sog"]
    track, samples = r["track"], r["coor"]
    color = SOG_COLOR.get(sog, "#d62728")
    layer_name = f"{frm} → {to} @ {sog} kn"
    fg = folium.FeatureGroup(name=layer_name, show=False)

    # 1) 折線：用完整軌跡（含港口與外推接駁）
    track_latlon = [[lat, lon_pacific(lon)] for lon, lat in track]
    folium.PolyLine(track_latlon, color=color, weight=5, opacity=0.95).add_to(fg)

    # 2) 每 h 小時的圓點
    for idx, (lon, lat) in enumerate(samples):
        folium.CircleMarker(
            [lat, lon_pacific(lon)],
            radius=POINT_RADIUS, color=color, weight=1,
            fill=True, fill_opacity=POINT_OPACITY,
            tooltip=f"{frm}→{to} @ {sog} kn | t#{idx}\n({lon:.6f}, {lat:.6f})"
        ).add_to(fg)

    # 3) 起點/終點強化（用港口點，track 首末就是港口）
    s_lon, s_lat = track[0]; e_lon, e_lat = track[-1]
    folium.CircleMarker([s_lat, lon_pacific(s_lon)], radius=POINT_RADIUS+2,
                        color="#2ecc71", weight=2, fill=True, fill_opacity=0.95,
                        tooltip=f"Start: {frm}").add_to(fg)
    folium.CircleMarker([e_lat, lon_pacific(e_lon)], radius=POINT_RADIUS+2,
                        color="#e74c3c", weight=2, fill=True, fill_opacity=0.95,
                        tooltip=f"End: {to}").add_to(fg)

    fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
os.makedirs(OUT_DIR, exist_ok=True)
m.save(OUT_HTML)
print(f"Map saved to: {OUT_HTML}")
m
import webbrowser
# path not found 的 debug 圖片
webbrowser.open("routes_samples_track_plus_markers.html")

RuntimeError: 沒有可視化的路線。

### Sea node 

In [ ]:
# === 一次性：把 land_node_route_planner.py 所在資料夾加進模組搜尋路徑 ===
import sys
from pathlib import Path

# 將此路徑改成你的 land_node_route_planner.py 所在資料夾
PROJECT_DIR = Path(r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA")
if str(PROJECT_DIR) not in sys.path:
    sys.path.append(str(PROJECT_DIR))

# 輸出與資料路徑
LAND_PATH = Path(r"C:\Users\slab\Desktop\Slab Project\Stage1\data\Land\ne_10m_land.shp")
OUT_DIR   = Path(r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA")
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# === 匯入（修改 .py 後可重載） ===
import importlib
import land_node_route_planner as lrp

# 若剛剛有編輯 land_node_route_planner.py，可執行 reload 立即生效
importlib.reload(lrp)

# （若要取樣每 h 小時航點）
import route_sampler as rs
importlib.reload(rs)


<module 'route_sampler' from 'c:\\Users\\slab\\Desktop\\Slab Project\\Stage2 ETA\\route_sampler.py'>

In [ ]:
# === 定義幾個港口座標（lon, lat）===
PORTS = {
    #"TWKHH": (120.3181,   22.58425),
    #"TWTXG": (120.5075,   24.255),
    #"CNSHG": (121.6147,   31.36636),
    "KRPUS": (129.033333, 35.116667),
    "JPUKW": (135.828056, 33.468056),
}

frm, to = "KRPUS", "JPUKW"  # 換成你要的兩個港口
origin = PORTS[frm]
dest   = PORTS[to]

html_path = OUT_DIR / f"route_{frm}_{to}.html"

# === 執行規劃 ===
waypoints_ll, total_km_simplified, html_saved, meta = lrp.plan_route(
    origin=origin,
    dest=dest,
    land_path=LAND_PATH,
    out_html=html_path,
    add_feature_layer=True,   # 想更清爽可改 False
)

print("簡化後總距離 (km):", round(total_km_simplified, 1))
print("HTML 地圖：", html_saved)
print("是否外推起點/終點:", meta["moved_o"], meta["moved_d"])

# 兩種軌跡（都含港口與外推接駁）：
track_for_polyline = meta["track_simplified_ll"]   # ✅ 建議前端/下游用
track_debug_full   = meta["track_ll"]              # 原始選路徑（調試對照）


簡化後總距離 (km): 1082.9
HTML 地圖： C:\Users\slab\Desktop\Slab Project\Stage2 ETA\route_KRPUS_JPUKW.html
是否外推起點/終點: True True


In [ ]:
import webbrowser
webbrowser.open(html_saved)  

True

## Modulize to 10 py file

In [1]:
import sys
sys.path.insert(0, r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA")
from routing.planner import plan_route

import importlib, routing.draw as D
importlib.reload(D)

from routing.planner import plan_route




In [2]:
LAND_PATH = r"C:\Users\slab\Desktop\Slab Project\Stage1\data\Land\ne_10m_land.shp"
OUT_HTML  = r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA\route_demo.html"
origin = (129.033333,35.116667)
dest   = (120.5075,24.255)

wps, total_km, html_path, meta = plan_route(origin, dest, LAND_PATH, OUT_HTML, add_feature_layer=True)
print("OK:", html_path, "simplified_km:", total_km)

    #| KAOHSIUNG  | TWKHH     | (22.58425,120.3181)  | (120.3181,22.58425)            |
    #| TAICHUNG   | TWTXG     | (24.255,120.5075)    | (120.5075,24.255)              |
    #| KEELUNG    | TWKEL     | (25.14515,121.7532)  | (121.7532,25.14515)            |
    #| SHANGHAI   | CNSHG     | (31.36636,121.6147)  | (121.6147,31.36636)            |
    #| NINGBO     | CNNBG     | (29.92654,121.8525)  | (121.8525,29.92654)            |
    #| ZHOUSHAN   | CNZOS     | (29.92161,122.2104)  | (122.2104,29.92161)            |
    #| SHENZHEN   | CNSZX     | (22.5045,113.8535)   | (113.8535,22.5045)             |
    #| LOS ANGELES| USLAX     | (33.74021,-118.265)  | (-118.265,33.74021)            |
    #| SEATTLE    | USSEA     | (47.6212,-122.3643)  | (-122.3643,47.6212)            |
    #| TOKYO      | JPTYO     | (35.61168,139.8268)  | (139.8268,35.61168)            |
    #| KOBE       | JPUKB     | (34.6867,135.2671)   | (135.2671,34.6867)             |
    #| WAKAYAMA   | JPUKW     | (33.468056,135.828056) | (135.828056,33.468056)       |
    #| BUSAN      | KRPUS     | (35.116667,129.033333) | (129.033333,35.116667)       |
    #| Sihanoukville| KHKOS   |                      | (103.5, 10.633333)             |

[PLAN] start | origin=(129.033333, 35.116667) dest=(120.5075, 24.255) use_scgraph=True
[LAND] polys=554 | layers ready (ring/union/prep/strtree)
[SCGRAPH] start O→D path planning...
[LVS] run_once start | O=0 D=1 nodes=2796
[LVS] run_once done  | O=0 D=1 path_len=2
[LVS] run_once start | O=0 D=1 nodes=2796
[LVS] run_once done  | O=0 D=1 path_len=2
[SIMPLIFY] tag=fwd simp_pts=2
[SIMPLIFY] tag=rev simp_pts=2
OK: C:\Users\slab\Desktop\Slab Project\Stage2 ETA\route_demo.html simplified_km: 1480.9818361007374


In [3]:
import webbrowser
webbrowser.open("route_demo.html")

True

In [ ]:
LAND_PATH = r"C:\Users\slab\Desktop\Slab Project\Stage1\data\Land\ne_10m_land.shp"
OUT_HTML  = r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA\route_demo.html"
origin = (129.033333,35.116667)
dest   = (135.828056,33.468056)

wps, total_km, html_path, meta = plan_route(origin, dest, LAND_PATH, OUT_HTML, add_feature_layer=True)
print("OK:", html_path, "simplified_km:", total_km)

    #| KAOHSIUNG  | TWKHH     | (22.58425,120.3181)  | (120.3181,22.58425)            |
    #| TAICHUNG   | TWTXG     | (24.255,120.5075)    | (120.5075,24.255)              |
    #| KEELUNG    | TWKEL     | (25.14515,121.7532)  | (121.7532,25.14515)            |
    #| SHANGHAI   | CNSHG     | (31.36636,121.6147)  | (121.6147,31.36636)            |
    #| NINGBO     | CNNBG     | (29.92654,121.8525)  | (121.8525,29.92654)            |
    #| ZHOUSHAN   | CNZOS     | (29.92161,122.2104)  | (122.2104,29.92161)            |
    #| SHENZHEN   | CNSZX     | (22.5045,113.8535)   | (113.8535,22.5045)             |
    #| LOS ANGELES| USLAX     | (33.74021,-118.265)  | (-118.265,33.74021)            |
    #| SEATTLE    | USSEA     | (47.6212,-122.3643)  | (-122.3643,47.6212)            |
    #| TOKYO      | JPTYO     | (35.61168,139.8268)  | (139.8268,35.61168)            |
    #| KOBE       | JPUKB     | (34.6867,135.2671)   | (135.2671,34.6867)             |
    #| WAKAYAMA   | JPUKW     | (33.468056,135.828056) | (135.828056,33.468056)       |
    #| BUSAN      | KRPUS     | (35.116667,129.033333) | (129.033333,35.116667)       |
    #| Sihanoukville| KHKOS   |                      | (103.5, 10.633333)             |
    #| QINZHOU    | CNQZH     | ----                 | (108.647, 21.68317)            |

[PLAN] start | origin=(129.033333, 35.116667) dest=(135.828056, 33.468056) use_scgraph=True
[LAND] polys=410 | layers ready (ring/union/prep/strtree)
[SCGRAPH] start O→D path planning...
[LVS] run_once start | O=0 D=1 nodes=1891
[LVS] run_once done  | O=0 D=1 path_len=18
[LVS] run_once start | O=0 D=1 nodes=1891
[LVS] run_once done  | O=0 D=1 path_len=14
[SIMPLIFY] tag=fwd simp_pts=14
[SIMPLIFY] tag=rev simp_pts=14
OK: C:\Users\slab\Desktop\Slab Project\Stage2 ETA\route_demo.html simplified_km: 780.9533078617455


In [4]:
# 修改節點後看他會不會走sc
LAND_PATH = r"C:\Users\slab\Desktop\Slab Project\Stage1\data\Land\ne_10m_land.shp"
OUT_HTML  = r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA\route_demo.html"
origin = (108.647, 21.68317)
dest   = (121.8525,29.92654)

wps, total_km, html_path, meta = plan_route(origin, dest, LAND_PATH, OUT_HTML, add_feature_layer=True)
print("OK:", html_path, "simplified_km:", total_km)

    #| KAOHSIUNG  | TWKHH     | (22.58425,120.3181)  | (120.3181,22.58425)            |
    #| TAICHUNG   | TWTXG     | (24.255,120.5075)    | (120.5075,24.255)              |
    #| KEELUNG    | TWKEL     | (25.14515,121.7532)  | (121.7532,25.14515)            |
    #| SHANGHAI   | CNSHG     | (31.36636,121.6147)  | (121.6147,31.36636)            |
    #| NINGBO     | CNNBG     | (29.92654,121.8525)  | (121.8525,29.92654)            |
    #| ZHOUSHAN   | CNZOS     | (29.92161,122.2104)  | (122.2104,29.92161)            |
    #| SHENZHEN   | CNSZX     | (22.5045,113.8535)   | (113.8535,22.5045)             |
    #| LOS ANGELES| USLAX     | (33.74021,-118.265)  | (-118.265,33.74021)            |
    #| SEATTLE    | USSEA     | (47.6212,-122.3643)  | (-122.3643,47.6212)            |
    #| TOKYO      | JPTYO     | (35.61168,139.8268)  | (139.8268,35.61168)            |
    #| KOBE       | JPUKB     | (34.6867,135.2671)   | (135.2671,34.6867)             |
    #| WAKAYAMA   | JPUKW     | (33.468056,135.828056) | (135.828056,33.468056)       |
    #| BUSAN      | KRPUS     | (35.116667,129.033333) | (129.033333,35.116667)       |
    #| Sihanoukville| KHKOS   | (10.633333,103.5)    | (103.5, 10.633333)             |
    #| QINZHOU    | CNQZH     | ----                 | (108.647, 21.68317)            |

[PLAN] start | origin=(108.647, 21.68317) dest=(121.8525, 29.92654) use_scgraph=True
[LAND] polys=359 | layers ready (ring/union/prep/strtree)
[SCGRAPH] start O→D path planning...
[LVS] run_once start | O=0 D=1 nodes=1972
[LVS] run_once done  | O=0 D=1 path_len=33
[LVS] run_once start | O=0 D=1 nodes=1972
[LVS] run_once done  | O=0 D=1 path_len=35
[SIMPLIFY] tag=fwd simp_pts=14
[SIMPLIFY] tag=rev simp_pts=14
OK: C:\Users\slab\Desktop\Slab Project\Stage2 ETA\route_demo.html simplified_km: 2115.3258171843363


In [ ]:
# 修改節點後看他會不會走sc
LAND_PATH = r"C:\Users\slab\Desktop\Slab Project\Stage1\data\Land\ne_10m_land.shp"
OUT_HTML  = r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA\route_demo.html"
origin = (121.6147,31.36636)
dest   = (103.5, 10.633333)

wps, total_km, html_path, meta = plan_route(origin, dest, LAND_PATH, OUT_HTML, add_feature_layer=True)
print("OK:", html_path, "simplified_km:", total_km)

    #| KAOHSIUNG  | TWKHH     | (22.58425,120.3181)  | (120.3181,22.58425)            |
    #| TAICHUNG   | TWTXG     | (24.255,120.5075)    | (120.5075,24.255)              |
    #| KEELUNG    | TWKEL     | (25.14515,121.7532)  | (121.7532,25.14515)            |
    #| SHANGHAI   | CNSHG     | (31.36636,121.6147)  | (121.6147,31.36636)            |
    #| NINGBO     | CNNBG     | (29.92654,121.8525)  | (121.8525,29.92654)            |
    #| ZHOUSHAN   | CNZOS     | (29.92161,122.2104)  | (122.2104,29.92161)            |
    #| SHENZHEN   | CNSZX     | (22.5045,113.8535)   | (113.8535,22.5045)             |
    #| LOS ANGELES| USLAX     | (33.74021,-118.265)  | (-118.265,33.74021)            |
    #| SEATTLE    | USSEA     | (47.6212,-122.3643)  | (-122.3643,47.6212)            |
    #| TOKYO      | JPTYO     | (35.61168,139.8268)  | (139.8268,35.61168)            |
    #| KOBE       | JPUKB     | (34.6867,135.2671)   | (135.2671,34.6867)             |
    #| WAKAYAMA   | JPUKW     | (33.468056,135.828056) | (135.828056,33.468056)       |
    #| BUSAN      | KRPUS     | (35.116667,129.033333) | (129.033333,35.116667)       |00
    #| Sihanoukville| KHKOS   | (10.633333,103.5)    | (103.5, 10.633333)             |
    #| QINZHOU    | CNQZH     | ----                 | (108.647, 21.68317)            |

[PLAN] start | origin=(121.6147, 31.36636) dest=(103.5, 10.633333) use_scgraph=True
[LAND] polys=722 | layers ready (ring/union/prep/strtree)
[SCGRAPH] start O→D path planning...
[LVS] run_once start | O=0 D=1 nodes=4000
[LVS] run_once done  | O=0 D=1 path_len=48
[LVS] run_once start | O=0 D=1 nodes=4000
[LVS] reverse FAILED | RuntimeError('LVS: path not found (graph fully blocked).')
[SIMPLIFY] tag=fwd simp_pts=13
OK: C:\Users\slab\Desktop\Slab Project\Stage2 ETA\route_demo.html simplified_km: 3543.925008882146


In [7]:
import webbrowser
webbrowser.open("route_demo.html")

True

## 12/16 復工--陸地特徵點平滑化

In [12]:
from pathlib import Path
import shapely
from shapely.geometry import Polygon, MultiPolygon
import folium
import webbrowser

# --- 依你現在專案的設定 ---
from routing.config import BUFFER_KM, PAD_DEG, COLLISION_SAFETY_KM
from routing.land_layers import (
    dynamic_bboxes_idl,
    load_polys_in_bboxes,
    build_land_layers,
    union_lonlat_bboxes,
    to_metric,   # land_layers 裡的小工具
    to_wgs,
)

# 這次要測試的區域（你指定的）
origin = (129.033333, 35.116667)
dest   = (120.5075,   24.255)

# land shapefile 路徑（如果不一樣請自行改一下）
LAND_FILE = Path(r"C:\Users\slab\Desktop\Slab Project\Stage1\data\Land\ne_10m_land.shp")

# --------------------------------------------------
# 1) 建立「原始」 land layers（跟 planner 裡一樣）
# --------------------------------------------------
bboxes = dynamic_bboxes_idl(origin, dest, pad_deg=PAD_DEG)
polys  = load_polys_in_bboxes(LAND_FILE, bboxes)
layers_orig = build_land_layers(polys)

union_m_orig     = layers_orig["UNION_M"]      # metric union
ring_m_orig      = layers_orig["RING_M"]       # metric buffer (BUFFER_KM)
land_raw_wgs     = layers_orig["LAND_RAW_WGS"] # WGS 用來畫原始陸地
ring_wgs_orig    = layers_orig["RING_WGS"]     # WGS 用來畫原始 ring

# --------------------------------------------------
# 2) 幫幾種不同平滑策略做出「平滑後的 union / ring」
# --------------------------------------------------

def smooth_union_buffer_simplify_safe(
    union_m,
    smooth_km: float = 5.0,
    tol_km: float = 6.0,
    keep_orig: bool = True,
):
    """
    比較偏 A2 的平滑：
      1) union_m.buffer(+smooth_km)
      2) buffer(-smooth_km)
      3) simplify(tol_km)
      4) 視需要與原始 union 做 union，避免削掉陸地
    回傳拓樸修正後的平滑 union。
    """
    # 1,2: buffer ±r 做幾何磨平
    smooth1 = union_m.buffer(smooth_km * 1000.0)
    smooth2 = smooth1.buffer(-smooth_km * 1000.0)

    # 3: Douglas-Peucker 簡化
    smooth3 = smooth2.simplify(tol_km * 1000.0, preserve_topology=True)

    # 4: 確保不削掉原始陸地：取 union
    if keep_orig:
        smooth3 = smooth3.union(union_m)

    # 5: buffer(0) 一下，保險修復拓樸
    smooth3 = smooth3.buffer(0)

    return smooth3

def simplify_only(union_m, tol_km: float):
    """
    單純做 simplify，不做 buffer +/-。
    """
    return union_m.simplify(tol_km * 1000.0, preserve_topology=True)

# Case A1：溫和 buffer± + simplify
smoothA1_union_m = smooth_union_buffer_simplify_safe(union_m_orig, smooth_km=2.0, tol_km=4.0)
smoothA1_ring_m  = smoothA1_union_m.buffer(BUFFER_KM * 1000.0)
smoothA1_union_wgs = to_wgs(smoothA1_union_m)
smoothA1_ring_wgs  = to_wgs(smoothA1_ring_m)

# Case A2：比較 aggressive 的 buffer± + simplify
smoothA2_union_m = smooth_union_buffer_simplify_safe(union_m_orig, smooth_km=5.0, tol_km=8.0)
smoothA2_ring_m  = smoothA2_union_m.buffer(BUFFER_KM * 1000.0)
smoothA2_union_wgs = to_wgs(smoothA2_union_m)
smoothA2_ring_wgs  = to_wgs(smoothA2_ring_m)

# Case B：單純 simplify（保守版）
simplifyB_union_m = simplify_only(union_m_orig, tol_km=3.0)
simplifyB_ring_m  = simplifyB_union_m.buffer(BUFFER_KM * 1000.0)
simplifyB_union_wgs = to_wgs(simplifyB_union_m)
simplifyB_ring_wgs  = to_wgs(simplifyB_ring_m)

# ---- Case A_mid : Smooth A_mid（介於 A1, A2，偏 A2） ----
smoothMid_union_m = smooth_union_buffer_simplify_safe(
    union_m_orig,
    smooth_km=5.0,
    tol_km=7.0,
    keep_orig=True,   # 保證不削掉原始陸地
)
smoothMid_ring_m    = smoothMid_union_m.buffer(BUFFER_KM * 1000.0)
smoothMid_union_wgs = to_wgs(smoothMid_union_m)
smoothMid_ring_wgs  = to_wgs(smoothMid_ring_m)

print("\n=== Vertex count with A_mid ===")
print(f"Smooth Mid union : {count_vertices(smoothMid_union_m)}  (smooth=4km, tol=6km, union with original)")


# --------------------------------------------------
# 3) 簡單算一下每種 union 的頂點數，幫你感覺複雜度差異
# --------------------------------------------------
def count_vertices(geom) -> int:
    if geom.is_empty:
        return 0
    if isinstance(geom, Polygon):
        return len(geom.exterior.coords)
    if isinstance(geom, MultiPolygon):
        return sum(len(g.exterior.coords) for g in geom.geoms if not g.is_empty)
    # 其他型態就粗略處理
    return len(getattr(geom, "coords", []))

print("=== Vertex count (outer boundary only) ===")
print(f"Original union    : {count_vertices(union_m_orig)}")
print(f"Smooth A1 union   : {count_vertices(smoothA1_union_m)}  (smooth=2km, tol=4km)")
print(f"Smooth A2 union   : {count_vertices(smoothA2_union_m)}  (smooth=5km, tol=8km)")
print(f"Simplify B union  : {count_vertices(simplifyB_union_m)}  (tol=3km)")



=== Vertex count with A_mid ===
Smooth Mid union : 7987  (smooth=4km, tol=6km, union with original)
=== Vertex count (outer boundary only) ===
Original union    : 18840
Smooth A1 union   : 12614  (smooth=2km, tol=4km)
Smooth A2 union   : 7998  (smooth=5km, tol=8km)
Simplify B union  : 4153  (tol=3km)


In [13]:
# --------------------------------------------------
# 4) 用 Folium 畫出原始 & 平滑後的 ring 對照
# --------------------------------------------------

# 地圖中心：用起訖點中點
mid_lat = (origin[1] + dest[1]) / 2.0
mid_lon = (origin[0] + dest[0]) / 2.0

m = folium.Map(
    location=[mid_lat, mid_lon],
    zoom_start=5,
    tiles=None,
)

# 底圖：Ocean Basemap + OSM（可切換）
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Ocean_Basemap/MapServer/tile/{z}/{y}/{x}',
    attr='Esri', name='Ocean Basemap', overlay=False, control=True
).add_to(m)
folium.TileLayer(
    tiles='https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',
    attr='© OpenStreetMap', name='OpenStreetMap', overlay=False, control=True
).add_to(m)

# 起訖點
folium.Marker(
    [origin[1], origin[0]],
    tooltip=f"Origin ({origin[0]:.3f}, {origin[1]:.3f})",
    icon=folium.Icon(color='green', icon='ship', prefix='fa')
).add_to(m)
folium.Marker(
    [dest[1], dest[0]],
    tooltip=f"Dest ({dest[0]:.3f}, {dest[1]:.3f})",
    icon=folium.Icon(color='red', icon='anchor', prefix='fa')
).add_to(m)

# ① 原始陸地
folium.GeoJson(
    land_raw_wgs,
    name="原始陸地 (LAND_RAW_WGS)",
    style_function=lambda x: {"color": "#2ca02c", "weight": 1, "fillOpacity": 0.15},
).add_to(m)

# ② 原始 ring（BUFFER_KM=5km）
folium.GeoJson(
    ring_wgs_orig,
    name=f"原始 ring (BUFFER={BUFFER_KM}km)",
    style_function=lambda x: {"color": "#6a5acd", "weight": 2, "fillOpacity": 0.05},
).add_to(m)

# ③ 平滑版本 A1 ring
folium.GeoJson(
    smoothA1_ring_wgs,
    name="平滑 ring A1 (smooth=2km, tol=4km)",
    style_function=lambda x: {"color": "#1f77b4", "weight": 2, "fillOpacity": 0.05},
).add_to(m)

# ④ 平滑版本 A2 ring（比較 aggressive）
folium.GeoJson(
    smoothA2_ring_wgs,
    name="平滑 ring A2 (smooth=5km, tol=8km)",
    style_function=lambda x: {"color": "#d62728", "weight": 2, "fillOpacity": 0.05, "dashArray": "6,4"},
).add_to(m)

# ⑤ 平滑版本 B ring（純 simplify）
folium.GeoJson(
    simplifyB_ring_wgs,
    name="平滑 ring B (simplify tol=3km)",
    style_function=lambda x: {"color": "#ff7f0e", "weight": 2, "fillOpacity": 0.05, "dashArray": "4,6"},
).add_to(m)

# ④ 平滑版本 A2 ring（比較 aggressive）
folium.GeoJson(
    smoothMid_ring_wgs,
    name="平滑 ring Mid (smooth=5km, tol=6km, safe)",
    style_function=lambda x: {"color": "#00ffff", "weight": 2, "fillOpacity": 0.05},
).add_to(m)
# 圖層控制
folium.LayerControl(collapsed=False).add_to(m)

# 存檔 + 開啟
out_html = Path("land_smoothing_test_busan_taiwan.html")
m.save(out_html)
print("Map saved to:", out_html.resolve())

try:
    webbrowser.open(out_html.resolve().as_uri())
except Exception as e:
    print("無法自動開啟瀏覽器，請手動用瀏覽器打開該 HTML 檔。", e)


Map saved to: C:\Users\slab\Desktop\Slab Project\Stage2 ETA\land_smoothing_test_busan_taiwan.html


### 試著用雙層地圖

In [ ]:
from pathlib import Path

import shapely
from shapely.geometry import Polygon, MultiPolygon
from shapely.ops import transform as shp_transform

from routing.config import PAD_DEG, AVOID_KM, BUFFER_KM
from routing.land_layers import (
    dynamic_bboxes_idl,
    load_polys_in_bboxes,
    build_land_layers,
    union_lonlat_bboxes,
)
from routing.geodesy import to_m, to_ll
import routing.features as feat_mod
from routing import planner

# 你的 land shapefile 路徑（跟 data_loader 裡一樣）
LAND_FILE = Path(r"C:\Users\slab\Desktop\Slab Project\Stage1\data\Land\ne_10m_land.shp")




### helper

In [29]:
def smooth_union_for_features(polys, smooth_km=5.0, tol_km=8.0):
    """
    給【特徵抽取】用的平滑版 union：
      union_m → buffer(+smooth_km) → buffer(-smooth_km) → simplify(tol_km)
    ※ 不跟原始 union 再 union，故可以大幅抹平小凹凸。
    """
    layers = build_land_layers(polys)
    union_m_orig = layers["UNION_M"]   # metric geometry

    smooth1 = union_m_orig.buffer(smooth_km * 1000.0)
    smooth2 = smooth1.buffer(-smooth_km * 1000.0)
    smooth3 = smooth2.simplify(tol_km * 1000.0, preserve_topology=True)
    smooth3 = smooth3.buffer(0)  # 修拓樸

    return smooth3  # metric geometry

def sample_boundary_points_metric(geom_m, step_km: float):
    """
    在 metric 幾何的外邊界上，約每 step_km 取一個點。
    回傳: [(lon, lat), ...]
    """
    if geom_m.is_empty:
        return []

    geoms = list(geom_m.geoms) if isinstance(geom_m, MultiPolygon) else [geom_m]

    pts_ll = []
    step_m = step_km * 1000.0

    for g in geoms:
        boundary = g.exterior
        coords = list(boundary.coords)
        if len(coords) < 2:
            continue

        acc = 0.0
        last_x, last_y = coords[0]

        # 起點也收一個
        lon0, lat0 = to_ll(last_x, last_y)
        pts_ll.append((lon0, lat0))

        for x, y in coords[1:]:
            dx = x - last_x
            dy = y - last_y
            seg = (dx * dx + dy * dy) ** 0.5
            acc += seg
            if acc >= step_m:
                lon, lat = to_ll(x, y)
                pts_ll.append((lon, lat))
                acc = 0.0
            last_x, last_y = x, y

    # 去重
    seen = set()
    uniq = []
    for lon, lat in pts_ll:
        key = (round(lon, 5), round(lat, 5))
        if key in seen:
            continue
        seen.add(key)
        uniq.append((lon, lat))
    return uniq

def extract_feature_points_bbox_smooth(
    shp_path: Path,
    bbox_ll_polygon,
    avoid_km: float = 15.0,
):
    """
    給【特徵點】用的版本：
      1) 用 bbox_ll_polygon 從 shapefile 切出這塊陸地 polys
      2) 在這塊 polys 上做「凶狠平滑 union」（A2 風格）
      3) 在平滑後 union 外再 buffer 一圈 avoid_km 當作航道外圈 ring
      4) 沿 ring 的外邊界，每 step_km≈avoid_km 取一點當特徵點

    回傳格式與原本 extract_feature_points_bbox 相同：
    {
        "convex_peaks": [(lon,lat), ...],
        "convex":       [(lon,lat), ...]
    }
    """
    # 1) 依 bbox 切出多邊形
    polys = load_polys_in_bboxes(shp_path, [bbox_ll_polygon])
    if not polys:
        return {"convex_peaks": [], "convex": []}

    # 2) 凶狠平滑 union（+5/-5, tol=8km）
    union_smooth_m = smooth_union_for_features(
        polys,
        smooth_km=5.0,
        tol_km=8.0,
    )

    # 3) 在平滑 union 外再推一圈 avoid_km，做成「外側避讓圈」
    ring_smooth_m = union_smooth_m.buffer(avoid_km * 1000.0)

    # 4) 沿 ring 的邊界，每 avoid_km 取一個點
    pts_ll = sample_boundary_points_metric(ring_smooth_m, step_km=avoid_km)

    return {
        "convex_peaks": pts_ll,
        "convex": [],
    }





### 改一下planner, 可改回來

In [30]:
import routing.planner as planner

# 備份原本的函式（只備份一次即可）
if not hasattr(planner, "extract_feature_points_bbox_backup"):
    planner.extract_feature_points_bbox_backup = planner.extract_feature_points_bbox

# 用我們的凶狠平滑版取代 planner 內部使用的函式
planner.extract_feature_points_bbox = extract_feature_points_bbox_smooth

print("  planner.extract_feature_points_bbox 已暫時改成凶狠平滑版（僅本次 session 有效）")

# 小檢查：看現在 planner 用的是哪個函式
print("planner.extract_feature_points_bbox ->", planner.extract_feature_points_bbox.__name__)


  planner.extract_feature_points_bbox 已暫時改成凶狠平滑版（僅本次 session 有效）
planner.extract_feature_points_bbox -> extract_feature_points_bbox_smooth


### 改回來

In [ ]:
planner.extract_feature_points_bbox = planner.extract_feature_points_bbox_backup
print(" 已恢復 planner.extract_feature_points_bbox 原本版本")


✅ planner.extract_feature_points_bbox 已暫時改成凶狠平滑版（僅本次 session 有效）
planner.extract_feature_points_bbox -> extract_feature_points_bbox_smooth


In [39]:
   #| KAOHSIUNG  | TWKHH     | (22.58425,120.3181)  | (120.3181,22.58425)            |
    #| TAICHUNG   | TWTXG     | (24.255,120.5075)    | (120.5075,24.255)              |
    #| KEELUNG    | TWKEL     | (25.14515,121.7532)  | (121.7532,25.14515)            |
    #| SHANGHAI   | CNSHG     | (31.36636,121.6147)  | (121.6147,31.36636)            |
    #| NINGBO     | CNNBG     | (29.92654,121.8525)  | (121.8525,29.92654)            |
    #| ZHOUSHAN   | CNZOS     | (29.92161,122.2104)  | (122.2104,29.92161)            |
    #| SHENZHEN   | CNSZX     | (22.5045,113.8535)   | (113.8535,22.5045)             |
    #| LOS ANGELES| USLAX     | (33.74021,-118.265)  | (-118.265,33.74021)            |
    #| SEATTLE    | USSEA     | (47.6212,-122.3643)  | (-122.3643,47.6212)            |
    #| TOKYO      | JPTYO     | (35.61168,139.8268)  | (139.8268,35.61168)            |
    #| KOBE       | JPUKB     | (34.6867,135.2671)   | (135.2671,34.6867)             |
    #| WAKAYAMA   | JPUKW     | (33.468056,135.828056) | (135.828056,33.468056)       |
    #| BUSAN      | KRPUS     | (35.116667,129.033333) | (129.033333,35.116667)       |
    #| Sihanoukville| KHKOS   |                      | (103.5, 10.633333)             |
origin = (129.033333,35.116667)
dest   = (135.828056,33.468056)
out_html = Path(r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA") / "route_smooth_features_strict_collision.html"

track_simplified, total_km, html_path, meta = planner.plan_route(
    origin=origin,
    dest=dest,
    land_path=LAND_FILE,
    out_html=out_html,
    add_feature_layer=True,   # 建議開著，才能看到特徵點分佈
    use_scgraph=True,
)

print("total_km_simplified:", total_km)
print("html_path:", html_path)
print("feature_count (meta):", meta.get("feature_count"))


[PLAN] start | origin=(129.033333, 35.116667) dest=(135.828056, 33.468056) use_scgraph=True
[LAND] polys=410 | layers ready (ring/union/prep/strtree)
[SCGRAPH] start O→D path planning...
[LVS] run_once start | O=0 D=1 nodes=761
[LVS] run_once done  | O=0 D=1 path_len=15
[LVS] run_once start | O=0 D=1 nodes=761
[LVS] run_once done  | O=0 D=1 path_len=14
[SIMPLIFY] tag=fwd simp_pts=14
[SIMPLIFY] tag=rev simp_pts=14
total_km_simplified: 783.5796465980077
html_path: C:\Users\slab\Desktop\Slab Project\Stage2 ETA\route_smooth_features_strict_collision.html
feature_count (meta): 656


In [40]:
webbrowser.open("route_smooth_features_strict_collision.html")


True